# PROJECT GENESIS Ω
## Universal Computation Discovery Laboratory — Continuous Colab Research Engine

> **Não construir um computador dentro de um universo.**
> **Construir um universo no qual alguma forma de computação tenha a oportunidade de ser descoberta.**
> E depois: **não acreditar nela até conseguirmos destruí-la, reproduzi-la e fazê-la funcionar novamente.**

### Como usar
1. `Ambiente de execução → Alterar o tipo de ambiente de execução → GPU` (funciona em CPU também, em modo DEBUG).
2. `Ambiente de execução → Executar tudo`. Autorize o Google Drive quando pedido (é o que guarda a pesquisa).
3. Desça até a seção **18. PESQUISA AO VIVO**: a saída dela é um **painel que se atualiza sozinho a cada segundo**
   (universos sendo simulados agora, fase e progresso, melhor candidato, descobertas, gráficos). Não é preciso
   parar nada nem rodar outras células para ver resultados.
4. A pesquisa continua sozinha até o Colab encerrar a sessão (checkpoint a cada poucos minutos).
   Para **pausar**: botão ■ da célula (salva tudo). Para **continuar**: rode a célula de novo, ou `Executar tudo`
   numa sessão nova — o último checkpoint válido é restaurado e a **mesma** pesquisa continua.

### Honestidade fundamental
Isto **não** simula o Universo físico real. O Genesis cria **universos artificiais completos dentro de suas próprias regras**:
campos contínuos `U(x,y,t) ∈ R^C` sem semântica pré-atribuída, com espaço, tempo, interações locais e distantes, topologia mutável,
conservação parcial, dissipação, recurso abstrato limitado, ruído/perturbação e evolução das próprias leis.
Nenhum conceito de computação (bit, CPU, memória, instrução, neurônio...) existe **dentro** do universo.
Os experimentos (sondas causais, de memória, de canal, de transformação...) são **instrumentos externos**,
cada um com controles positivo e negativo testados no boot.

Nada aqui é chamado de vida, organismo, inteligência, consciência ou AGI. LEVEL 10 permanece reservado.

In [ ]:
# @title ⚙️ Configuração (defaults inteligentes — nada precisa ser mudado) { display-mode: "form" }
import os

PROFILE = "AUTO"  # @param ["AUTO", "DEBUG", "FAST", "RESEARCH", "EXTREME"]
USE_GOOGLE_DRIVE = True  # @param {type:"boolean"}
LIVE_UPDATE_SECONDS = 1.0  # @param {type:"number"}
MAX_SESSION_HOURS = 24.0  # @param {type:"number"}
MAX_GENERATIONS_THIS_SESSION = 0  # @param {type:"integer"}
RUN_BOOT_TESTS = True  # @param {type:"boolean"}
START_RESEARCH = True  # @param {type:"boolean"}
FORCE_NEW_GENESIS = False  # @param {type:"boolean"}
MASTER_SEED = 20261122  # @param {type:"integer"}
GENESIS_ROOT_OVERRIDE = ""  # @param {type:"string"}

# variáveis de ambiente (execução local/automatizada) têm precedência sobre o formulário
_env = os.environ.get
PROFILE = _env("GENESIS_PROFILE", PROFILE)
USE_GOOGLE_DRIVE = _env("GENESIS_USE_DRIVE", "1" if USE_GOOGLE_DRIVE else "0") == "1"
LIVE_UPDATE_SECONDS = float(_env("GENESIS_LIVE_SECONDS", LIVE_UPDATE_SECONDS))
MAX_SESSION_HOURS = float(_env("GENESIS_MAX_HOURS", MAX_SESSION_HOURS))      # 0–24 h; checkpoints frequentes de qualquer forma
MAX_GENERATIONS_THIS_SESSION = int(_env("GENESIS_MAX_GENERATIONS", MAX_GENERATIONS_THIS_SESSION))  # 0 = sem limite
RUN_BOOT_TESTS = _env("GENESIS_RUN_TESTS", "1" if RUN_BOOT_TESTS else "0") == "1"
START_RESEARCH = _env("GENESIS_START_RESEARCH", "1" if START_RESEARCH else "0") == "1"
FORCE_NEW_GENESIS = _env("GENESIS_FORCE_NEW", "1" if FORCE_NEW_GENESIS else "0") == "1"
MASTER_SEED = int(_env("GENESIS_SEED", MASTER_SEED))
GENESIS_ROOT_OVERRIDE = _env("GENESIS_ROOT", GENESIS_ROOT_OVERRIDE)

## 1. Núcleo: imports, armazenamento, logging estruturado, utilidades e estatística
Tudo aqui é infraestrutura **externa** ao universo (o simulador, o laboratório e seus instrumentos).

In [ ]:
# @title 1. Núcleo: imports, armazenamento, logging estruturado, utilidades e estatística { display-mode: "form" }
import sys, time, json, math, uuid, zlib, gzip, shutil, pickle, hashlib, random, platform
import traceback, warnings, copy, heapq, itertools, io, csv, glob, datetime, subprocess, contextlib, tempfile
import base64, collections
import html as _html
from dataclasses import dataclass, field, asdict, fields as dc_fields, replace as dc_replace
from typing import Any, Dict, List, Optional, Tuple, Sequence, Callable

import numpy as np
import torch
import torch.nn.functional as F

IN_NOTEBOOK = "ipykernel" in sys.modules
IN_COLAB = "google.colab" in sys.modules
if not IN_COLAB:
    try:
        import google.colab  # noqa: F401
        IN_COLAB = True
    except Exception:
        IN_COLAB = False

import matplotlib
if not IN_NOTEBOOK:
    matplotlib.use("Agg")
import matplotlib.pyplot as plt

try:
    from IPython.display import display, clear_output, Markdown, HTML, Image as IPImage
except Exception:  # execução como script
    def display(*a, **k):
        for x in a:
            print(x)
    def clear_output(*a, **k):
        return None
    Markdown = str
    HTML = str
    IPImage = None

try:
    import scipy.stats as sstats
    import scipy.ndimage as sndi
    HAVE_SCIPY = True
except Exception:
    sstats, sndi, HAVE_SCIPY = None, None, False

try:
    import pandas as pd
    HAVE_PANDAS = True
except Exception:
    pd, HAVE_PANDAS = None, False

try:
    import psutil
    HAVE_PSUTIL = True
except Exception:
    psutil, HAVE_PSUTIL = None, False

warnings.filterwarnings("ignore", category=UserWarning)

GENESIS_VERSION = "1.0.0"
CHECKPOINT_VERSION = 1
VALIDATION_ENGINE_VERSION = "VE-1.0"
GENESIS_CHANGELOG = [
    ("1.0.0", "Primeira versão completa: física contínua multicanal evoluível, QD (MAP-Elites + novelty + especiação), "
              "Validation Engine congelado com controles, Scientist por valor de informação, checkpoints atômicos."),
]

STATUS_NOT_TESTED, STATUS_RUNNING, STATUS_PASS, STATUS_FAIL, STATUS_INCONCLUSIVE, STATUS_ERROR = (
    "NOT_TESTED", "RUNNING", "PASS", "FAIL", "INCONCLUSIVE", "ERROR")


class _LiveHook:
    """Ponto de acoplamento opcional do painel ao vivo (módulo 12b). Somente observação: o painel lê o estado,
    nunca participa da seleção, do fitness ou dos testes."""
    monitor = None


LIVE = _LiveHook()


def live_progress(phase: Optional[str] = None, done: Optional[float] = None, total: Optional[float] = None,
                  detail: Optional[str] = None):
    m = LIVE.monitor
    if m is not None:
        m.progress(phase, done, total, detail)


# ----------------------------------------------------------------------------
# Armazenamento
# ----------------------------------------------------------------------------
@dataclass
class GenesisPaths:
    root: str
    on_drive: bool

    def __post_init__(self):
        self.checkpoints = os.path.join(self.root, "checkpoints")
        self.backups = os.path.join(self.root, "checkpoints", "backups")
        self.logs = os.path.join(self.root, "logs")
        self.discoveries = os.path.join(self.root, "discoveries")
        self.reports = os.path.join(self.root, "reports")
        self.exports = os.path.join(self.root, "exports")
        self.plots = os.path.join(self.root, "plots")
        self.champion = os.path.join(self.root, "champion")
        self.crash = os.path.join(self.root, "crash_reports")
        for d in [self.root, self.checkpoints, self.backups, self.logs, self.discoveries, self.reports,
                  self.exports, self.plots, self.champion, self.crash]:
            os.makedirs(d, exist_ok=True)
        self.stop_file = os.path.join(self.root, "STOP")


def setup_storage(use_drive: bool, override: str = "") -> GenesisPaths:
    """Google Drive se disponível/montável; senão /content; fora do Colab, ./genesis_runs."""
    if override:
        return GenesisPaths(os.path.abspath(override), on_drive=override.startswith("/content/drive"))
    if IN_COLAB:
        drive_root = "/content/drive/MyDrive"
        if use_drive and not os.path.isdir(drive_root):
            try:
                from google.colab import drive
                drive.mount("/content/drive")
            except Exception as e:
                print(f"[storage] Drive não montado ({e}); usando /content (não persiste se a sessão cair).")
        if use_drive and os.path.isdir(drive_root):
            return GenesisPaths(os.path.join(drive_root, "GENESIS_OMEGA"), on_drive=True)
        return GenesisPaths("/content/GENESIS_OMEGA", on_drive=False)
    return GenesisPaths(os.path.abspath("./genesis_runs"), on_drive=False)


# ----------------------------------------------------------------------------
# Utilidades
# ----------------------------------------------------------------------------
def now_ts() -> float:
    return time.time()


def iso_now() -> str:
    return datetime.datetime.now(datetime.timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")


def new_id(prefix: str) -> str:
    """IDs únicos, seguros contra colisão (uuid4)."""
    return f"{prefix}-{uuid.uuid4().hex[:12]}"


def derive_seed(*parts) -> int:
    h = hashlib.sha256(("|".join(str(p) for p in parts)).encode()).digest()
    return int.from_bytes(h[:4], "little") & 0x7FFFFFFF


def to_jsonable(o):
    if isinstance(o, dict):
        return {str(k): to_jsonable(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)):
        return [to_jsonable(v) for v in o]
    if isinstance(o, np.ndarray):
        return o.tolist()
    if isinstance(o, (np.floating,)):
        v = float(o)
        return v if math.isfinite(v) else None
    if isinstance(o, float):
        return o if math.isfinite(o) else None
    if isinstance(o, (np.integer,)):
        return int(o)
    if isinstance(o, (np.bool_,)):
        return bool(o)
    if isinstance(o, torch.Tensor):
        return to_jsonable(o.detach().cpu().numpy())
    if hasattr(o, "__dataclass_fields__"):
        return to_jsonable(asdict(o))
    return o


def hash_obj(obj) -> str:
    """Hash estável de objetos críticos (dicts/listas/arrays)."""
    hsh = hashlib.sha256()

    def feed(x):
        if isinstance(x, dict):
            hsh.update(b"{")
            for k in sorted(x.keys(), key=str):
                hsh.update(str(k).encode())
                feed(x[k])
            hsh.update(b"}")
        elif isinstance(x, (list, tuple)):
            hsh.update(b"[")
            for v in x:
                feed(v)
            hsh.update(b"]")
        elif isinstance(x, (np.ndarray, np.generic)):
            x = np.asarray(x)
            hsh.update(str(x.dtype).encode() + str(x.shape).encode())
            hsh.update(np.ascontiguousarray(x).tobytes())
        elif isinstance(x, torch.Tensor):
            feed(x.detach().cpu().numpy())
        elif isinstance(x, float):
            hsh.update(repr(round(x, 12)).encode())
        else:
            hsh.update(repr(x).encode())
    feed(obj)
    return hsh.hexdigest()


def fmt_hms(seconds: float) -> str:
    seconds = max(0, int(seconds))
    h, r = divmod(seconds, 3600)
    m, s = divmod(r, 60)
    return f"{h:d}h{m:02d}m{s:02d}s"


def human_bytes(n: float) -> str:
    for u in ["B", "KB", "MB", "GB", "TB"]:
        if abs(n) < 1024:
            return f"{n:.1f}{u}"
        n /= 1024
    return f"{n:.1f}PB"


def is_oom_error(e: BaseException) -> bool:
    if hasattr(torch.cuda, "OutOfMemoryError") and isinstance(e, torch.cuda.OutOfMemoryError):
        return True
    return isinstance(e, RuntimeError) and "out of memory" in str(e).lower()


# ----------------------------------------------------------------------------
# Logging estruturado (events.jsonl + metrics.csv) com rotação
# ----------------------------------------------------------------------------
class EventLogger:
    MAX_LOG_BYTES = 64 * 1024 * 1024

    def __init__(self, paths: GenesisPaths, verbose: bool = True):
        self.paths = paths
        self.events_path = os.path.join(paths.logs, "events.jsonl")
        self.metrics_path = os.path.join(paths.logs, "metrics.csv")
        self.verbose = verbose
        self._metric_fields: Optional[List[str]] = None
        self.recent: List[dict] = []

    def _rotate(self, path):
        try:
            if os.path.exists(path) and os.path.getsize(path) > self.MAX_LOG_BYTES:
                stamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
                with open(path, "rb") as fi, gzip.open(f"{path}.{stamp}.gz", "wb") as fo:
                    shutil.copyfileobj(fi, fo)
                os.remove(path)
        except Exception as e:
            print(f"[log] rotação falhou: {e}")

    def event(self, kind: str, echo: bool = False, **data):
        rec = {"ts": iso_now(), "kind": kind, **to_jsonable(data)}
        self.recent.append(rec)
        self.recent = self.recent[-200:]
        try:
            self._rotate(self.events_path)
            with open(self.events_path, "a") as f:
                f.write(json.dumps(rec) + "\n")
        except Exception as e:
            print(f"[log] falha ao gravar evento: {e}")
        if echo and self.verbose:
            msg = ", ".join(f"{k}={v}" for k, v in data.items() if not isinstance(v, (dict, list)))
            print(f"[{kind}] {msg}")

    def metrics(self, row: Dict[str, Any]):
        row = {k: to_jsonable(v) for k, v in row.items()}
        try:
            self._rotate(self.metrics_path)
            exists = os.path.exists(self.metrics_path)
            if exists and self._metric_fields is None:
                with open(self.metrics_path) as f:
                    header = f.readline().strip()
                    self._metric_fields = header.split(",") if header else None
            if self._metric_fields is None:
                self._metric_fields = list(row.keys())
                exists = False
            with open(self.metrics_path, "a", newline="") as f:
                w = csv.DictWriter(f, fieldnames=self._metric_fields, extrasaction="ignore")
                if not exists:
                    w.writeheader()
                w.writerow(row)
        except Exception as e:
            print(f"[log] falha ao gravar métricas: {e}")


# ----------------------------------------------------------------------------
# Estatística (instrumentos externos)
# ----------------------------------------------------------------------------
def summary_stats(x) -> Dict[str, float]:
    x = np.asarray([v for v in np.ravel(x) if v is not None and np.isfinite(v)], dtype=np.float64)
    if x.size == 0:
        return {"n": 0, "mean": float("nan"), "median": float("nan"), "std": float("nan"),
                "ci_lo": float("nan"), "ci_hi": float("nan")}
    lo, hi = bootstrap_ci(x)
    return {"n": int(x.size), "mean": float(x.mean()), "median": float(np.median(x)),
            "std": float(x.std(ddof=1)) if x.size > 1 else 0.0, "ci_lo": lo, "ci_hi": hi}


def bootstrap_ci(x, n_boot: int = 2000, alpha: float = 0.05, seed: int = 0, stat=np.mean) -> Tuple[float, float]:
    x = np.asarray(x, dtype=np.float64)
    x = x[np.isfinite(x)]
    if x.size == 0:
        return float("nan"), float("nan")
    if x.size == 1:
        return float(x[0]), float(x[0])
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, x.size, size=(n_boot, x.size))
    bs = stat(x[idx], axis=1)
    return float(np.quantile(bs, alpha / 2)), float(np.quantile(bs, 1 - alpha / 2))


def cohens_d(a, b) -> float:
    a = np.asarray(a, dtype=np.float64); b = np.asarray(b, dtype=np.float64)
    a = a[np.isfinite(a)]; b = b[np.isfinite(b)]
    if a.size < 2 or b.size < 2:
        return float("nan")
    s = math.sqrt(((a.size - 1) * a.var(ddof=1) + (b.size - 1) * b.var(ddof=1)) / (a.size + b.size - 2))
    if s < 1e-12:
        diff = a.mean() - b.mean()
        return float(np.sign(diff) * 10.0) if abs(diff) > 1e-12 else 0.0
    return float((a.mean() - b.mean()) / s)


def mann_whitney_greater(a, b) -> float:
    """p-valor unilateral (a > b)."""
    a = np.asarray(a, dtype=np.float64); b = np.asarray(b, dtype=np.float64)
    a = a[np.isfinite(a)]; b = b[np.isfinite(b)]
    if a.size == 0 or b.size == 0:
        return float("nan")
    if HAVE_SCIPY:
        try:
            return float(sstats.mannwhitneyu(a, b, alternative="greater").pvalue)
        except Exception as e:
            warnings.warn(f"mannwhitneyu (scipy) falhou ({e}); usando aproximação normal")
    # aproximação normal
    allv = np.concatenate([a, b])
    ranks = np.argsort(np.argsort(allv)) + 1.0
    u = ranks[:a.size].sum() - a.size * (a.size + 1) / 2
    mu = a.size * b.size / 2
    sd = math.sqrt(a.size * b.size * (a.size + b.size + 1) / 12) + 1e-12
    z = (u - mu) / sd
    return float(0.5 * math.erfc(z / math.sqrt(2)))


def binom_p_greater(k: int, n: int, p0: float) -> float:
    if n <= 0:
        return 1.0
    if HAVE_SCIPY:
        try:
            return float(sstats.binomtest(int(k), int(n), p0, alternative="greater").pvalue)
        except Exception as e:
            warnings.warn(f"binomtest (scipy) falhou ({e}); usando soma exata")
    return float(sum(math.comb(n, i) * p0 ** i * (1 - p0) ** (n - i) for i in range(int(k), n + 1)))


def holm_correction(pvals: Sequence[float]) -> List[float]:
    p = np.asarray([1.0 if (v is None or not np.isfinite(v)) else v for v in pvals], dtype=np.float64)
    m = p.size
    order = np.argsort(p)
    adj = np.empty(m)
    running = 0.0
    for rank, i in enumerate(order):
        val = min(1.0, (m - rank) * p[i])
        running = max(running, val)
        adj[i] = running
    return adj.tolist()


print(f"PROJECT GENESIS Ω v{GENESIS_VERSION} | Python {platform.python_version()} | torch {torch.__version__} | "
      f"Colab={IN_COLAB} | notebook={IN_NOTEBOOK} | scipy={HAVE_SCIPY}")

## 2. Configuração central, perfis de execução e detecção de hardware
Todos os thresholds ficam centralizados aqui (sem números mágicos espalhados). A configuração é serializável,
vai em cada checkpoint e em cada descoberta (histórico de configuração / thresholds).

In [ ]:
# @title 2. Configuração central, perfis de execução e detecção de hardware { display-mode: "form" }
@dataclass
class Thresholds:
    # sobrevivência dinâmica (fase 1)
    dead_abs: float = 2e-3
    frozen_rel_activity: float = 2e-4
    explosive_saturation: float = 0.6
    noise_tac: float = 0.15
    noise_sac: float = 0.15
    noise_entropy: float = 0.80
    # triagem -> experimentos
    persistence: float = 0.6
    causal_spread: float = 0.10
    anomaly_surprise: float = 4.0
    chaos_lyap: float = 0.03
    # estatística dos probes
    alpha: float = 0.01
    min_norm_acc: float = 0.30
    causal_spread_probe: float = 0.03
    causal_repeatability: float = 0.5
    memory_min_delay: int = 10
    recovery_sim: float = 0.6
    recovery_gain: float = 0.3
    robust_sim: float = 0.75
    robustness: float = 0.6
    generalization_drop: float = 0.30
    adaptation_p: float = 0.05
    replication_pass_rate: float = 0.8
    # champion gate
    champion_level: int = 7
    champion_reproducibility: float = 0.8
    champion_robustness: float = 0.6
    dominance_p: float = 0.05
    dominance_effect: float = 0.5
    # anti-metric-gaming
    metric_exploit_gain: float = 0.10
    metric_exploit_window: int = 15


@dataclass
class ProbeSettings:
    warmup: int = 48
    trials: int = 8
    permutations: int = 100
    folds: int = 4
    memory_delays: List[int] = field(default_factory=lambda: [1, 10, 100, 1000])
    memory_history_steps: int = 6
    channel_symbols: int = 4
    channel_distance: int = 16
    channel_times: List[int] = field(default_factory=lambda: [0, 2, 4, 8, 16, 32, 64])
    channel_noise: float = 0.02
    stim_radius: int = 3
    obs_radius: int = 4
    stim_amp: float = 0.8
    stim_pulses: int = 2
    transform_times: List[int] = field(default_factory=lambda: [4, 8, 16, 32, 64])
    damage_levels: List[float] = field(default_factory=lambda: [0.10, 0.25, 0.50, 0.75, 0.90])
    recovery_warmup: int = 96
    recovery_steps: int = 96
    robust_steps: int = 128
    robust_trials: int = 4
    noise_levels: List[float] = field(default_factory=lambda: [0.0, 0.01, 0.03, 0.1])
    replication_seeds: int = 10
    critical_seeds: int = 30
    causal_steps: int = 64
    causal_trials: int = 8
    adaptation_steps: int = 96


@dataclass
class GenesisConfig:
    profile: str = "RESEARCH"
    # universo
    world_size: int = 64
    channels_max: int = 6
    kernel_slots: int = 3
    radius_max: int = 7
    links_max: int = 4
    delay_max: int = 4
    state_bound: float = 6.0
    # população / triagem
    population_size: int = 128
    batch_size: int = 256
    screen_envs: int = 2
    screen_steps: int = 192
    screen_burn_in: int = 64
    record_every: int = 6
    metric_res: int = 32
    immigrant_rate: float = 0.05
    species_cap_fraction: float = 0.3
    species_threshold: float = 0.35
    novelty_k: int = 10
    novelty_archive_max: int = 6000
    map_bins: List[int] = field(default_factory=lambda: [8, 8, 5])
    map_axes: List[str] = field(default_factory=lambda: ["persistence", "causal_spread", "temporal_complexity"])
    stagnation_window: int = 12
    # numérica
    precision: str = "fp32"
    use_compile: bool = False
    deterministic: bool = False
    # tempo e orçamento
    max_session_hours: float = 11.0
    checkpoint_every_minutes: float = 10.0
    dashboard_every_seconds: float = 30.0
    live_interval: float = 1.0
    max_generations_session: int = 0
    experiment_time_fraction: float = 0.35
    max_experiments_per_generation: int = 6
    champion_lab_max_per_session: int = 2
    # sementes
    master_seed: int = 20261122
    thresholds: Thresholds = field(default_factory=Thresholds)
    probe: ProbeSettings = field(default_factory=ProbeSettings)

    def to_dict(self) -> dict:
        return to_jsonable(asdict(self))

    @staticmethod
    def from_dict(d: dict) -> "GenesisConfig":
        d = copy.deepcopy(d)
        th = Thresholds(**{k: v for k, v in d.pop("thresholds", {}).items() if k in Thresholds.__dataclass_fields__})
        pr = ProbeSettings(**{k: v for k, v in d.pop("probe", {}).items() if k in ProbeSettings.__dataclass_fields__})
        cfg = GenesisConfig(**{k: v for k, v in d.items() if k in GenesisConfig.__dataclass_fields__})
        cfg.thresholds, cfg.probe = th, pr
        return cfg

    def thresholds_hash(self) -> str:
        return hash_obj(to_jsonable(asdict(self.thresholds)))[:16]


def detect_hardware() -> Dict[str, Any]:
    info: Dict[str, Any] = {
        "python": platform.python_version(), "torch": torch.__version__,
        "cuda_version": torch.version.cuda, "platform": platform.platform(),
        "cpu_count": os.cpu_count(), "timestamp": iso_now(), "genesis_version": GENESIS_VERSION,
        "device": "cuda" if torch.cuda.is_available() else "cpu",
    }
    if HAVE_PSUTIL:
        try:
            info["ram_total"] = psutil.virtual_memory().total
        except Exception as e:
            info["ram_total_error"] = repr(e)
    if torch.cuda.is_available():
        p = torch.cuda.get_device_properties(0)
        info["gpu_name"] = p.name
        info["vram_total"] = int(p.total_memory)
        try:
            free, total = torch.cuda.mem_get_info()
            info["vram_free"] = int(free)
        except Exception:
            info["vram_free"] = int(p.total_memory)
        info["compute_capability"] = f"{p.major}.{p.minor}"
        try:
            info["bf16_supported"] = bool(torch.cuda.is_bf16_supported())
        except Exception:
            info["bf16_supported"] = False
    else:
        info["gpu_name"] = "none (CPU)"
        info["vram_total"] = 0
        info["vram_free"] = 0
        info["bf16_supported"] = False
    return info


def profile_config(profile: str, hw: Dict[str, Any], master_seed: int, max_hours: float, max_gens: int) -> GenesisConfig:
    """Escolhe defaults seguros por perfil. Tamanho de batch/população é refinado depois pelo autotune."""
    gpu = hw["device"] == "cuda"
    if profile == "AUTO":
        if not gpu:
            profile = "DEBUG"
        else:
            profile = "RESEARCH"
    cfg = GenesisConfig(profile=profile, master_seed=master_seed, max_session_hours=max_hours,
                        max_generations_session=max_gens)
    pr = cfg.probe
    if profile == "DEBUG":
        cfg.world_size, cfg.channels_max, cfg.kernel_slots, cfg.radius_max = 32, 4, 2, 5
        cfg.links_max, cfg.delay_max = 2, 2
        cfg.population_size, cfg.batch_size = 24, 48
        cfg.screen_envs, cfg.screen_steps, cfg.screen_burn_in, cfg.record_every, cfg.metric_res = 2, 96, 32, 4, 16
        cfg.map_bins = [6, 6, 4]
        cfg.checkpoint_every_minutes, cfg.dashboard_every_seconds = 2.0, 10.0
        cfg.max_experiments_per_generation = 3
        cfg.stagnation_window = 6
        pr.trials, pr.permutations, pr.warmup = 8, 40, 24
        pr.memory_delays, pr.channel_distance = [1, 10, 40], 8
        pr.channel_times, pr.transform_times = [0, 2, 4, 8, 16, 32], [4, 8, 16, 32]
        pr.recovery_warmup, pr.recovery_steps, pr.robust_steps, pr.robust_trials = 48, 48, 64, 3
        pr.replication_seeds, pr.critical_seeds, pr.causal_steps, pr.causal_trials = 4, 6, 32, 6
        pr.adaptation_steps, pr.stim_radius, pr.obs_radius = 48, 2, 3
        pr.damage_levels = [0.10, 0.50, 0.90]
        pr.noise_levels = [0.0, 0.03, 0.1]
    elif profile == "FAST":
        cfg.world_size, cfg.channels_max = 48, 5
        cfg.population_size, cfg.batch_size = 96, 256
        cfg.screen_steps, cfg.screen_burn_in, cfg.record_every, cfg.metric_res = 160, 48, 6, 24
        cfg.checkpoint_every_minutes = 5.0
        cfg.dashboard_every_seconds = 15.0
        pr.trials, pr.permutations = 8, 60
        pr.memory_delays, pr.channel_distance = [1, 10, 100], 12
        pr.replication_seeds, pr.critical_seeds = 6, 12
    elif profile == "RESEARCH":
        cfg.world_size = 64
        cfg.population_size, cfg.batch_size = 256, 512
        cfg.screen_steps, cfg.screen_burn_in = 224, 64
        cfg.checkpoint_every_minutes = 5.0
        pr.memory_delays = [1, 10, 100, 1000]
    elif profile == "EXTREME":
        cfg.world_size = 96 if hw.get("vram_total", 0) >= 20e9 else 64
        cfg.population_size, cfg.batch_size = 512, 1024
        cfg.screen_steps, cfg.screen_burn_in = 288, 96
        cfg.checkpoint_every_minutes = 5.0
        pr.memory_delays = [1, 10, 100, 1000, 10000]
        pr.trials, pr.permutations = 12, 200
        pr.critical_seeds = 30
    else:
        raise ValueError(f"perfil desconhecido: {profile}")
    return cfg


VRAM_TARGET = {"DEBUG": 0.30, "FAST": 0.50, "RESEARCH": 0.70, "EXTREME": 0.87}

## 3. Genoma de universo
O genoma é um conjunto estruturado de **leis** (não de "programa"): kernels espaciais (forma, raio, anisotropia),
tensor de interação entre canais, família de não linearidades, difusão, dissipação, reação, transporte, inércia (ondas),
constantes temporais, atraso, histerese, grau de conservação, saturação, topologia de longo alcance e sua plasticidade,
dinâmica de recurso abstrato, condição de borda, assincronia e os **próprios parâmetros de mutação** (auto-adaptativos).

Nenhum canal tem significado. Os nomes abaixo descrevem **o papel matemático do termo na lei**, não a semântica do estado.

In [ ]:
# @title 3. Genoma de universo { display-mode: "form" }
KERNEL_SHAPES = ["ring", "gaussian", "moore", "von_neumann", "anisotropic_ring"]
BOUNDARIES = ["periodic", "reflective", "absorbing"]
MUTATION_OPERATORS = ["gauss_small", "gauss_strong", "structural", "temporal", "topology", "crossover"]


@dataclass
class ParamSpec:
    name: str
    shape: Tuple[int, ...]
    low: float
    high: float
    kind: str          # float | int | bool | cat | logfloat
    group: str         # local_laws, interaction, nonlinear, diffusion, ...
    channel_axes: Tuple[int, ...] = ()


class ParamSpace:
    """Define o espaço de genomas, amostragem, limites, mutação e crossover."""

    def __init__(self, cfg: GenesisConfig):
        C, K, L, D = cfg.channels_max, cfg.kernel_slots, cfg.links_max, cfg.delay_max
        R = cfg.radius_max
        self.C, self.K, self.L, self.D, self.R = C, K, L, D, R
        S = [
            ParamSpec("ch_active", (C,), 0, 1, "bool", "channels", (0,)),
            ParamSpec("k_active", (K,), 0, 1, "bool", "topology"),
            ParamSpec("k_shape", (K,), 0, len(KERNEL_SHAPES) - 1, "cat", "topology"),
            ParamSpec("k_radius", (K,), 1.0, float(R), "float", "topology"),
            ParamSpec("k_mu", (K,), 0.1, 0.9, "float", "topology"),
            ParamSpec("k_sigma", (K,), 0.05, 0.5, "float", "topology"),
            ParamSpec("k_aniso", (K,), 0.3, 1.0, "float", "topology"),
            ParamSpec("k_angle", (K,), 0.0, math.pi, "float", "topology"),
            ParamSpec("W", (C, K, C), -2.0, 2.0, "float", "interaction", (0, 2)),
            ParamSpec("g_mix", (C, 3), -1.5, 1.5, "float", "nonlinear", (0,)),
            ParamSpec("g_mu", (C,), -1.0, 1.5, "float", "nonlinear", (0,)),
            ParamSpec("g_sigma", (C,), 0.05, 1.0, "float", "nonlinear", (0,)),
            ParamSpec("g_bias", (C,), -1.0, 1.0, "float", "nonlinear", (0,)),
            ParamSpec("g_gain", (C,), 0.0, 2.0, "float", "nonlinear", (0,)),
            ParamSpec("diff", (C,), 0.0, 1.0, "float", "diffusion", (0,)),
            ParamSpec("diss", (C,), 0.0, 1.0, "float", "dissipation", (0,)),
            ParamSpec("Q", (C, C), -1.0, 1.0, "float", "reaction", (0, 1)),
            ParamSpec("vel", (C, 2), -0.5, 0.5, "float", "propagation", (0,)),
            ParamSpec("inertia", (C,), 0.0, 0.95, "float", "propagation", (0,)),
            ParamSpec("tau", (C,), 0.5, 20.0, "logfloat", "temporal", (0,)),
            ParamSpec("delay_idx", (C,), 0, D, "int", "temporal", (0,)),
            ParamSpec("delay_w", (C,), -1.0, 1.0, "float", "temporal", (0,)),
            ParamSpec("hyst_rate", (C,), 0.001, 0.5, "logfloat", "temporal", (0,)),
            ParamSpec("hyst_w", (C,), -1.0, 1.0, "float", "temporal", (0,)),
            ParamSpec("conserve", (C,), 0.0, 1.0, "float", "conservation", (0,)),
            ParamSpec("sat", (C,), 0.5, 4.0, "float", "saturation", (0,)),
            ParamSpec("link_coup", (C,), -1.0, 1.0, "float", "topology", (0,)),
            ParamSpec("link_active", (L,), 0, 1, "bool", "topology"),
            ParamSpec("link_offset", (L, 2), -0.5, 0.5, "float", "topology"),
            ParamSpec("link_weight", (L,), -1.0, 1.0, "float", "topology"),
            ParamSpec("link_eta", (), 0.0, 0.2, "float", "topology"),
            ParamSpec("link_decay", (), 0.0, 0.2, "float", "topology"),
            ParamSpec("res_enabled", (), 0, 1, "bool", "resource_dynamics"),
            ParamSpec("res_rho", (), 0.0, 0.2, "float", "resource_dynamics"),
            ParamSpec("res_cons", (), 0.0, 1.0, "float", "resource_dynamics"),
            ParamSpec("res_khalf", (), 0.05, 2.0, "logfloat", "resource_dynamics"),
            ParamSpec("res_diff", (), 0.0, 0.2, "float", "resource_dynamics"),
            ParamSpec("max_rate", (), 0.05, 1.0, "float", "local_laws"),
            ParamSpec("noise", (), 0.0, 0.05, "float", "local_laws"),
            ParamSpec("async_p", (), 0.5, 1.0, "float", "temporal"),
            ParamSpec("time_scale", (), 0.05, 0.5, "float", "temporal"),
            ParamSpec("boundary", (), 0, len(BOUNDARIES) - 1, "cat", "topology"),
            ParamSpec("mut_sigma", (), 0.005, 0.5, "logfloat", "mutation_parameters"),
            ParamSpec("mut_strong", (), 0.0, 0.3, "float", "mutation_parameters"),
            ParamSpec("mut_struct", (), 0.0, 0.5, "float", "mutation_parameters"),
            ParamSpec("mut_topo", (), 0.0, 0.5, "float", "mutation_parameters"),
        ]
        self.specs: Dict[str, ParamSpec] = {s.name: s for s in S}
        self.mutation_param_names = ["mut_sigma", "mut_strong", "mut_struct", "mut_topo"]
        self.groups = sorted(set(s.group for s in S))

    # ------------------------------------------------------------------
    def clamp(self, params: Dict[str, np.ndarray]) -> Dict[str, np.ndarray]:
        for n, s in self.specs.items():
            v = np.asarray(params[n])
            if s.kind == "bool":
                v = v.astype(bool)
            elif s.kind in ("int", "cat"):
                v = np.clip(np.rint(np.nan_to_num(v.astype(np.float64))), s.low, s.high).astype(np.int64)
            else:
                v = np.clip(np.nan_to_num(v.astype(np.float64), nan=(s.low + s.high) / 2), s.low, s.high).astype(np.float32)
            params[n] = np.array(v).reshape(s.shape)
        if not params["ch_active"].any():
            params["ch_active"][0] = True
        if not params["k_active"].any():
            params["k_active"][0] = True
        return params

    def random_params(self, rng: np.random.Generator) -> Dict[str, np.ndarray]:
        p: Dict[str, np.ndarray] = {}
        for n, s in self.specs.items():
            if s.kind == "bool":
                p[n] = rng.random(s.shape) < 0.6
            elif s.kind in ("int", "cat"):
                p[n] = rng.integers(int(s.low), int(s.high) + 1, size=s.shape)
            elif s.kind == "logfloat":
                p[n] = np.exp(rng.uniform(math.log(s.low), math.log(s.high), size=s.shape))
            else:
                p[n] = rng.uniform(s.low, s.high, size=s.shape)
        C = self.C
        n_ch = int(rng.integers(2, C + 1)) if C >= 2 else 1
        act = np.zeros(C, bool); act[rng.choice(C, n_ch, replace=False)] = True
        p["ch_active"] = act
        # priors mais suaves para leis aleatórias (a maioria ainda será lixo — é para isso que existe seleção)
        p["W"] = rng.normal(0, 0.6, size=self.specs["W"].shape) * (rng.random(self.specs["W"].shape) < 0.6)
        p["Q"] = rng.normal(0, 0.3, size=self.specs["Q"].shape) * (rng.random(self.specs["Q"].shape) < 0.3)
        p["diss"] = rng.uniform(0.0, 0.4, size=(C,))
        p["diff"] = rng.uniform(0.0, 0.4, size=(C,))
        p["vel"] = rng.normal(0, 0.1, size=(C, 2))
        p["inertia"] = rng.uniform(0, 0.5, size=(C,)) * (rng.random(C) < 0.4)
        p["delay_w"] = rng.normal(0, 0.3, size=(C,)) * (rng.random(C) < 0.3)
        p["hyst_w"] = rng.normal(0, 0.3, size=(C,)) * (rng.random(C) < 0.3)
        p["conserve"] = rng.uniform(0, 1, size=(C,)) * (rng.random(C) < 0.4)
        p["link_active"] = rng.random(self.L) < 0.3
        p["link_eta"] = np.float32(rng.uniform(0, 0.05) if rng.random() < 0.3 else 0.0)
        p["noise"] = np.float32(rng.uniform(0, 0.01))
        p["async_p"] = np.float32(1.0 if rng.random() < 0.6 else rng.uniform(0.5, 1.0))
        p["mut_sigma"] = np.float32(0.05)
        p["mut_strong"] = np.float32(0.05)
        p["mut_struct"] = np.float32(0.1)
        p["mut_topo"] = np.float32(0.1)
        p["boundary"] = np.int64(rng.choice([0, 0, 1, 2]))
        return self.clamp(p)

    def zero_params(self) -> Dict[str, np.ndarray]:
        """Lei nula (baseline/null model): nada interage, nada muda exceto saturação suave."""
        p = {}
        for n, s in self.specs.items():
            if s.kind == "bool":
                p[n] = np.zeros(s.shape, bool)
            elif s.kind in ("int", "cat"):
                p[n] = np.zeros(s.shape, np.int64)
            else:
                p[n] = np.full(s.shape, s.low, np.float32) if s.low > 0 else np.zeros(s.shape, np.float32)
        p["ch_active"][:] = True
        p["k_active"][0] = True
        p["W"][:] = 0; p["g_gain"][:] = 0; p["diss"][:] = 0; p["diff"][:] = 0
        p["sat"][:] = 4.0; p["tau"][:] = 1.0; p["time_scale"] = np.float32(0.2)
        p["max_rate"] = np.float32(1.0); p["async_p"] = np.float32(1.0)
        p["mut_sigma"] = np.float32(0.05)
        return self.clamp(p)

    # ------------------------------------------------------------------
    def vector(self, params: Dict[str, np.ndarray]) -> np.ndarray:
        """Vetor normalizado do genótipo (para especiação/distância genética)."""
        out = []
        for n, s in self.specs.items():
            if s.group == "mutation_parameters":
                continue
            v = np.asarray(params[n], dtype=np.float64).ravel()
            if s.kind == "logfloat":
                v = (np.log(v) - math.log(s.low)) / (math.log(s.high) - math.log(s.low))
            elif s.high > s.low:
                v = (v - s.low) / (s.high - s.low)
            out.append(v)
        return np.concatenate(out).astype(np.float32)

    # ------------------------------------------------------------------
    def _perturb(self, p, name, rng, sigma, frac=0.5):
        s = self.specs[name]
        v = np.asarray(p[name], dtype=np.float64)
        mask = rng.random(s.shape) < frac if s.shape else np.array(True)
        if s.kind == "logfloat":
            v = np.exp(np.log(v) + mask * rng.normal(0, sigma * 2.0, size=s.shape))
        elif s.kind in ("int", "cat"):
            v = v + mask * rng.integers(-1, 2, size=s.shape)
        elif s.kind == "bool":
            flip = rng.random(s.shape) < sigma * frac if s.shape else (rng.random() < sigma * frac)
            v = np.logical_xor(v.astype(bool), flip)
        else:
            v = v + mask * rng.normal(0, sigma * (s.high - s.low), size=s.shape)
        p[name] = v

    def _resample(self, p, name, rng, frac=0.3):
        s = self.specs[name]
        fresh = self.random_params(rng)[name]
        mask = rng.random(s.shape) < frac if s.shape else np.array(True)
        p[name] = np.where(mask, fresh, p[name])

    def duplicate_channel(self, p, rng) -> str:
        act = np.where(p["ch_active"])[0]; ina = np.where(~p["ch_active"])[0]
        if act.size == 0 or ina.size == 0:
            return "dup_channel(skip)"
        src, dst = int(rng.choice(act)), int(rng.choice(ina))
        for n, s in self.specs.items():
            for ax in s.channel_axes:
                a = np.array(p[n])
                idx_src = [slice(None)] * a.ndim; idx_src[ax] = src
                idx_dst = [slice(None)] * a.ndim; idx_dst[ax] = dst
                a[tuple(idx_dst)] = a[tuple(idx_src)]
                p[n] = a
        p["ch_active"][dst] = True
        # pequena quebra de simetria
        self._perturb(p, "W", rng, 0.02, frac=0.2)
        return f"dup_channel({src}->{dst})"

    def remove_channel(self, p, rng) -> str:
        act = np.where(p["ch_active"])[0]
        if act.size <= 1:
            return "remove_channel(skip)"
        c = int(rng.choice(act))
        p["ch_active"][c] = False
        return f"remove_channel({c})"

    def mutate(self, params: Dict[str, np.ndarray], rng: np.random.Generator, op: str,
               sigma_mult: float = 1.0, other: Optional[Dict[str, np.ndarray]] = None) -> Tuple[Dict[str, np.ndarray], List[str]]:
        p = {k: np.array(v, copy=True) for k, v in params.items()}
        rec: List[str] = []
        # auto-adaptação dos parâmetros de mutação (log-normal) — meta-evolução restrita ao genoma
        tau_m = 0.2
        p["mut_sigma"] = np.float32(float(p["mut_sigma"]) * math.exp(tau_m * rng.normal()))
        for n in ["mut_strong", "mut_struct", "mut_topo"]:
            p[n] = np.float32(float(p[n]) * math.exp(tau_m * rng.normal()) + 0.002 * rng.normal())
        p = self.clamp(p)
        sigma = float(p["mut_sigma"]) * sigma_mult
        float_names = [n for n, s in self.specs.items() if s.kind in ("float", "logfloat") and s.group != "mutation_parameters"]
        if op == "gauss_small":
            chosen = [n for n in float_names if rng.random() < 0.35] or [str(rng.choice(float_names))]
            for n in chosen:
                self._perturb(p, n, rng, sigma)
            rec.append(f"gauss_small(sigma={sigma:.3f},n={len(chosen)})")
        elif op == "gauss_strong":
            chosen = list(rng.choice(float_names, size=int(rng.integers(1, 4)), replace=False))
            for n in chosen:
                self._resample(p, n, rng, frac=0.5)
            rec.append(f"gauss_strong({','.join(chosen)})")
        elif op == "structural":
            r = rng.random()
            if r < 0.25:
                rec.append(self.duplicate_channel(p, rng))
            elif r < 0.40:
                rec.append(self.remove_channel(p, rng))
            elif r < 0.60:
                k = int(rng.integers(0, self.K))
                p["k_shape"][k] = int(rng.integers(0, len(KERNEL_SHAPES)))
                p["k_active"][k] = True
                rec.append(f"kernel_shape(k={k}->{KERNEL_SHAPES[p['k_shape'][k]]})")
            elif r < 0.75:
                k = int(rng.integers(0, self.K))
                p["k_active"][k] = ~p["k_active"][k]
                rec.append(f"kernel_toggle(k={k})")
            else:
                mask = rng.random(self.specs["W"].shape) < 0.15
                p["W"] = np.where(mask, rng.normal(0, 0.8, size=mask.shape) * (rng.random(mask.shape) < 0.7), p["W"])
                rec.append("interaction_rewire")
        elif op == "temporal":
            for n in ["tau", "inertia", "delay_w", "hyst_rate", "hyst_w", "time_scale", "async_p"]:
                if rng.random() < 0.5:
                    self._perturb(p, n, rng, sigma)
            if rng.random() < 0.3:
                self._perturb(p, "delay_idx", rng, 1.0)
            rec.append("temporal")
        elif op == "topology":
            r = rng.random()
            if r < 0.3:
                l = int(rng.integers(0, self.L)); p["link_active"][l] = ~p["link_active"][l]
                rec.append(f"link_toggle(l={l})")
            elif r < 0.6:
                self._perturb(p, "link_offset", rng, sigma * 2); self._perturb(p, "link_weight", rng, sigma)
                rec.append("link_shift")
            elif r < 0.75:
                self._perturb(p, "link_eta", rng, sigma); self._perturb(p, "link_decay", rng, sigma)
                rec.append("link_plasticity")
            elif r < 0.92:
                self._perturb(p, "k_radius", rng, sigma * 2); self._perturb(p, "k_aniso", rng, sigma)
                self._perturb(p, "k_angle", rng, sigma)
                rec.append("kernel_geometry")
            else:
                p["boundary"] = np.int64(rng.integers(0, len(BOUNDARIES)))
                rec.append(f"boundary->{BOUNDARIES[int(p['boundary'])]}")
        elif op == "crossover" and other is not None:
            groups = sorted(set(s.group for s in self.specs.values()))
            taken = []
            for g in groups:
                if g == "mutation_parameters":
                    continue
                if rng.random() < 0.5:
                    for n, s in self.specs.items():
                        if s.group == g:
                            p[n] = np.array(other[n], copy=True)
                    taken.append(g)
            self._perturb(p, str(rng.choice(float_names)), rng, sigma * 0.5)
            rec.append(f"crossover(groups={'+'.join(taken) or 'none'})")
        else:
            self._perturb(p, str(rng.choice(float_names)), rng, sigma)
            rec.append("gauss_single")
        # mutação estrutural / topológica ocasional segundo as taxas herdadas
        if op != "structural" and rng.random() < float(p["mut_struct"]) * 0.3:
            rec.append(self.duplicate_channel(p, rng) if rng.random() < 0.5 else self.remove_channel(p, rng))
        if op != "topology" and rng.random() < float(p["mut_topo"]) * 0.3:
            l = int(rng.integers(0, self.L)); p["link_active"][l] = ~p["link_active"][l]
            rec.append(f"link_toggle(l={l})")
        return self.clamp(p), rec


@dataclass
class Genome:
    params: Dict[str, np.ndarray]
    genome_id: str = ""
    parent_ids: List[str] = field(default_factory=list)
    generation: int = 0
    mutation_record: List[str] = field(default_factory=list)
    birth_timestamp: float = field(default_factory=now_ts)
    species_id: int = -1
    origin: str = "random"
    operator: str = "init"

    def __post_init__(self):
        if not self.genome_id:
            self.genome_id = new_id("G")

    def hash(self) -> str:
        return hash_obj(self.params)[:20]

    def n_channels(self) -> int:
        return int(np.asarray(self.params["ch_active"]).sum())

    def to_dict(self) -> dict:
        return {"params": {k: np.array(v) for k, v in self.params.items()}, "genome_id": self.genome_id,
                "parent_ids": list(self.parent_ids), "generation": self.generation,
                "mutation_record": list(self.mutation_record), "birth_timestamp": self.birth_timestamp,
                "species_id": self.species_id, "origin": self.origin, "operator": self.operator}

    @staticmethod
    def from_dict(d: dict) -> "Genome":
        return Genome(params={k: np.array(v) for k, v in d["params"].items()}, genome_id=d["genome_id"],
                      parent_ids=list(d.get("parent_ids", [])), generation=int(d.get("generation", 0)),
                      mutation_record=list(d.get("mutation_record", [])),
                      birth_timestamp=float(d.get("birth_timestamp", now_ts())),
                      species_id=int(d.get("species_id", -1)), origin=d.get("origin", "imported"),
                      operator=d.get("operator", "init"))

    def summary(self, space: "ParamSpace") -> str:
        p = self.params
        act = np.where(p["ch_active"])[0].tolist()
        ks = [f"{KERNEL_SHAPES[int(p['k_shape'][k])]}(r={float(p['k_radius'][k]):.1f})"
              for k in range(space.K) if p["k_active"][k]]
        links = int(np.asarray(p["link_active"]).sum())
        return (f"{self.genome_id} gen={self.generation} ch={act} kernels={ks} links={links} "
                f"boundary={BOUNDARIES[int(p['boundary'])]} dt={float(p['time_scale']):.3f} "
                f"mut_sigma={float(p['mut_sigma']):.3f} origin={self.origin}")


def control_genome(space: ParamSpace, kind: str, channel_distance: int = 8, world_size: int = 32) -> Genome:
    """Leis artificiais conhecidas, usadas SOMENTE como controles positivos/negativos dos instrumentos
    e como baselines (lei simples, lei nula, ruído). Não entram na evolução."""
    p = space.zero_params()
    if kind == "identity":          # preserva o estado (controle positivo de memória / negativo de causalidade)
        p["time_scale"] = np.float32(0.2)     # lei nula: só a saturação suave age
    elif kind == "noise_replace":   # substitui o estado por ruído a cada passo (controle negativo de memória)
        p["time_scale"] = np.float32(0.5); p["tau"][:] = 0.5; p["diss"][:] = 1.0
        p["noise"] = np.float32(0.05); p["max_rate"] = np.float32(1.0)
    elif kind == "diffusion":       # difusão pura (controle positivo de causalidade/recuperação)
        p["time_scale"] = np.float32(0.5); p["tau"][:] = 0.5; p["diff"][:] = 0.2
    elif kind == "link_copy":       # conexão distante que copia a região A em B (controle positivo de canal)
        p["time_scale"] = np.float32(0.5); p["tau"][:] = 0.5; p["diss"][:] = 1.0
        p["link_active"][:] = False; p["link_active"][0] = True
        p["link_offset"][0] = [0.0, -channel_distance / float(world_size)]
        p["link_weight"][0] = 1.0; p["link_coup"][:] = 1.0
    elif kind == "reaction_diffusion":   # baseline de lei simples (tipo ativador-inibidor)
        p["time_scale"] = np.float32(0.2)
        p["diff"][:] = 0.05; p["diff"][1 % space.C] = 0.4
        p["W"][0, 0, 0] = 1.2; p["W"][0, 0, 1 % space.C] = -1.5
        p["W"][1 % space.C, 0, 0] = 1.0
        p["g_gain"][:] = 1.0; p["g_mix"][:, 0] = 1.0; p["diss"][:] = 0.3
        p["k_shape"][0] = 0; p["k_radius"][0] = min(4.0, float(space.R)); p["k_mu"][0] = 0.5; p["k_sigma"][0] = 0.15
    else:
        raise ValueError(kind)
    return Genome(params=space.clamp(p), origin=f"control:{kind}")

## 4. Física do universo artificial
$$U_{t+\Delta t} = F(U_t, N_t, G_t, R_t, \Theta)$$

* `U` — campo `[B, C, S, S]` (B universos em paralelo na GPU). Nenhum canal tem significado.
* `N_t` — vizinhança via kernels evoluídos (anel, gaussiano, Moore, von Neumann, anisotrópico, multiescala) por FFT,
  mais operadores fixos de gradiente/laplaciano (difusão, transporte).
* `G_t` — topologia: conexões distantes esparsas cujo peso é um **campo local plástico** (surgem/somem/fortalecem/enfraquecem).
* `R_t` — recurso abstrato limitado, regenerável, difusível, consumido por crescimento positivo.
* `Θ` — leis do genoma: interação linear, não linearidades mistas (tanh / sino / ímpar saturante), reação quadrática,
  difusão, advecção, inércia (ondas), atraso, histerese (traço lento), dissipação, conservação parcial, saturação,
  constantes temporais por canal, atualização assíncrona parcial, ruído intrínseco.

**Discretização numérica externa ≠ relógio físico interno.** O passo do backend é discreto; cada canal tem sua
própria taxa (`time_scale/tau`), há atraso e assincronia. O *dt adaptativo* (redução do passo quando |ΔU| excede o limite)
é uma salvaguarda numérica registrada, não física emergente.

Bordas: `periodic`, `reflective`, `absorbing` (por universo, misturáveis no mesmo batch). A arquitetura (vizinhança por
kernels + links indexados) foi desenhada para estender a 1D/3D/grafos trocando apenas `pad`/FFT e a indexação dos links.

In [ ]:
# @title 4. Física do universo artificial { display-mode: "form" }
@dataclass
class Environment:
    env_id: str
    split: str             # evolution | validation | holdout
    init_kind: str         # blobs | noise | sparse | gradient | center | uniform
    density: float
    amplitude: float
    resource_scale: float
    ext_noise: float
    shock_period: int
    shock_amp: float
    seed: int

    def to_dict(self):
        return asdict(self)


def make_environment_suite(master_seed: int) -> List[Environment]:
    """Ensemble de ambientes separado em evolução / validação / holdout final (nunca visto pela seleção)."""
    rng = np.random.default_rng(derive_seed(master_seed, "envs"))
    spec = [
        ("ENV-A", "evolution", "blobs", 0.5, 0.8, 1.0, 0.000, 0, 0.0),
        ("ENV-B", "evolution", "noise", 1.0, 0.5, 1.0, 0.002, 0, 0.0),
        ("ENV-C", "evolution", "sparse", 0.05, 1.0, 0.8, 0.000, 64, 0.05),
        ("ENV-D", "evolution", "gradient", 0.5, 0.6, 1.2, 0.001, 0, 0.0),
        ("ENV-E", "validation", "blobs", 0.3, 1.0, 0.9, 0.002, 96, 0.03),
        ("ENV-F", "validation", "center", 0.5, 0.9, 1.0, 0.001, 0, 0.0),
        ("ENV-G", "holdout", "noise", 0.7, 0.7, 1.1, 0.003, 80, 0.04),
        ("ENV-H", "holdout", "sparse", 0.10, 0.8, 0.7, 0.001, 0, 0.0),
    ]
    return [Environment(n, sp, k, d, a, r, en, spp, sa, int(rng.integers(1, 2 ** 31 - 1)))
            for n, sp, k, d, a, r, en, spp, sa in spec]


def make_initial_field(env: Environment, C: int, S: int, seed: int, ch_active: np.ndarray) -> np.ndarray:
    """Condição inicial reprodutível por universo (semente própria)."""
    rng = np.random.default_rng(seed)
    U = np.zeros((C, S, S), np.float32)
    yy, xx = np.mgrid[0:S, 0:S].astype(np.float32)
    a = env.amplitude
    if env.init_kind == "blobs":
        nb = max(1, int(round(env.density * S * S / 256)))
        for _ in range(nb):
            cy, cx = rng.uniform(0, S, 2); r = rng.uniform(2, max(3, S / 8))
            d2 = np.minimum(np.abs(yy - cy), S - np.abs(yy - cy)) ** 2 + np.minimum(np.abs(xx - cx), S - np.abs(xx - cx)) ** 2
            blob = np.exp(-d2 / (2 * r * r))
            U += blob[None] * rng.normal(0, a, size=(C, 1, 1)).astype(np.float32)
    elif env.init_kind == "noise":
        U = rng.uniform(-a, a, size=(C, S, S)).astype(np.float32) * env.density
    elif env.init_kind == "sparse":
        m = rng.random((C, S, S)) < env.density
        U = np.where(m, rng.choice([-a, a], size=(C, S, S)), 0).astype(np.float32)
    elif env.init_kind == "gradient":
        g = (xx / S - 0.5) * 2 * a
        U = g[None] * rng.normal(1, 0.3, size=(C, 1, 1)) + rng.normal(0, 0.05 * a, size=(C, S, S))
    elif env.init_kind == "center":
        d2 = (yy - S / 2) ** 2 + (xx - S / 2) ** 2
        U = np.exp(-d2 / (2 * (S / 8) ** 2))[None] * rng.normal(0, a, size=(C, 1, 1)) + rng.normal(0, 0.02, size=(C, S, S))
    elif env.init_kind == "uniform":
        U = np.full((C, S, S), a, np.float32) * (rng.choice([-1, 1], size=(C, 1, 1)))
    else:
        raise ValueError(env.init_kind)
    return (U * np.asarray(ch_active, np.float32)[:, None, None]).astype(np.float32)


_KERNEL_CACHE: Dict[tuple, np.ndarray] = {}


def build_kernel_np(shape: int, radius: float, mu: float, sigma: float, aniso: float, angle: float, R: int) -> np.ndarray:
    key = (int(shape), round(float(radius), 3), round(float(mu), 3), round(float(sigma), 3),
           round(float(aniso), 3), round(float(angle), 3), R)
    if key in _KERNEL_CACHE:
        return _KERNEL_CACHE[key]
    yy, xx = np.mgrid[-R:R + 1, -R:R + 1].astype(np.float64)
    r = max(float(radius), 1.0)
    if shape == 2:      # Moore
        ri = max(1, int(round(r)))
        k = ((np.abs(yy) <= ri) & (np.abs(xx) <= ri)).astype(np.float64); k[R, R] = 0
    elif shape == 3:    # von Neumann
        ri = max(1, int(round(r)))
        k = ((np.abs(yy) + np.abs(xx)) <= ri).astype(np.float64); k[R, R] = 0
    else:
        if shape == 4:
            c, s = math.cos(angle), math.sin(angle)
            u = c * xx + s * yy; v = (-s * xx + c * yy) / max(aniso, 0.2)
            d = np.sqrt(u * u + v * v) / r
        else:
            d = np.sqrt(xx * xx + yy * yy) / r
        if shape == 1:
            k = np.exp(-0.5 * (d / max(sigma, 0.05)) ** 2)
        else:
            k = np.exp(-0.5 * ((d - mu) / max(sigma, 0.05)) ** 2)
        k[d > 1.0] = 0.0
    s = k.sum()
    if s <= 1e-9:
        k = np.zeros_like(k); k[R - 1, R] = k[R + 1, R] = k[R, R - 1] = k[R, R + 1] = 1.0; s = 4.0
    k = (k / s).astype(np.float32)
    if len(_KERNEL_CACHE) < 50000:
        _KERNEL_CACHE[key] = k
    return k


class GenomeBatch:
    """Tensores das leis de B universos (genomas podem se repetir no batch)."""
    TENSOR_NAMES = ["chm", "W", "m0", "m1", "m2", "g_mu", "g_sig", "g_bias", "g_gain", "diff", "diss", "Q",
                    "vy", "vx", "inertia", "rate", "delay_k", "delay_w", "hyst_rate", "hyst_w", "conserve", "sat",
                    "link_coup", "link_on", "link_oy", "link_ox", "link_w0", "link_eta", "link_decay",
                    "res_on", "res_rho", "res_cons", "res_khalf", "res_diff", "max_rate", "noise", "async_p",
                    "bmode", "Khat"]

    def __init__(self, sim: "Simulator", genomes: List[Genome], S: int, _empty: bool = False):
        self.sim, self.S = sim, S
        cfg = sim.cfg
        self.C, self.K, self.L, self.D = cfg.channels_max, cfg.kernel_slots, cfg.links_max, cfg.delay_max
        self.P = cfg.radius_max
        self.Hp = S + 2 * self.P
        self.genome_ids = [g.genome_id for g in genomes]
        if _empty:
            return
        dev = sim.device
        B = len(genomes)
        C, K, L, R = self.C, self.K, self.L, cfg.radius_max

        def st(name, dtype=np.float32):
            return np.stack([np.asarray(g.params[name]) for g in genomes]).astype(dtype)

        def T(a, dtype=torch.float32):
            return torch.as_tensor(np.ascontiguousarray(a), device=dev, dtype=dtype)

        chm = st("ch_active")
        kact = st("k_active")
        self.chm = T(chm).view(B, C, 1, 1)
        W = st("W") * chm[:, :, None, None] * chm[:, None, None, :] * kact[:, None, :, None]
        self.W = T(W)
        gm = st("g_mix")
        self.m0, self.m1, self.m2 = (T(gm[:, :, i]).view(B, C, 1, 1) for i in range(3))
        self.g_mu = T(st("g_mu")).view(B, C, 1, 1)
        self.g_sig = T(st("g_sigma")).view(B, C, 1, 1)
        self.g_bias = T(st("g_bias")).view(B, C, 1, 1)
        self.g_gain = T(st("g_gain")).view(B, C, 1, 1)
        rate = st("time_scale")[:, None] / st("tau")
        self.rate = T(rate).view(B, C, 1, 1)
        diff = np.minimum(st("diff"), 0.2 / np.maximum(rate, 1e-6))         # estabilidade explícita
        self.diff = T(diff).view(B, C, 1, 1)
        self.diss = T(st("diss")).view(B, C, 1, 1)
        self.Q = T(st("Q") * chm[:, :, None] * chm[:, None, :])
        vel = st("vel"); vmax = 0.5 / np.maximum(rate, 1e-6)
        self.vy = T(np.clip(vel[:, :, 0], -vmax, vmax)).view(B, C, 1, 1)
        self.vx = T(np.clip(vel[:, :, 1], -vmax, vmax)).view(B, C, 1, 1)
        self.inertia = T(st("inertia")).view(B, C, 1, 1)
        self.delay_k = T(st("delay_idx", np.int64), torch.long).view(B, C)
        self.delay_w = T(st("delay_w")).view(B, C, 1, 1)
        self.hyst_rate = T(st("hyst_rate")).view(B, C, 1, 1)
        self.hyst_w = T(st("hyst_w")).view(B, C, 1, 1)
        self.conserve = T(st("conserve")).view(B, C, 1, 1)
        self.sat = T(st("sat")).view(B, C, 1, 1)
        self.link_coup = T(st("link_coup")).view(B, C, 1, 1)
        lon = st("link_active")
        self.link_on = T(lon).view(B, L)
        off = st("link_offset")
        oy = np.rint(off[:, :, 0] * S).astype(np.int64); ox = np.rint(off[:, :, 1] * S).astype(np.int64)
        zero = (oy == 0) & (ox == 0); ox = np.where(zero, 1, ox)
        self.link_oy, self.link_ox = T(oy, torch.long), T(ox, torch.long)
        self.link_w0 = T(st("link_weight") * lon).view(B, L, 1, 1)
        self.link_eta = T(st("link_eta")).view(B, 1, 1, 1)
        self.link_decay = T(st("link_decay")).view(B, 1, 1, 1)
        self.res_on = T(st("res_enabled")).view(B, 1, 1, 1)
        self.res_rho = T(st("res_rho")).view(B, 1, 1, 1)
        self.res_cons = T(st("res_cons")).view(B, 1, 1, 1)
        self.res_khalf = T(st("res_khalf")).view(B, 1, 1, 1)
        self.res_diff = T(st("res_diff")).view(B, 1, 1, 1)
        self.max_rate = T(st("max_rate")).view(B, 1, 1, 1)
        self.noise = T(st("noise")).view(B, 1, 1, 1)
        self.async_p = T(st("async_p")).view(B, 1, 1, 1)
        self.bmode = T(st("boundary", np.int64), torch.long).view(B)
        # kernels evoluídos (FFT sobre a grade com padding)
        D = 2 * R + 1
        ks = np.zeros((B, K, D, D), np.float32)
        for b, g in enumerate(genomes):
            p = g.params
            for k in range(K):
                if p["k_active"][k]:
                    ks[b, k] = build_kernel_np(int(p["k_shape"][k]), float(p["k_radius"][k]), float(p["k_mu"][k]),
                                               float(p["k_sigma"][k]), float(p["k_aniso"][k]),
                                               float(p["k_angle"][k]), R)
        kt = T(ks)
        img = F.pad(kt, (0, self.Hp - D, 0, self.Hp - D))
        img = torch.roll(img, shifts=(-R, -R), dims=(2, 3))
        kh = torch.fft.rfft2(img)
        self.Khat = torch.cat([kh, sim.fixed_hat(self.Hp).unsqueeze(0).expand(B, -1, -1, -1)], dim=1)
        self._flags()

    def _flags(self):
        self.B = int(self.chm.shape[0])
        self.has_links = bool((self.link_on > 0).any().item())
        self.link_any = (self.link_on > 0).any(0).tolist()
        self.has_res = bool((self.res_on > 0).any().item())
        self.has_async = bool((self.async_p < 0.999).any().item())
        self.has_noise = bool((self.noise > 0).any().item())
        self.has_delay = self.D > 0 and bool((self.delay_w.abs() > 0).any().item())
        self.has_hyst = bool((self.hyst_w.abs() > 0).any().item())
        self.has_inertia = bool((self.inertia > 0).any().item())
        self.has_plastic = self.has_links and bool((self.link_eta > 0).any().item())
        self.modes_present = sorted(set(self.bmode.tolist()))

    def select(self, idx) -> "GenomeBatch":
        idx_t = torch.as_tensor(idx, device=self.sim.device, dtype=torch.long)
        nb = GenomeBatch(self.sim, [], self.S, _empty=True)
        nb.genome_ids = [self.genome_ids[i] for i in (idx.tolist() if hasattr(idx, "tolist") else idx)]
        for n in self.TENSOR_NAMES:
            setattr(nb, n, getattr(self, n).index_select(0, idx_t))
        nb._flags()
        return nb


@dataclass
class SimState:
    U: torch.Tensor
    Uprev: torch.Tensor
    H: torch.Tensor
    Dbuf: Optional[torch.Tensor]
    ptr: int
    R: torch.Tensor
    R0: torch.Tensor
    G: Optional[torch.Tensor]
    t: int
    bad: torch.Tensor
    group_idx: torch.Tensor
    n_groups: int
    gen: torch.Generator
    ext_noise: torch.Tensor
    shock_period: torch.Tensor
    shock_amp: torch.Tensor
    dt_scale_sum: torch.Tensor
    seeds: List[int]

    def select(self, idx, groups: Optional[Sequence[int]] = None, seed: int = 0) -> "SimState":
        dev = self.U.device
        it = torch.as_tensor(idx, device=dev, dtype=torch.long)
        sel = lambda x: None if x is None else x.index_select(0, it).clone()
        if groups is None:
            groups = list(range(len(it)))
        g = torch.Generator(device=dev); g.manual_seed(int(seed))
        Dbuf = None if self.Dbuf is None else self.Dbuf.index_select(1, it).clone()
        idx_list = idx.tolist() if hasattr(idx, "tolist") else list(idx)
        return SimState(U=sel(self.U), Uprev=sel(self.Uprev), H=sel(self.H), Dbuf=Dbuf, ptr=self.ptr,
                        R=sel(self.R), R0=sel(self.R0), G=sel(self.G), t=self.t, bad=sel(self.bad),
                        group_idx=torch.as_tensor(groups, device=dev, dtype=torch.long),
                        n_groups=int(max(groups)) + 1, gen=g, ext_noise=sel(self.ext_noise),
                        shock_period=sel(self.shock_period), shock_amp=sel(self.shock_amp),
                        dt_scale_sum=sel(self.dt_scale_sum), seeds=[self.seeds[i] for i in idx_list])

    def damage_cells(self, mask: torch.Tensor):
        """Zera todo o estado local (inclusive histórico e traço) nas células marcadas. mask: [B,1,S,S] bool."""
        keep = (~mask).float()
        self.U = self.U * keep; self.Uprev = self.Uprev * keep; self.H = self.H * keep
        if self.Dbuf is not None:
            self.Dbuf = self.Dbuf * keep.unsqueeze(0)


class Simulator:
    """Motor numérico vetorizado. Externo ao universo."""

    def __init__(self, cfg: GenesisConfig, device: torch.device):
        self.cfg, self.device = cfg, device
        self._fixed_cache: Dict[int, torch.Tensor] = {}
        self._grid_cache: Dict[int, Tuple[torch.Tensor, torch.Tensor]] = {}
        self.autocast_dtype = None
        if cfg.precision == "bf16" and device.type == "cuda":
            self.autocast_dtype = torch.bfloat16
        elif cfg.precision == "fp16" and device.type == "cuda":
            self.autocast_dtype = torch.float16
        self.steps_done = 0
        self._physics = self._physics_eager
        if cfg.use_compile:
            try:
                self._physics = torch.compile(self._physics_eager, dynamic=False)
            except Exception:
                self._physics = self._physics_eager

    # ------------------------------------------------------------------
    def fixed_hat(self, Hp: int) -> torch.Tensor:
        """Operadores fixos: laplaciano (5 pontos), gradiente y, gradiente x (diferença central)."""
        if Hp not in self._fixed_cache:
            img = torch.zeros(3, Hp, Hp, device=self.device)
            img[0, 0, 0] = -4; img[0, 1, 0] = 1; img[0, -1, 0] = 1; img[0, 0, 1] = 1; img[0, 0, -1] = 1
            img[1, -1, 0] = 0.5; img[1, 1, 0] = -0.5      # (U[y+1]-U[y-1])/2 como convolução
            img[2, 0, -1] = 0.5; img[2, 0, 1] = -0.5
            self._fixed_cache[Hp] = torch.fft.rfft2(img)
        return self._fixed_cache[Hp]

    def grid(self, S: int):
        if S not in self._grid_cache:
            yy, xx = torch.meshgrid(torch.arange(S, device=self.device), torch.arange(S, device=self.device), indexing="ij")
            self._grid_cache[S] = (yy.reshape(-1), xx.reshape(-1))
        return self._grid_cache[S]

    def prepare(self, genomes: List[Genome], S: Optional[int] = None) -> GenomeBatch:
        return GenomeBatch(self, genomes, S or self.cfg.world_size)

    # ------------------------------------------------------------------
    def init_state(self, gb: GenomeBatch, envs: List[Environment], seeds: List[int],
                   groups: Optional[List[int]] = None, run_seed: int = 0,
                   genomes: Optional[List[Genome]] = None, fields: Optional[np.ndarray] = None) -> SimState:
        B, C, S, dev = gb.B, gb.C, gb.S, self.device
        if fields is None:
            chm = gb.chm.view(B, C).cpu().numpy()
            fields = np.stack([make_initial_field(envs[i], C, S, seeds[i], chm[i]) for i in range(B)])
        U = torch.as_tensor(fields, device=dev, dtype=torch.float32)
        U = torch.clamp(U, -self.cfg.state_bound, self.cfg.state_bound)
        groups = list(range(B)) if groups is None else list(groups)
        g = torch.Generator(device=dev); g.manual_seed(int(run_seed) & 0x7FFFFFFF)
        Dbuf = U.unsqueeze(0).repeat(self.cfg.delay_max + 1, 1, 1, 1, 1) if self.cfg.delay_max > 0 else None
        R0 = torch.as_tensor([e.resource_scale for e in envs], device=dev, dtype=torch.float32).view(B, 1, 1, 1)
        G = gb.link_w0.expand(B, gb.L, S, S).clone() if gb.L > 0 else None
        return SimState(U=U, Uprev=U.clone(), H=U.clone(), Dbuf=Dbuf, ptr=0, R=R0.expand(B, 1, S, S).clone(), R0=R0,
                        G=G, t=0, bad=torch.zeros(B, dtype=torch.bool, device=dev),
                        group_idx=torch.as_tensor(groups, device=dev, dtype=torch.long), n_groups=max(groups) + 1,
                        gen=g, ext_noise=torch.as_tensor([e.ext_noise for e in envs], device=dev).view(B, 1, 1, 1).float(),
                        shock_period=torch.as_tensor([e.shock_period for e in envs], device=dev, dtype=torch.long),
                        shock_amp=torch.as_tensor([e.shock_amp for e in envs], device=dev).view(B, 1, 1, 1).float(),
                        dt_scale_sum=torch.zeros(B, device=dev), seeds=list(seeds))

    # ------------------------------------------------------------------
    def pad(self, X: torch.Tensor, gb: GenomeBatch, P: int) -> torch.Tensor:
        modes = gb.modes_present
        if len(modes) == 1:
            m = modes[0]
            return F.pad(X, (P, P, P, P), mode=["circular", "reflect", "constant"][m])
        out = torch.zeros(X.shape[0], X.shape[1], X.shape[2] + 2 * P, X.shape[3] + 2 * P, device=X.device, dtype=X.dtype)
        for m in modes:
            sel = (gb.bmode == m).view(-1, 1, 1, 1).to(X.dtype)
            out = out + sel * F.pad(X, (P, P, P, P), mode=["circular", "reflect", "constant"][m])
        return out

    def link_source(self, U: torch.Tensor, gb: GenomeBatch, l: int) -> torch.Tensor:
        B, C, S, _ = U.shape
        yy, xx = self.grid(S)
        ny = yy.unsqueeze(0) + gb.link_oy[:, l:l + 1]
        nx = xx.unsqueeze(0) + gb.link_ox[:, l:l + 1]
        mode = gb.bmode.view(B, 1)
        py, px = ny % S, nx % S
        ry = ny.abs(); ry = torch.where(ry > S - 1, 2 * (S - 1) - ry, ry).clamp(0, S - 1)
        rx = nx.abs(); rx = torch.where(rx > S - 1, 2 * (S - 1) - rx, rx).clamp(0, S - 1)
        valid = ((ny >= 0) & (ny < S) & (nx >= 0) & (nx < S)) | (mode != 2)
        fy = torch.where(mode == 0, py, torch.where(mode == 1, ry, ny.clamp(0, S - 1)))
        fx = torch.where(mode == 0, px, torch.where(mode == 1, rx, nx.clamp(0, S - 1)))
        idx = (fy * S + fx)
        src = torch.gather(U.reshape(B, C, S * S), 2, idx.unsqueeze(1).expand(B, C, S * S))
        src = src * valid.unsqueeze(1).to(U.dtype)
        return src.view(B, C, S, S)

    def _grand(self, st: SimState, shape_tail, kind: str = "normal") -> torch.Tensor:
        """Aleatoriedade compartilhada por grupo (pares contrafactuais recebem o MESMO ruído)."""
        shp = (st.n_groups,) + tuple(shape_tail)
        if kind == "normal":
            r = torch.randn(shp, generator=st.gen, device=self.device)
        else:
            r = torch.rand(shp, generator=st.gen, device=self.device)
        ident = getattr(st, "_groups_identity", None)
        if ident is None:
            ident = st.n_groups == st.U.shape[0] and bool(
                (st.group_idx == torch.arange(st.n_groups, device=self.device)).all())
            st._groups_identity = ident
        if ident:
            return r
        return r.index_select(0, st.group_idx)

    # ------------------------------------------------------------------
    def _physics_eager(self, U, Uprev, H, Ud, N, R, link_sum, W, m0, m1, m2, g_mu, g_sig, g_bias, g_gain, diff, diss, Q,
                       vy, vx, rate, delay_w, hyst_w, link_coup, res_on, res_khalf, K: int):
        Nev = N[:, :K]
        lap, gyy, gxx = N[:, K], N[:, K + 1], N[:, K + 2]
        A = torch.einsum("bckd,bkdhw->bchw", W, Nev)
        x = A + g_bias
        f1 = torch.tanh(x)
        z = (x - g_mu) / g_sig
        f2 = 2.0 * torch.exp(-0.5 * z * z) - 1.0
        f3 = 2.33 * x * torch.exp(-x * x)
        growth = g_gain * (m0 * f1 + m1 * f2 + m2 * f3)
        lim = R / (R + res_khalf)
        gpos = torch.relu(growth)
        growth = res_on * (gpos * lim - torch.relu(-growth)) + (1 - res_on) * growth
        react = U * torch.einsum("bcd,bdhw->bchw", Q, U)
        f = (growth + react + diff * lap - (vy * gyy + vx * gxx) + link_coup * link_sum
             + hyst_w * H + delay_w * Ud - diss * U)
        return rate * f, gpos

    def step(self, gb: GenomeBatch, st: SimState) -> SimState:
        U = st.U
        B, C, S, _ = U.shape
        P, Hp, K = gb.P, gb.Hp, gb.K
        Up = self.pad(U, gb, P)
        Uh = torch.fft.rfft2(Up)
        N = torch.fft.irfft2(Uh.unsqueeze(1) * gb.Khat.unsqueeze(2), s=(Hp, Hp))[..., P:P + S, P:P + S]
        # topologia de longo alcance (campo plástico G)
        link_sum = torch.zeros_like(U)
        new_G = st.G
        if gb.has_links and st.G is not None:
            new_G = st.G.clone()
            for l in range(gb.L):
                if not gb.link_any[l]:
                    continue
                src = self.link_source(U, gb, l)
                on = gb.link_on[:, l].view(B, 1, 1, 1)
                link_sum = link_sum + on * st.G[:, l:l + 1] * src
                if gb.has_plastic:
                    co = (U * src).mean(1, keepdim=True)
                    gl = st.G[:, l:l + 1] + gb.link_eta * (co - gb.link_decay * (st.G[:, l:l + 1] - gb.link_w0[:, l:l + 1]))
                    new_G[:, l:l + 1] = gl.clamp(-2.0, 2.0) * on
        if gb.has_delay and st.Dbuf is not None:
            Dn = st.Dbuf.shape[0]
            idx = ((st.ptr - 1 - gb.delay_k) % Dn).view(1, B, C, 1, 1).expand(1, B, C, S, S)
            Ud = torch.gather(st.Dbuf, 0, idx).squeeze(0)
        else:
            Ud = U
        ctx = torch.autocast(device_type="cuda", dtype=self.autocast_dtype) if self.autocast_dtype is not None else _nullctx()
        args = (U, st.Uprev, st.H, Ud, N, st.R, link_sum, gb.W, gb.m0, gb.m1, gb.m2, gb.g_mu, gb.g_sig, gb.g_bias,
                gb.g_gain, gb.diff, gb.diss, gb.Q, gb.vy, gb.vx, gb.rate, gb.delay_w, gb.hyst_w, gb.link_coup, gb.res_on,
                gb.res_khalf, K)
        # torch.compile só é usado no tamanho fixo do batch de triagem (formas estáticas, sem recompilações
        # a cada probe); os probes, de tamanhos variados, usam o modo eager.
        compiled = self._physics is not self._physics_eager
        fn = self._physics if (compiled and B == self.cfg.batch_size) else self._physics_eager
        try:
            with ctx:
                dU, gpos = fn(*args)
        except Exception as e:
            if fn is self._physics_eager or is_oom_error(e):
                raise
            print(f"[sim] torch.compile falhou em execução ({type(e).__name__}); voltando ao modo eager")
            self._physics = self._physics_eager
            with ctx:
                dU, gpos = self._physics_eager(*args)
        dU = dU.float(); gpos = gpos.float()
        # flags calculadas uma vez por tensor (evita sincronizar GPU->CPU a cada passo)
        if getattr(st, "_ext_src", None) is not st.ext_noise:
            st._ext_src, st._ext_any = st.ext_noise, bool((st.ext_noise > 0).any())
        if getattr(st, "_shock_src", None) is not st.shock_period:
            st._shock_src, st._shock_np = st.shock_period, st.shock_period.cpu().numpy()
        if gb.has_noise or st._ext_any:
            noise_amp = gb.noise * gb.rate.mean(1, keepdim=True).sqrt() + st.ext_noise
            dU = dU + noise_amp * self._grand(st, (C, S, S))
        # choque ambiental periódico (perturbação externa do ambiente)
        sp = st._shock_np
        if (sp > 0).any():
            hit_np = (sp > 0) & ((st.t + 1) % np.maximum(sp, 1) == 0)
            if hit_np.any():
                hit = torch.as_tensor(hit_np, device=self.device, dtype=torch.float32).view(B, 1, 1, 1)
                dU = dU + hit * st.shock_amp * self._grand(st, (C, S, S))
        # dt adaptativo (salvaguarda numérica externa)
        amax = dU.abs().amax(dim=(1, 2, 3), keepdim=True)
        scale = torch.clamp(gb.max_rate / (amax + 1e-12), max=1.0)
        dU = dU * scale
        st.dt_scale_sum = st.dt_scale_sum + scale.view(B)
        if gb.has_async:
            m = (self._grand(st, (1, S, S), "uniform") < gb.async_p).float()
            dU = dU * m
        dU = dU - gb.conserve * dU.mean(dim=(2, 3), keepdim=True)
        Unew = U + dU
        if gb.has_inertia:
            Unew = Unew + gb.inertia * (U - st.Uprev)
        Unew = gb.sat * torch.tanh(Unew / gb.sat)
        Unew = Unew * gb.chm
        bad = ~torch.isfinite(Unew).flatten(1).all(1)
        st.bad = st.bad | bad
        Unew = torch.where(bad.view(B, 1, 1, 1), torch.zeros_like(Unew), Unew)
        Unew = torch.nan_to_num(Unew, nan=0.0, posinf=0.0, neginf=0.0).clamp(-self.cfg.state_bound, self.cfg.state_bound)
        # traço lento (histerese)
        if gb.has_hyst:
            st.H = st.H + gb.hyst_rate * (Unew - st.H)
        # recurso abstrato
        if gb.has_res:
            act = gpos.mean(1, keepdim=True)
            Rp = self.pad(st.R, gb, 1)
            Rlap = (Rp[:, :, :-2, 1:-1] + Rp[:, :, 2:, 1:-1] + Rp[:, :, 1:-1, :-2] + Rp[:, :, 1:-1, 2:] - 4 * st.R)
            Rn = st.R + gb.res_rho * (st.R0 - st.R) - gb.res_cons * act * st.R + gb.res_diff * Rlap
            st.R = torch.where(gb.res_on > 0, Rn.clamp(0.0, 4.0), st.R)
        if st.Dbuf is not None:
            st.Dbuf[st.ptr] = Unew
            st.ptr = (st.ptr + 1) % st.Dbuf.shape[0]
        st.Uprev = U
        st.U = Unew
        st.G = new_G
        st.t += 1
        self.steps_done += B
        m = LIVE.monitor
        if m is not None:
            m.on_step(self, gb, st)
        return st

    def run(self, gb: GenomeBatch, st: SimState, n: int, callback: Optional[Callable] = None) -> SimState:
        for _ in range(int(n)):
            st = self.step(gb, st)
            if callback is not None:
                callback(st)
        return st


class _nullctx:
    def __enter__(self):
        return self

    def __exit__(self, *a):
        return False

## 5. Observação: descritores comportamentais, triagem populacional e decodificadores externos
A triagem (fase rápida) roda cada genoma em vários ambientes de **evolução**, cada um com uma cópia contrafactual
(`U_A(0)=U_B(0)`, perturbação local apenas em B, **mesmo ruído** nas duas cópias) e mede, sem olhar médias apenas:
atividade, variância, autocorrelação temporal e espacial, espectros, periodicidade, entropia (proxy), compressibilidade,
número aproximado de estruturas (componentes conexas por propagação de rótulos), persistência de identidade e
mobilidade (correlação cruzada normalizada com deslocamento), propagação causal e divergência (proxy de Lyapunov).

Decodificadores são **instrumentos científicos externos**: ridge linear (primário, conservador) e kNN (não linear),
sempre com validação cruzada estratificada, teste de permutação, acurácia balanceada e informação mútua com correção de viés.

In [ ]:
# @title 5. Observação: descritores comportamentais, triagem populacional e decodificadores externos { display-mode: "form" }
DESCRIPTOR_NAMES = ["rel_activity", "variance", "tac1", "sac", "spatial_spectrum", "dominant_freq", "periodicity",
                    "mobility", "structures", "persistence", "causal_spread", "lyapunov", "entropy",
                    "compressibility", "saturation", "magnitude", "temporal_complexity", "spatial_complexity"]


def count_components(mask: torch.Tensor, max_iters: int) -> torch.Tensor:
    """Componentes conexas aproximadas (vizinhança 8) por propagação de rótulo máximo. mask [N,m,m] bool."""
    N, m, _ = mask.shape
    ids = torch.arange(1, m * m + 1, device=mask.device, dtype=torch.float32).view(1, m, m)
    mf = mask.float()
    lab = ids * mf
    for i in range(max_iters):
        new = F.max_pool2d(lab.unsqueeze(1), 3, 1, 1).squeeze(1) * mf
        if i % 8 == 7 and torch.equal(new, lab):
            lab = new
            break
        lab = new
    roots = (lab == ids) & mask
    return roots.flatten(1).sum(1).float()


def compute_descriptors(Fr: torch.Tensor, Dm: Optional[torch.Tensor], D0: Optional[torch.Tensor], chm: torch.Tensor,
                        sat: torch.Tensor, S: int, rec_every: int, steps_after: int) -> Dict[str, np.ndarray]:
    """Fr: [T,B,C,m,m] quadros registrados; Dm: [T,B,m,m] |U_A-U_B| (contrafactual); chm,sat: [B,C]."""
    T, B, C, m, _ = Fr.shape
    eps = 1e-8
    dev = Fr.device
    nch = chm.sum(1).clamp(min=1)
    absmean = Fr.abs().sum(2).mean((0, 2, 3)) / nch
    dF = (Fr[1:] - Fr[:-1]).abs().sum(2).mean((2, 3)) / nch
    activity = dF.mean(0)
    rel_act = activity / (absmean + 1e-6)
    mag = Fr.pow(2).sum(2).sqrt()
    variance = mag.var(dim=(2, 3)).mean(0)
    X = Fr - Fr.mean(0, keepdim=True)

    def tac(lag):
        if T <= lag:
            return torch.ones(B, device=dev)
        num = (X[lag:] * X[:-lag]).sum((0, 2, 3, 4))
        den = torch.sqrt((X[lag:] ** 2).sum((0, 2, 3, 4)) * (X[:-lag] ** 2).sum((0, 2, 3, 4)))
        return torch.where(den > 1e-10, num / (den + eps), torch.ones_like(num))
    tac1 = tac(1)
    M = mag - mag.mean((2, 3), keepdim=True)
    den = (M * M).sum((0, 2, 3))
    sac = ((M * M.roll(1, 2)).sum((0, 2, 3)) + (M * M.roll(1, 3)).sum((0, 2, 3))) / (2 * den + eps)
    sac = torch.where(den > 1e-10, sac, torch.zeros_like(sac))
    Pw = (torch.fft.fft2(M).abs() ** 2).mean(0)
    fy = torch.fft.fftfreq(m, device=dev).abs()
    kr = torch.sqrt(fy.view(m, 1) ** 2 + fy.view(1, m) ** 2); kr = kr / kr.max()
    spec_c = (Pw * kr).sum((1, 2)) / (Pw.sum((1, 2)) + eps)
    spec_c = torch.where(Pw.sum((1, 2)) > 1e-10, spec_c, torch.zeros_like(spec_c))
    Mt = mag - mag.mean(0, keepdim=True)
    Pt = (torch.fft.rfft(Mt, dim=0).abs() ** 2)[1:]
    Fq = max(2, Pt.shape[0])
    Ptot = Pt.sum((2, 3))
    tot = Ptot.sum(0)
    has_t = tot > 1e-10
    dom = (Ptot.argmax(0).float() + 1) / Fq
    periodicity = torch.where(has_t, Ptot.max(0).values / (tot + eps), torch.zeros_like(tot))
    pc = Pt / (Pt.sum(0, keepdim=True) + eps)
    ent_c = -(pc * torch.log(pc + 1e-12)).sum(0) / math.log(Fq)
    wcell = Pt.sum(0)
    tcell = (ent_c * wcell).sum((1, 2)) / (wcell.sum((1, 2)) + eps)
    temporal_complexity = torch.where(has_t, tcell * tac1.clamp(0, 1), torch.zeros_like(tcell))
    # entropia (proxy) do histograma de magnitude
    nb = 16
    smag = torch.sqrt((sat ** 2 * chm).sum(1)).clamp(min=1e-3)
    q = (mag / smag.view(1, B, 1, 1)).clamp(0, 0.9999) * nb
    idx = q.long().permute(1, 0, 2, 3).reshape(B, -1)
    hist = torch.zeros(B, nb, device=dev).scatter_add_(1, idx, torch.ones_like(idx, dtype=torch.float32))
    pp = hist / hist.sum(1, keepdim=True)
    entropy = -(pp * torch.log(pp + 1e-12)).sum(1) / math.log(nb)
    spatial_complexity = entropy * sac.clamp(0, 1)
    satm = (Fr.abs() > 0.95 * sat.view(1, B, C, 1, 1)) & (chm.view(1, B, C, 1, 1) > 0)
    saturation = satm.float().sum((0, 2, 3, 4)) / (T * nch * m * m)
    # persistência de identidade + mobilidade (NCC com deslocamento)
    lag = max(1, T // 4)
    A_, B_ = M[:-lag], M[lag:]
    xc = torch.fft.ifft2(torch.conj(torch.fft.fft2(A_)) * torch.fft.fft2(B_)).real
    nA = torch.sqrt((A_ * A_).sum((2, 3))); nB = torch.sqrt((B_ * B_).sum((2, 3)))
    ncc = xc / (nA * nB + eps).unsqueeze(-1).unsqueeze(-1)
    peak, arg = ncc.flatten(2).max(2)
    valid = (nA > 1e-5) & (nB > 1e-5)
    peak = torch.where(valid, peak, torch.zeros_like(peak))
    dy = (arg // m).float(); dx = (arg % m).float()
    dy = torch.where(dy > m // 2, dy - m, dy); dx = torch.where(dx > m // 2, dx - m, dx)
    disp = torch.sqrt(dy ** 2 + dx ** 2) * (S / m) / (lag * rec_every)
    moving = (peak > 0.5) & (disp > 0)
    vel = (disp * moving).sum(0) / moving.sum(0).clamp(min=1)
    dirc = torch.sqrt((dy * moving).sum(0) ** 2 + (dx * moving).sum(0) ** 2) / (
        (torch.sqrt(dy ** 2 + dx ** 2) * moving).sum(0) + eps)
    mobility = (vel / 0.25).clamp(0, 1) * moving.float().mean(0)
    persistence = peak.mean(0).clamp(0, 1) * (variance > 1e-7).float()
    # estruturas
    last = mag[-2:] if T >= 2 else mag[-1:]
    mm = last.mean((2, 3), keepdim=True); ss = last.std((2, 3), keepdim=True)
    mask = (last > mm + ss) & (last > 0.05 * smag.view(1, B, 1, 1))
    nc = count_components(mask.reshape(-1, m, m), 2 * m).view(-1, B).mean(0)
    structures = nc
    # causalidade
    if Dm is not None:
        Dend = Dm[-1]
        thr = 1e-3 + 1e-2 * absmean.view(B, 1, 1)
        spread = (Dend > thr).float().mean((1, 2))
        Dmean_end = Dm[-1].mean((1, 2))
        lyap = (torch.log(Dmean_end + 1e-12) - torch.log(D0 + 1e-12)) / max(1, steps_after)
        c = (m - 1) / 2.0
        yy, xx = torch.meshgrid(torch.arange(m, device=dev), torch.arange(m, device=dev), indexing="ij")
        dist = torch.sqrt((yy - c) ** 2 + (xx - c) ** 2).float() * (S / m)
        rmax = (dist.unsqueeze(0) * (Dend > thr).float()).amax((1, 2))
        speed = rmax / max(1, steps_after)
    else:
        spread = lyap = speed = torch.zeros(B, device=dev)
    # compressibilidade (zlib) — CPU
    qm = (mag[-4:] / (mag[-4:].amax((0, 2, 3), keepdim=True) + 1e-9) * 255).clamp(0, 255).to(torch.uint8)
    qm = qm.permute(1, 0, 2, 3).contiguous().cpu().numpy()
    comp = np.array([len(zlib.compress(qm[b].tobytes(), 6)) / max(1, qm[b].size) for b in range(B)], np.float32)
    out = {
        "rel_activity": rel_act, "activity": activity, "variance": variance, "tac1": tac1, "sac": sac,
        "spatial_spectrum": spec_c, "dominant_freq": dom, "periodicity": periodicity, "mobility": mobility,
        "velocity": vel, "direction_consistency": dirc, "shape_retention": peak.mean(0),
        "structures": structures, "persistence": persistence, "causal_spread": spread, "lyapunov": lyap,
        "spread_speed": speed, "entropy": entropy, "saturation": saturation, "magnitude": absmean,
        "temporal_complexity": temporal_complexity, "spatial_complexity": spatial_complexity,
    }
    res = {k: torch.nan_to_num(v.float(), nan=0.0, posinf=0.0, neginf=0.0).cpu().numpy() for k, v in out.items()}
    res["compressibility"] = comp
    return res


def descriptor_vector(d: Dict[str, float]) -> np.ndarray:
    """Vetor comportamental normalizado ~[0,1] para novelty / MAP / surpresa."""
    sq = lambda x: x / (1.0 + abs(x))
    v = [sq(d["rel_activity"] * 20), math.tanh(d["variance"] * 10), (d["tac1"] + 1) / 2, (d["sac"] + 1) / 2,
         d["spatial_spectrum"], d["dominant_freq"], d["periodicity"], d["mobility"],
         sq(d["structures"] / 4.0), d["persistence"], d["causal_spread"], 1 / (1 + math.exp(-d["lyapunov"] * 30)),
         d["entropy"], min(1.0, d["compressibility"]), d["saturation"], math.tanh(d["magnitude"]),
         d["temporal_complexity"], d["spatial_complexity"]]
    return np.nan_to_num(np.asarray(v, np.float32), nan=0.0)


def classify_viability(d: Dict[str, float], th: Thresholds, bad: bool) -> str:
    if bad:
        return "NUMERICALLY_UNSTABLE"
    if d["saturation"] > th.explosive_saturation:
        return "EXPLOSIVE"
    if d["magnitude"] < th.dead_abs:
        return "DEAD"
    if d["rel_activity"] < th.frozen_rel_activity:
        return "FROZEN"
    if d["tac1"] < th.noise_tac and d["sac"] < th.noise_sac and d["entropy"] > th.noise_entropy:
        return "PURE_NOISE"
    return "VIABLE"


def quality_objectives(d: Dict[str, float], th: Thresholds, probe_bonus: float = 0.0) -> np.ndarray:
    """Várias dimensões independentes (nunca uma única métrica)."""
    frozen = d["rel_activity"] < th.frozen_rel_activity * 10
    chaos = 1.0 / (1.0 + math.exp((d["lyapunov"] - th.chaos_lyap) / 0.01))
    return np.asarray([
        d["persistence"] * (0.5 if frozen else 1.0),
        d["temporal_complexity"],
        d["spatial_complexity"],
        d["causal_spread"] * chaos,
        d["mobility"],
        min(1.0, d["structures"] / 4.0),
        probe_bonus,
    ], np.float32)


QUALITY_WEIGHTS = np.asarray([0.2, 0.2, 0.15, 0.2, 0.1, 0.15, 0.5], np.float32)


class Screener:
    """Triagem populacional rápida, em batch, multi-ambiente, com par contrafactual."""

    def __init__(self, sim: Simulator, cfg: GenesisConfig):
        self.sim, self.cfg = sim, cfg

    def screen(self, genomes: List[Genome], envs: List[Environment], seed_base: int,
               steps: Optional[int] = None, S: Optional[int] = None, n_envs: Optional[int] = None,
               hook: Optional[Callable] = None, env_choice_seed: Optional[int] = None,
               label: Optional[str] = None) -> List[Dict[str, Any]]:
        cfg = self.cfg
        steps = steps or cfg.screen_steps
        S = S or cfg.world_size
        n_envs = min(n_envs or cfg.screen_envs, len(envs))
        rng = np.random.default_rng(derive_seed(seed_base, "envchoice") if env_choice_seed is None else env_choice_seed)
        jobs = []
        for gi, g in enumerate(genomes):
            chosen = rng.choice(len(envs), size=n_envs, replace=False)
            for ei in chosen:
                jobs.append((gi, envs[int(ei)], derive_seed(seed_base, g.genome_id, envs[int(ei)].env_id)))
        per = max(1, cfg.batch_size // 2)
        rows: List[Dict[str, Any]] = []
        n_chunks = max(1, int(math.ceil(len(jobs) / per)))
        for ci, c0 in enumerate(range(0, len(jobs), per)):
            base = None
            if label is not None:
                base = ci * steps
                live_progress(phase=label, total=n_chunks * steps, done=base,
                              detail=f"lote {ci + 1}/{n_chunks} · {len(genomes)} genomas × {n_envs} ambientes "
                                     f"(+ cópia contrafactual de cada universo)")
            rows.extend(self._screen_chunk(genomes, jobs[c0:c0 + per], steps, S, seed_base + c0, hook, base))
        # agrega por genoma (média entre ambientes; instável se qualquer ambiente instável)
        agg: Dict[int, List[Dict[str, Any]]] = {}
        for r in rows:
            agg.setdefault(r["gi"], []).append(r)
        out = []
        for gi in range(len(genomes)):
            rs = agg.get(gi, [])
            d = {k: float(np.mean([r[k] for r in rs])) for k in rs[0].keys() if k not in ("gi", "env", "seed", "bad")}
            d["bad"] = any(r["bad"] for r in rs)
            d["envs"] = [r["env"] for r in rs]
            d["seeds"] = [r["seed"] for r in rs]
            d["viability"] = classify_viability(d, cfg.thresholds, d["bad"])
            d["env_viability"] = [classify_viability(r, cfg.thresholds, r["bad"]) for r in rs]
            out.append(d)
        return out

    def _screen_chunk(self, genomes, jobs, steps, S, run_seed, hook, progress_base: Optional[int] = None):
        cfg, sim = self.cfg, self.sim
        n = len(jobs)
        gl = [genomes[gi] for gi, _, _ in jobs]
        gb = sim.prepare(gl + gl, S)
        envs = [e for _, e, _ in jobs]
        seeds = [s for _, _, s in jobs]
        st = sim.init_state(gb, envs + envs, seeds + seeds, groups=list(range(n)) * 2, run_seed=run_seed)
        burn = min(cfg.screen_burn_in, steps // 2)
        for t in range(burn):
            st = sim.step(gb, st)
            if hook is not None:
                hook(st)
            if progress_base is not None:
                live_progress(done=progress_base + t + 1)
        # perturbação contrafactual local somente na cópia B
        r = 2
        c = S // 2
        delta = torch.zeros_like(st.U[n:])
        delta[:, :, c - r:c + r + 1, c - r:c + r + 1] = 0.5
        st.U[n:] = st.U[n:] + delta * gb.chm[n:]
        m = min(cfg.metric_res, S)
        pool = lambda X: F.adaptive_avg_pool2d(X, m) if m != S else X
        D0 = pool((st.U[:n] - st.U[n:]).abs().sum(1, keepdim=True)).squeeze(1).mean((1, 2))
        frames, dms = [], []
        rest = steps - burn
        for t in range(rest):
            st = sim.step(gb, st)
            if hook is not None:
                hook(st)
            if progress_base is not None:
                live_progress(done=progress_base + burn + t + 1)
            if t % cfg.record_every == cfg.record_every - 1 or t == rest - 1:
                frames.append(pool(st.U[:n]).float())
                dms.append(pool((st.U[:n] - st.U[n:]).abs().sum(1, keepdim=True)).squeeze(1))
        Fr = torch.stack(frames); Dm = torch.stack(dms)
        del frames, dms
        chm = gb.chm[:n].view(n, -1).clone(); sat = gb.sat[:n].view(n, -1).clone()
        bad = (st.bad[:n] | st.bad[n:]).cpu().numpy()
        del st, gb                                   # libera o estado do batch antes da análise
        parts = []
        for a in range(0, n, 256):                   # descritores em sub-lotes: memória limitada mesmo com batch grande
            b = min(n, a + 256)
            parts.append(compute_descriptors(Fr[:, a:b], Dm[:, a:b], D0[a:b], chm[a:b], sat[a:b], S, cfg.record_every, rest))
        desc = {k: np.concatenate([p_[k] for p_ in parts]) for k in parts[0]}
        out = []
        for i in range(n):
            row = {k: float(v[i]) for k, v in desc.items()}
            row.update({"gi": jobs[i][0], "env": jobs[i][1].env_id, "seed": jobs[i][2], "bad": bool(bad[i])})
            out.append(row)
        return out


# ----------------------------------------------------------------------------
# Decodificadores externos (instrumentos)
# ----------------------------------------------------------------------------
def _standardize(Xtr, Xte):
    mu = Xtr.mean(0, keepdims=True); sd = Xtr.std(0, keepdims=True) + 1e-6
    return (Xtr - mu) / sd, (Xte - mu) / sd


def ridge_predict(Xtr, ytr, Xte, n_classes, lam=1.0):
    Xtr, Xte = _standardize(Xtr, Xte)
    Xtr = np.hstack([Xtr, np.ones((Xtr.shape[0], 1))]); Xte = np.hstack([Xte, np.ones((Xte.shape[0], 1))])
    Y = np.eye(n_classes)[ytr]
    n, d = Xtr.shape
    if n < d:   # forma dual
        Kt = Xtr @ Xtr.T
        alpha = np.linalg.solve(Kt + lam * np.eye(n), Y)
        scores = Xte @ (Xtr.T @ alpha)
    else:
        Wt = np.linalg.solve(Xtr.T @ Xtr + lam * np.eye(d), Xtr.T @ Y)
        scores = Xte @ Wt
    return scores.argmax(1)


def knn_predict(Xtr, ytr, Xte, n_classes, k=5):
    Xtr, Xte = _standardize(Xtr, Xte)
    d = ((Xte[:, None, :] - Xtr[None, :, :]) ** 2).sum(-1)
    k = max(1, min(k, Xtr.shape[0]))
    nn = np.argsort(d, axis=1)[:, :k]
    votes = np.zeros((Xte.shape[0], n_classes))
    for j in range(k):
        votes[np.arange(Xte.shape[0]), ytr[nn[:, j]]] += 1.0 / (1 + j * 0.01)
    return votes.argmax(1)


def stratified_folds(y, k, rng):
    folds = np.zeros(len(y), int)
    for c in np.unique(y):
        idx = np.where(y == c)[0]
        rng.shuffle(idx)
        folds[idx] = np.arange(len(idx)) % k
    return folds


def balanced_accuracy(y, yp, n_classes):
    accs = [np.mean(yp[y == c] == c) for c in range(n_classes) if np.any(y == c)]
    return float(np.mean(accs)) if accs else 0.0


def cv_predictions(X, y, n_classes, decoder, k, seed):
    rng = np.random.default_rng(seed)
    k = max(2, min(k, int(np.bincount(y).min()) if len(np.unique(y)) > 1 else 2))
    folds = stratified_folds(y, k, rng)
    yp = np.zeros_like(y)
    for f in range(k):
        tr, te = folds != f, folds == f
        if te.sum() == 0 or tr.sum() == 0:
            continue
        yp[te] = decoder(X[tr], y[tr], X[te], n_classes)
    return yp


def mutual_info_bits(y, yp, n_classes) -> float:
    """MI plug-in da matriz de confusão com correção de Miller–Madow (viés reconhecido)."""
    conf = np.zeros((n_classes, n_classes))
    for a, b in zip(y, yp):
        conf[a, b] += 1
    n = conf.sum()
    if n == 0:
        return 0.0
    pj = conf / n
    px = pj.sum(1, keepdims=True); py = pj.sum(0, keepdims=True)
    nz = pj > 0
    mi = float((pj[nz] * np.log2(pj[nz] / (px @ py)[nz])).sum())
    k_joint, k_x, k_y = nz.sum(), (px > 0).sum(), (py > 0).sum()
    mm = (k_joint - k_x - k_y + 1) / (2 * n * math.log(2))
    return max(0.0, mi - mm)


def decode_report(X: np.ndarray, y: np.ndarray, n_classes: int, n_perm: int, folds: int, seed: int,
                  nonlinear: bool = True) -> Dict[str, Any]:
    """Acurácia balanceada CV (linear e kNN), p-valor por permutação, confusão, MI."""
    X = np.nan_to_num(np.asarray(X, np.float64)); y = np.asarray(y, int)
    chance = 1.0 / n_classes
    out: Dict[str, Any] = {"n": int(len(y)), "chance": chance, "n_classes": n_classes,
                           "method": "balanced acc, stratified CV (ridge linear + kNN), label-permutation null (empirical + Gaussian tail)"}
    if len(np.unique(y)) < 2 or X.shape[0] < 2 * n_classes:
        out.update({"linear_acc": chance, "linear_p": 1.0, "nonlinear_acc": chance, "nonlinear_p": 1.0, "mi_bits": 0.0})
        return out
    if np.all(X.std(0) < 1e-9):
        out.update({"linear_acc": chance, "linear_p": 1.0, "nonlinear_acc": chance, "nonlinear_p": 1.0, "mi_bits": 0.0,
                    "note": "features constantes"})
        return out
    rng = np.random.default_rng(seed)
    for name, dec in [("linear", ridge_predict)] + ([("nonlinear", knn_predict)] if nonlinear else []):
        yp = cv_predictions(X, y, n_classes, dec, folds, seed)
        acc = balanced_accuracy(y, yp, n_classes)
        null = []
        for i in range(n_perm):
            if i % 10 == 0:
                live_progress()
            yperm = rng.permutation(y)
            ypp = cv_predictions(X, yperm, n_classes, dec, folds, seed + 1 + i)
            null.append(balanced_accuracy(yperm, ypp, n_classes))
        null = np.asarray(null)
        p_emp = (np.sum(null >= acc - 1e-12) + 1) / (n_perm + 1)
        # cauda gaussiana da distribuição nula de permutação: permite p pequenos com poucas permutações;
        # só é usada quando a acurácia observada supera TODAS as permutações (senão vale o p empírico).
        sd = max(float(null.std()), 1e-3) if null.size else 1.0
        z = (acc - (float(null.mean()) if null.size else chance)) / sd
        p_par = 0.5 * math.erfc(z / math.sqrt(2))
        p = p_par if (null.size and acc > null.max()) else max(p_emp, p_par)
        out[f"{name}_acc"] = acc
        out[f"{name}_p"] = float(p)
        out[f"{name}_p_empirical"] = float(p_emp)
        out[f"{name}_null_mean"] = float(null.mean()) if null.size else chance
        if name == "linear":
            out["mi_bits"] = mutual_info_bits(y, yp, n_classes)
            conf = np.zeros((n_classes, n_classes), int)
            for a, b in zip(y, yp):
                conf[a, b] += 1
            out["confusion"] = conf.tolist()
            out["effect_size"] = float((acc - out[f"{name}_null_mean"]) / (null.std() + 1e-6)) if null.size else 0.0
    if not nonlinear:
        out["nonlinear_acc"], out["nonlinear_p"] = float("nan"), 1.0
    return out


def norm_acc(acc: float, chance: float) -> float:
    return float((acc - chance) / (1 - chance))

## 6. Validation Engine (congelado e versionado) — biblioteca de probes
O universo **não controla seu avaliador**. Este módulo é separado do motor evolutivo, tem versão própria
(`VE-1.0`), cada probe tem versão, cada resultado guarda o protocolo completo (sementes, ambientes, tempos, tamanho)
para ser reproduzido por `reproduce_discovery`. Resultados ausentes são `NOT_TESTED` (zero significa "testado e falhou").
Estados possíveis: `NOT_TESTED, RUNNING, PASS, FAIL, INCONCLUSIVE, ERROR`. Inconclusivo é um resultado válido.

| probe | pergunta | controles |
|---|---|---|
| causality | perturbação local produz efeito reproduzível e seletivo (não só caos)? | cópia contrafactual com ruído idêntico, repetibilidade entre sementes, seletividade entre 2 estímulos |
| memory | o estado futuro permite inferir qual história ocorreu, após T passos sem estímulo? | grupo nulo (sem estímulo), permutação de rótulos, Holm entre atrasos, controle +/− no boot |
| channel | símbolos aplicados em A são decodificáveis em B (distância d)? | mapeamento símbolo→estímulo aleatorizado, leitura pré-estímulo (vazamento), estímulo nulo |
| transformation | a região de saída torna linearmente decodificável uma função NÃO linear (XOR) das entradas? | baseline linear das entradas (XOR impossível), leitura pré-estímulo |
| composition | A→B e B→C se compõem (paridade de 3 entradas em C, XOR intermediário em M)? | idem |
| generalization | decodificador treinado em condições vistas funciona em amplitudes, posições, sementes e ambientes NÃO vistos? | split evolução/validação; t* escolhido só no treino |
| recovery | estrutura persistente danificada (10–90%) volta em geometria, estado interno e trajetória? | ramo de referência sem dano com ruído idêntico; recuperação relativa ao dano imediato |
| robustness | comportamento sobrevive a ruído, sal-e-pimenta, deleção, drift de parâmetro, dt, borda, estado inicial, tamanho? | baseline com mesmas sementes |
| adaptation | o sistema muda comportamento quando o ambiente muda e mantém função? | condição sem mudança (deriva natural) |
| replication / ablation | o resultado se repete com sementes independentes? o que realmente importa? | genoma congelado; zero evolução durante validação |

In [ ]:
# @title 6. Validation Engine (congelado e versionado) — biblioteca de probes { display-mode: "form" }
@dataclass
class TestResult:
    probe: str
    candidate_id: str
    status: str
    score: float = float("nan")
    p_value: float = float("nan")
    effect_size: float = float("nan")
    ci: Tuple[float, float] = (float("nan"), float("nan"))
    details: Dict[str, Any] = field(default_factory=dict)
    protocol: Dict[str, Any] = field(default_factory=dict)
    version: str = ""
    result_id: str = ""
    timestamp: str = ""
    duration_s: float = 0.0
    split: str = "evolution"
    thresholds_hash: str = ""

    def __post_init__(self):
        if not self.result_id:
            self.result_id = new_id("R")
        if not self.timestamp:
            self.timestamp = iso_now()

    def to_dict(self):
        return to_jsonable(asdict(self))

    @staticmethod
    def from_dict(d):
        d = dict(d)
        d["ci"] = tuple(x if x is not None else float("nan") for x in d.get("ci", (None, None)))
        for k in ["score", "p_value", "effect_size"]:
            if d.get(k) is None:
                d[k] = float("nan")
        return TestResult(**{k: v for k, v in d.items() if k in TestResult.__dataclass_fields__})


class ValidationEngine:
    VERSION = VALIDATION_ENGINE_VERSION
    PROBE_VERSIONS = {"causality": "1.0", "memory": "1.0", "channel": "1.0", "transformation": "1.0",
                      "composition": "1.0", "generalization": "1.0", "recovery": "1.0", "robustness": "1.0",
                      "adaptation": "1.0", "replication": "1.0", "ablation": "1.0", "long_screen": "1.0",
                      "advanced": "1.0"}

    def __init__(self, sim: Simulator, cfg: GenesisConfig, envs: List[Environment], screener: Screener,
                 space: ParamSpace, curriculum: Optional[Dict[str, float]] = None):
        self.sim, self.cfg, self.envs, self.screener, self.space = sim, cfg, envs, screener, space
        self.curriculum = curriculum if curriculum is not None else {}
        self.th = cfg.thresholds
        self.ps = cfg.probe
        self._frozen_hash = cfg.thresholds_hash()

    # ------------------------------------------------------------------
    def envs_for(self, split: str) -> List[Environment]:
        e = [x for x in self.envs if x.split == split]
        return e if e else [x for x in self.envs if x.split == "evolution"]

    def channel_distance(self, S: int) -> int:
        d = int(self.ps.channel_distance + self.curriculum.get("channel_distance_add", 0))
        return int(max(2 * self.ps.stim_radius + 2, min(d, S // 2 - self.ps.obs_radius - 1)))

    def memory_delays(self) -> List[int]:
        mult = self.curriculum.get("memory_delay_mult", 1.0)
        return sorted(set(max(1, int(round(d * mult))) if d > 1 else d for d in self.ps.memory_delays))

    def run_probe(self, name: str, genome: Genome, seed: int, **kw) -> TestResult:
        """Wrapper: tempo, protocolo completo, tratamento de erro (sem falhas silenciosas)."""
        t0 = time.time()
        fn = getattr(self, f"probe_{name}")
        protocol = {"probe": name, "probe_version": self.PROBE_VERSIONS.get(name, "?"), "engine": self.VERSION,
                    "seed": int(seed), "kwargs": to_jsonable(kw), "world_size": kw.get("S", self.cfg.world_size),
                    "curriculum": dict(self.curriculum), "genome_hash": genome.hash(),
                    "device": self.sim.device.type}
        try:
            res: TestResult = fn(genome, seed, **kw)
        except Exception as e:
            if is_oom_error(e):
                raise
            res = TestResult(probe=name, candidate_id=genome.genome_id, status=STATUS_ERROR,
                             details={"error": repr(e), "traceback": traceback.format_exc()[-3000:]})
        res.probe = name
        res.candidate_id = genome.genome_id
        res.version = f"{self.VERSION}/{name}-{self.PROBE_VERSIONS.get(name, '?')}"
        res.protocol = {**protocol, **res.protocol}
        res.duration_s = time.time() - t0
        res.thresholds_hash = self._frozen_hash
        res.split = kw.get("split", res.split)
        return res

    # ------------------------------------------------------------------
    def _setup(self, genome: Genome, n_unique: int, split: str, seed: int, repeat: int = 1, S: Optional[int] = None,
               envs: Optional[List[Environment]] = None, genomes: Optional[List[Genome]] = None):
        S = S or self.cfg.world_size
        pool = envs if envs is not None else self.envs_for(split)
        env_u = [pool[i % len(pool)] for i in range(n_unique)]
        seeds_u = [derive_seed(seed, "trial", i) for i in range(n_unique)]
        gl = genomes if genomes is not None else [genome] * (n_unique * repeat)
        gb = self.sim.prepare(gl, S)
        st = self.sim.init_state(gb, env_u * repeat, seeds_u * repeat, groups=list(range(n_unique)) * repeat,
                                 run_seed=derive_seed(seed, "run"))
        return gb, st, {"S": S, "envs": [e.env_id for e in env_u], "seeds": seeds_u, "split": split}

    @staticmethod
    def _pattern(rng: np.random.Generator, C: int, r: int) -> np.ndarray:
        yy, xx = np.mgrid[-r:r + 1, -r:r + 1]
        disk = ((yy ** 2 + xx ** 2) <= r * r + 0.5).astype(np.float32)
        p = rng.choice([-1.0, 1.0], size=(C, 2 * r + 1, 2 * r + 1)).astype(np.float32)
        return p * disk[None]

    def _stim_field(self, B: int, C: int, S: int, items: List[Tuple[int, np.ndarray, int, int, float]]) -> np.ndarray:
        fld = np.zeros((B, C, S, S), np.float32)
        for (b, pat, cy, cx, amp) in items:
            r = pat.shape[-1] // 2
            ys = (np.arange(-r, r + 1) + cy) % S; xs = (np.arange(-r, r + 1) + cx) % S
            fld[b][:, ys[:, None], xs[None, :]] += amp * pat
        return fld

    def _region_feats(self, U: torch.Tensor, cy: int, cx: int, r: int, pool: int = 3) -> np.ndarray:
        S = U.shape[-1]
        ys = torch.as_tensor((np.arange(-r, r + 1) + cy) % S, device=U.device)
        xs = torch.as_tensor((np.arange(-r, r + 1) + cx) % S, device=U.device)
        crop = U.index_select(2, ys).index_select(3, xs)
        f = F.adaptive_avg_pool2d(crop, min(pool, 2 * r + 1))
        return f.flatten(1).float().cpu().numpy()

    def _global_feats(self, U: torch.Tensor, pool: int = 6) -> np.ndarray:
        return F.adaptive_avg_pool2d(U, min(pool, U.shape[-1])).flatten(1).float().cpu().numpy()

    def _run_protocol(self, genome: Genome, units: List[Dict[str, Any]], readouts: List[Dict[str, Any]], T_total: int,
                      seed: int, S: Optional[int] = None, noise_level: float = 0.0,
                      env_change: Optional[Dict[str, Any]] = None, damage: Optional[Dict[str, Any]] = None,
                      warmup: Optional[int] = None):
        """Motor genérico de protocolo estímulo→leitura.
        units: [{env, seed, group, stims:[(t, pattern, cy, cx, amp)]}] ; readouts: [{name, kind, cy, cx, r, times}]."""
        S = S or self.cfg.world_size
        B = len(units)
        gb = self.sim.prepare([genome] * B, S)
        st = self.sim.init_state(gb, [u["env"] for u in units], [u["seed"] for u in units],
                                 groups=[u["group"] for u in units], run_seed=derive_seed(seed, "protocol-run"))
        C = gb.C
        st = self.sim.run(gb, st, self.ps.warmup if warmup is None else warmup)
        if env_change is not None and env_change.get("t", 0) < 0:
            self._apply_env_change(st, env_change, units)
        if damage is not None:
            rng = np.random.default_rng(derive_seed(seed, "damage"))
            mask = np.zeros((B, 1, S, S), bool)
            y0, y1, x0, x1 = damage["box"]
            sub = rng.random((B, 1, y1 - y0, x1 - x0)) < damage["frac"]
            mask[:, :, y0:y1, x0:x1] = sub
            st.damage_cells(torch.as_tensor(mask, device=self.sim.device))
            st = self.sim.run(gb, st, damage.get("recover", self.ps.recovery_steps))
        if noise_level > 0:
            st.ext_noise = st.ext_noise + noise_level
        sched: Dict[int, List] = {}
        for b, u in enumerate(units):
            for (t, pat, cy, cx, amp) in u.get("stims", []):
                sched.setdefault(int(t), []).append((b, pat, cy, cx, amp))
        feats: Dict[str, Dict[int, np.ndarray]] = {ro["name"]: {} for ro in readouts}
        times_needed = sorted(set(t for ro in readouts for t in ro["times"]))
        last_t = max([T_total] + times_needed)
        act_last = None
        for t in range(last_t + 1):
            if env_change is not None and env_change.get("t", -1) == t:
                self._apply_env_change(st, env_change, units)
            for ro in readouts:
                if t in ro["times"]:
                    if ro["kind"] == "global":
                        feats[ro["name"]][t] = self._global_feats(st.U)
                    else:
                        feats[ro["name"]][t] = self._region_feats(st.U, ro["cy"], ro["cx"], ro["r"], ro.get("pool", 3))
            if t in sched:
                fld = self._stim_field(B, C, S, sched[t])
                st.U = st.U + torch.as_tensor(fld, device=self.sim.device) * gb.chm
            if t == last_t:
                break
            Uold = st.U
            st = self.sim.step(gb, st)
            if t == last_t - 1:
                act_last = (st.U - Uold).abs().mean().item()
        return feats, {"bad": st.bad.cpu().numpy().tolist(), "final_activity": act_last,
                       "magnitude": st.U.abs().mean().item()}

    def _apply_env_change(self, st: SimState, ch: Dict[str, Any], units):
        sel = torch.as_tensor([1.0 if u.get("changed", True) else 0.0 for u in units], device=self.sim.device).view(-1, 1, 1, 1)
        st.R0 = st.R0 * (1 - sel) + st.R0 * ch.get("resource_mult", 0.5) * sel
        st.ext_noise = st.ext_noise + sel * ch.get("noise_add", 0.01)

    def _series(self, feats_t: Dict[int, np.ndarray], idx: np.ndarray, y: np.ndarray, n_classes: int, seed: int,
                times: Optional[List[int]] = None) -> List[Dict[str, Any]]:
        out = []
        for t in (times or sorted(feats_t.keys())):
            rep = decode_report(feats_t[t][idx], y, n_classes, self.ps.permutations, self.ps.folds, derive_seed(seed, t))
            rep["t"] = int(t)
            out.append(rep)
        adj = holm_correction([r["linear_p"] for r in out])
        for r, a in zip(out, adj):
            r["p_holm"] = a
            r["norm_acc"] = norm_acc(r["linear_acc"], r["chance"])
            r["sig"] = bool(a < self.th.alpha and r["norm_acc"] >= self.th.min_norm_acc)
            r["nonlinear_only"] = bool((not r["sig"]) and r.get("nonlinear_p", 1.0) < self.th.alpha / max(1, len(out)))
        return out

    # ------------------------------------------------------------------
    # LEVEL 2 — CAUSALIDADE
    # ------------------------------------------------------------------
    def probe_causality(self, genome: Genome, seed: int, split: str = "evolution", trials: Optional[int] = None,
                        steps: Optional[int] = None, S: Optional[int] = None) -> TestResult:
        n = trials or self.ps.causal_trials; T = steps or self.ps.causal_steps
        gb, st, prot = self._setup(genome, n, split, seed, repeat=3, S=S)
        S = gb.S; C = gb.C
        rng = np.random.default_rng(derive_seed(seed, "causal-patterns"))
        r = self.ps.stim_radius
        p1, p2 = self._pattern(rng, C, r), self._pattern(rng, C, r)
        st = self.sim.run(gb, st, self.ps.warmup)
        c = S // 2
        items = [(n + i, p1, c, c, self.ps.stim_amp) for i in range(n)] + [(2 * n + i, p2, c, c, self.ps.stim_amp) for i in range(n)]
        st.U = st.U + torch.as_tensor(self._stim_field(gb.B, C, S, items), device=self.sim.device) * gb.chm
        D_start = (st.U[n:2 * n] - st.U[:n]).abs().sum(1).mean().item()
        m = min(self.cfg.metric_res, S)
        maps = []
        snap_every = max(1, T // 8)
        for t in range(T):
            st = self.sim.step(gb, st)
            if t % snap_every == snap_every - 1:
                maps.append(F.adaptive_avg_pool2d((st.U[n:n + 1] - st.U[:1]).abs().sum(1, keepdim=True), m)[0, 0].cpu().numpy())
        U0, U1, U2 = st.U[:n], st.U[n:2 * n], st.U[2 * n:]
        D1 = (U1 - U0).abs().sum(1)
        yy, xx = np.mgrid[0:S, 0:S]
        outside = torch.as_tensor(np.sqrt((yy - c) ** 2 + (xx - c) ** 2) > r + 2, device=self.sim.device)
        thr = 1e-3 + 1e-2 * U0.abs().mean().item()
        spread = ((D1 > thr) & outside).float().sum((1, 2)) / outside.float().sum()
        strength = float(spread.mean().item())
        Dm = F.adaptive_avg_pool2d(D1.unsqueeze(1), m).flatten(1).cpu().numpy()
        Dn = Dm - Dm.mean(1, keepdims=True); Dn = Dn / (np.linalg.norm(Dn, axis=1, keepdims=True) + 1e-12)
        cm = Dn @ Dn.T
        rep_corr = float((cm.sum() - np.trace(cm)) / max(1, n * (n - 1)))
        Xs = np.concatenate([self._global_feats(U1 - U0), self._global_feats(U2 - U0)])
        ys = np.array([0] * n + [1] * n)
        sel = decode_report(Xs, ys, 2, self.ps.permutations, self.ps.folds, derive_seed(seed, "sel"))
        D_end = D1.mean().item()
        lyap = (math.log(D_end + 1e-12) - math.log(D_start + 1e-12)) / T
        if not np.isfinite(rep_corr):
            rep_corr = 0.0
        sel_ok = sel["linear_p"] < self.th.alpha and norm_acc(sel["linear_acc"], 0.5) >= self.th.min_norm_acc
        if bool(st.bad.any()):
            status = STATUS_ERROR
        elif strength < self.th.causal_spread_probe:
            status = STATUS_FAIL
        elif rep_corr >= self.th.causal_repeatability or sel_ok:
            status = STATUS_PASS
        else:
            status = STATUS_INCONCLUSIVE
        score = float(min(1.0, strength * 3) * max(rep_corr, norm_acc(sel["linear_acc"], 0.5), 0.0))
        lo, hi = bootstrap_ci(spread.cpu().numpy())
        return TestResult(probe="causality", candidate_id=genome.genome_id, status=status, score=score,
                          p_value=sel["linear_p"], effect_size=sel.get("effect_size", float("nan")), ci=(lo, hi),
                          details={"spread_mean": strength, "spread_per_trial": spread.cpu().numpy().tolist(),
                                   "repeatability": rep_corr, "selectivity_acc": sel["linear_acc"],
                                   "selectivity_p": sel["linear_p"], "lyapunov_like": lyap,
                                   "chaos_suspected": bool(strength >= self.th.causal_spread_probe and rep_corr < 0.2 and not sel_ok),
                                   "causal_map": [mp.astype(np.float16) for mp in maps], "map_times": [snap_every * (i + 1) for i in range(len(maps))]},
                          protocol={**prot, "trials": n, "steps": T}, split=split)

    # ------------------------------------------------------------------
    # LEVEL 3 — MEMÓRIA
    # ------------------------------------------------------------------
    def probe_memory(self, genome: Genome, seed: int, split: str = "evolution", trials: Optional[int] = None,
                     delays: Optional[List[int]] = None, S: Optional[int] = None) -> TestResult:
        n = trials or self.ps.trials
        delays = sorted(delays or self.memory_delays())
        S = S or self.cfg.world_size
        C = self.cfg.channels_max
        rng = np.random.default_rng(derive_seed(seed, "memory-patterns"))
        r = self.ps.stim_radius + 1
        pats = [self._pattern(rng, C, r), self._pattern(rng, C, r)]
        pool = self.envs_for(split)
        c = S // 2
        H = self.ps.memory_history_steps
        units = []
        labels = []
        for i in range(4 * n):
            lab = i % 2
            stim = i < 2 * n
            stims = [(h, pats[lab], c, c, self.ps.stim_amp * 0.5) for h in range(H)] if stim else []
            units.append({"env": pool[(i // 2) % len(pool)], "seed": derive_seed(seed, "mem", i), "group": i, "stims": stims})
            labels.append(lab)
        times = [H + d for d in delays]
        feats, info = self._run_protocol(genome, units, [{"name": "g", "kind": "region", "cy": c, "cx": c,
                                                          "r": r + 2, "pool": 4, "times": times}],
                                         max(times), seed, S=S)
        y = np.asarray(labels)
        idx_s, idx_n = np.arange(2 * n), np.arange(2 * n, 4 * n)
        ser = self._series(feats["g"], idx_s, y[idx_s], 2, derive_seed(seed, "ms"), times)
        nul = self._series(feats["g"], idx_n, y[idx_n], 2, derive_seed(seed, "mn"), times)
        curve = []
        for d, a, b in zip(delays, ser, nul):
            curve.append({"T": int(d), "acc": a["linear_acc"], "p_holm": a["p_holm"], "sig": a["sig"],
                          "nonlinear_acc": a.get("nonlinear_acc"), "null_acc": b["linear_acc"], "null_sig": b["sig"],
                          "mi_bits": a.get("mi_bits", 0.0)})
        long_sig = [cv for cv in curve if cv["sig"] and cv["T"] >= self.th.memory_min_delay]
        any_sig = [cv for cv in curve if cv["sig"]]
        leak = any(cv["null_sig"] for cv in curve)
        nonlin_only = any(r_["nonlinear_only"] for r_ in ser if r_["t"] - H >= self.th.memory_min_delay)
        if any(info["bad"]):
            status = STATUS_ERROR
        elif leak:
            status = STATUS_INCONCLUSIVE
        elif long_sig:
            status = STATUS_PASS
        elif nonlin_only:
            status = STATUS_INCONCLUSIVE
        else:
            status = STATUS_FAIL
        score = max([norm_acc(cv["acc"], 0.5) for cv in long_sig], default=0.0)
        best = max(ser, key=lambda r_: r_["linear_acc"])
        static = info["final_activity"] is not None and info["final_activity"] < 1e-6
        return TestResult(probe="memory", candidate_id=genome.genome_id, status=status, score=float(score),
                          p_value=float(min([cv["p_holm"] for cv in curve], default=1.0)),
                          effect_size=float(best.get("effect_size", float("nan"))),
                          details={"curve": curve, "horizon": max([cv["T"] for cv in any_sig], default=0),
                                   "leakage_detected": leak, "short_term_only": bool(any_sig and not long_sig),
                                   "static_storage": bool(static), "nonlinear_only": nonlin_only,
                                   "decoder": "ridge (primary), kNN (reported)", "mi_method": "plug-in + Miller-Madow"},
                          protocol={"S": S, "trials_per_class": n, "delays": delays, "history_steps": H, "split": split,
                                    "envs": [e.env_id for e in pool]}, split=split)

    # ------------------------------------------------------------------
    # LEVEL 4 — CANAL
    # ------------------------------------------------------------------
    def _channel_geometry(self, S: int):
        d = self.channel_distance(S)
        c = S // 2
        return d, (c, c - d // 2), (c, c - d // 2 + d)

    def probe_channel(self, genome: Genome, seed: int, split: str = "evolution", trials: Optional[int] = None,
                      noise_level: float = 0.0, pre_damage: float = 0.0, S: Optional[int] = None,
                      env_change: Optional[Dict[str, Any]] = None) -> TestResult:
        n = trials or self.ps.trials
        S = S or self.cfg.world_size
        C = self.cfg.channels_max
        nsym = self.ps.channel_symbols
        rng = np.random.default_rng(derive_seed(seed, "channel"))
        r = self.ps.stim_radius
        pats = [self._pattern(rng, C, r) for _ in range(nsym)]
        mapping = rng.permutation(nsym)           # controle: correspondência símbolo→estímulo aleatorizada
        d, (ay, ax), (by, bx) = self._channel_geometry(S)
        pool = self.envs_for(split)
        n_null = max(2, n // 2)
        units, labels, is_null = [], [], []
        P = self.ps.stim_pulses
        for i in range(nsym * n + nsym * n_null):
            null = i >= nsym * n
            s = (i % nsym)
            pat = pats[mapping[0]] if null else pats[mapping[s]]
            units.append({"env": pool[(i // nsym) % len(pool)], "seed": derive_seed(seed, "ch", i), "group": i,
                          "stims": [(k + 1, pat, ay, ax, self.ps.stim_amp) for k in range(P)]})
            labels.append(s); is_null.append(null)
        times = sorted(set([0] + [t + 1 for t in self.ps.channel_times if t > 0]))
        damage = None
        if pre_damage > 0:
            y0 = max(0, ay - 2 * r); y1 = min(S, ay + 2 * r + 1)
            damage = {"box": (y0, y1, max(0, ax), min(S, bx + 1)), "frac": pre_damage, "recover": self.ps.recovery_steps}
        feats, info = self._run_protocol(genome, units, [{"name": "B", "kind": "region", "cy": by, "cx": bx,
                                                          "r": self.ps.obs_radius, "times": times}],
                                         max(times), seed, S=S, noise_level=noise_level, damage=damage,
                                         env_change=env_change)
        y = np.asarray(labels); nm = np.asarray(is_null)
        idx_s, idx_n = np.where(~nm)[0], np.where(nm)[0]
        ser = self._series(feats["B"], idx_s, y[idx_s], nsym, derive_seed(seed, "cs"), times)
        nul = self._series(feats["B"], idx_n, y[idx_n], nsym, derive_seed(seed, "cn"), times)
        pre = ser[0]
        post = [r_ for r_ in ser if r_["t"] > 0]
        sig = [r_ for r_ in post if r_["sig"]]
        leak = pre["sig"] or any(r_["sig"] for r_ in nul)
        best = max(post, key=lambda r_: r_["linear_acc"]) if post else pre
        latency = min([r_["t"] - 1 for r_ in sig], default=None)
        mi = best.get("mi_bits", 0.0)
        if any(info["bad"]):
            status = STATUS_ERROR
        elif leak:
            status = STATUS_INCONCLUSIVE
        elif sig:
            status = STATUS_PASS
        elif any(r_["nonlinear_only"] for r_ in post):
            status = STATUS_INCONCLUSIVE
        else:
            status = STATUS_FAIL
        return TestResult(probe="channel", candidate_id=genome.genome_id, status=status,
                          score=float(max(0.0, best["norm_acc"]) if sig else 0.0), p_value=float(best["p_holm"]),
                          effect_size=float(best.get("effect_size", float("nan"))),
                          details={"series": [{k: r_[k] for k in ["t", "linear_acc", "p_holm", "sig", "nonlinear_acc", "mi_bits"] if k in r_} for r_ in ser],
                                   "null_series": [{"t": r_["t"], "acc": r_["linear_acc"], "sig": r_["sig"]} for r_ in nul],
                                   "latency": latency, "distance": d, "mi_bits": mi,
                                   "capacity_bits_per_step": (mi / max(1, latency)) if latency is not None else 0.0,
                                   "confusion": best.get("confusion"), "leakage_detected": bool(leak),
                                   "noise_level": noise_level, "pre_damage": pre_damage, "mapping": mapping.tolist(),
                                   "best_t": best["t"]},
                          protocol={"S": S, "trials_per_symbol": n, "symbols": nsym, "distance": d, "times": times,
                                    "split": split, "A": (ay, ax), "B": (by, bx)}, split=split)

    # ------------------------------------------------------------------
    # LEVEL 5 — TRANSFORMAÇÃO
    # ------------------------------------------------------------------
    def probe_transformation(self, genome: Genome, seed: int, split: str = "evolution", trials: Optional[int] = None,
                             S: Optional[int] = None, amp_scale: float = 1.0) -> TestResult:
        n = trials or self.ps.trials
        S = S or self.cfg.world_size
        C = self.cfg.channels_max
        rng = np.random.default_rng(derive_seed(seed, "transform"))
        r = self.ps.stim_radius
        P1, P2 = self._pattern(rng, C, r), self._pattern(rng, C, r)
        d, (ay, ax), (by, bx) = self._channel_geometry(S)
        off = max(r + 1, d // 3)
        a1c, a2c = ((ay - off) % S, ax), ((ay + off) % S, ax)
        pool = self.envs_for(split)
        combos = [(0, 0), (0, 1), (1, 0), (1, 1)]
        units, A1, A2 = [], [], []
        amp = self.ps.stim_amp * amp_scale
        for i in range(4 * n):
            a1, a2 = combos[i % 4]
            st_ = []
            for k in range(self.ps.stim_pulses):
                st_.append((k + 1, P1 * (1 if a1 else -1), a1c[0], a1c[1], amp))
                st_.append((k + 1, P2 * (1 if a2 else -1), a2c[0], a2c[1], amp))
            units.append({"env": pool[(i // 4) % len(pool)], "seed": derive_seed(seed, "tf", i), "group": i, "stims": st_})
            A1.append(a1); A2.append(a2)
        times = sorted(set([0] + [t + 1 for t in self.ps.transform_times]))
        feats, info = self._run_protocol(genome, units, [{"name": "B", "kind": "region", "cy": by, "cx": bx,
                                                          "r": self.ps.obs_radius, "times": times}], max(times), seed, S=S)
        A1, A2 = np.asarray(A1), np.asarray(A2)
        idx = np.arange(4 * n)
        targets = {"XOR": A1 ^ A2, "AND": A1 & A2, "OR": A1 | A2}
        res = {k: self._series(feats["B"], idx, v, 2, derive_seed(seed, k), times) for k, v in targets.items()}
        Xin = np.stack([2 * A1 - 1, 2 * A2 - 1], 1) + np.random.default_rng(seed).normal(0, 0.05, (4 * n, 2))
        base = decode_report(Xin, targets["XOR"], 2, self.ps.permutations, self.ps.folds, seed, nonlinear=False)
        xor_post = [r_ for r_ in res["XOR"] if r_["t"] > 0]
        sig = [r_ for r_ in xor_post if r_["sig"]]
        leak = res["XOR"][0]["sig"]
        base_sig = base["linear_p"] < self.th.alpha
        best = max(xor_post, key=lambda r_: r_["linear_acc"])
        if any(info["bad"]):
            status = STATUS_ERROR
        elif leak or base_sig:
            status = STATUS_INCONCLUSIVE
        elif sig:
            status = STATUS_PASS
        else:
            status = STATUS_FAIL
        return TestResult(probe="transformation", candidate_id=genome.genome_id, status=status,
                          score=float(max(0.0, best["norm_acc"]) if sig else 0.0), p_value=float(best["p_holm"]),
                          effect_size=float(best.get("effect_size", float("nan"))),
                          details={k: [{"t": r_["t"], "acc": r_["linear_acc"], "p_holm": r_["p_holm"], "sig": r_["sig"]} for r_ in v]
                                   for k, v in res.items()} | {"baseline_linear_xor_acc": base["linear_acc"],
                                                              "best_t": best["t"], "leakage_detected": bool(leak)},
                          protocol={"S": S, "trials_per_combo": n, "times": times, "split": split, "amp_scale": amp_scale},
                          split=split)

    # ------------------------------------------------------------------
    # LEVEL 6 — COMPOSIÇÃO
    # ------------------------------------------------------------------
    def probe_composition(self, genome: Genome, seed: int, split: str = "evolution", trials: Optional[int] = None,
                          S: Optional[int] = None, t_mid: int = 8) -> TestResult:
        n = max(3, (trials or self.ps.trials) // 2)
        S = S or self.cfg.world_size
        C = self.cfg.channels_max
        rng = np.random.default_rng(derive_seed(seed, "compose"))
        r = self.ps.stim_radius
        P1, P2, P3 = (self._pattern(rng, C, r) for _ in range(3))
        d = int(max(2 * r + 2, min(self.channel_distance(S), (S - 4 * r) // 3)))
        c = S // 2
        ax = (c - (3 * d) // 2) % S
        mx, cx = (ax + d) % S, (ax + 2 * d) % S
        off = max(r + 1, d // 3)
        pool = self.envs_for(split)
        units, labs = [], []
        for i in range(8 * n):
            a1, a2, a3 = (i % 8) >> 2 & 1, (i % 8) >> 1 & 1, (i % 8) & 1
            st_ = []
            for k in range(self.ps.stim_pulses):
                st_.append((k + 1, P1 * (1 if a1 else -1), (c - off) % S, ax, self.ps.stim_amp))
                st_.append((k + 1, P2 * (1 if a2 else -1), (c + off) % S, ax, self.ps.stim_amp))
                st_.append((t_mid + k + 1, P3 * (1 if a3 else -1), c, mx, self.ps.stim_amp))
            units.append({"env": pool[(i // 8) % len(pool)], "seed": derive_seed(seed, "cp", i), "group": i, "stims": st_})
            labs.append((a1, a2, a3))
        labs = np.asarray(labs)
        times_c = sorted(set(t_mid + 1 + t for t in self.ps.transform_times))
        feats, info = self._run_protocol(genome, units,
                                         [{"name": "M", "kind": "region", "cy": c, "cx": mx, "r": self.ps.obs_radius, "times": [t_mid + 1]},
                                          {"name": "C", "kind": "region", "cy": c, "cx": cx, "r": self.ps.obs_radius, "times": times_c}],
                                         max(times_c), seed, S=S)
        idx = np.arange(8 * n)
        xor12 = labs[:, 0] ^ labs[:, 1]
        par = labs[:, 0] ^ labs[:, 1] ^ labs[:, 2]
        st1 = self._series(feats["M"], idx, xor12, 2, derive_seed(seed, "s1"))
        st2 = self._series(feats["C"], idx, par, 2, derive_seed(seed, "s2"))
        sig1 = any(r_["sig"] for r_ in st1)
        sig2 = [r_ for r_ in st2 if r_["sig"]]
        best = max(st2, key=lambda r_: r_["linear_acc"])
        if any(info["bad"]):
            status = STATUS_ERROR
        elif sig1 and sig2:
            status = STATUS_PASS
        elif sig2 and not sig1:
            status = STATUS_INCONCLUSIVE
        else:
            status = STATUS_FAIL
        return TestResult(probe="composition", candidate_id=genome.genome_id, status=status,
                          score=float(max(0.0, best["norm_acc"]) if (sig1 and sig2) else 0.0), p_value=float(best["p_holm"]),
                          effect_size=float(best.get("effect_size", float("nan"))),
                          details={"stage1_xor_at_M": [{"t": r_["t"], "acc": r_["linear_acc"], "sig": r_["sig"]} for r_ in st1],
                                   "stage2_parity_at_C": [{"t": r_["t"], "acc": r_["linear_acc"], "sig": r_["sig"]} for r_ in st2],
                                   "distance": d},
                          protocol={"S": S, "trials_per_combo": n, "t_mid": t_mid, "split": split}, split=split)

    # ------------------------------------------------------------------
    # LEVEL 7 — GENERALIZAÇÃO (entradas e ambientes não vistos)
    # ------------------------------------------------------------------
    def probe_generalization(self, genome: Genome, seed: int, trials: Optional[int] = None, S: Optional[int] = None,
                             task: str = "channel", split: str = "validation") -> TestResult:
        n = trials or self.ps.trials
        S = S or self.cfg.world_size
        C = self.cfg.channels_max
        rng = np.random.default_rng(derive_seed(seed, "gen"))
        r = self.ps.stim_radius
        d, (ay, ax), (by, bx) = self._channel_geometry(S)
        train_envs, test_envs = self.envs_for("evolution"), self.envs_for(split)
        nsym = self.ps.channel_symbols if task == "channel" else 4
        pats = [self._pattern(rng, C, r) for _ in range(max(nsym, 2))]
        units, labels, is_test = [], [], []
        for i in range(nsym * n * 2):
            test = i >= nsym * n
            s = i % nsym
            amp = self.ps.stim_amp * (rng.choice([0.6, 1.4]) if test else 1.0)
            jy, jx = (int(rng.integers(-1, 2)), int(rng.integers(-1, 2))) if test else (0, 0)
            env = (test_envs if test else train_envs)[(i // nsym) % len(test_envs if test else train_envs)]
            st_ = []
            for k in range(self.ps.stim_pulses):
                if task == "channel":
                    st_.append((k + 1, pats[s], ay + jy, ax + jx, amp))
                else:
                    a1, a2 = s >> 1 & 1, s & 1
                    off = max(r + 1, d // 3)
                    st_.append((k + 1, pats[0] * (1 if a1 else -1), (ay - off + jy) % S, ax + jx, amp))
                    st_.append((k + 1, pats[1] * (1 if a2 else -1), (ay + off + jy) % S, ax + jx, amp))
            units.append({"env": env, "seed": derive_seed(seed, "gn", i, "test" if test else "train"), "group": i, "stims": st_})
            labels.append(s if task == "channel" else ((s >> 1 & 1) ^ (s & 1)))
            is_test.append(test)
        times = sorted(set(t + 1 for t in (self.ps.channel_times if task == "channel" else self.ps.transform_times) if t > 0))
        feats, info = self._run_protocol(genome, units, [{"name": "B", "kind": "region", "cy": by, "cx": bx,
                                                          "r": self.ps.obs_radius, "times": times}], max(times), seed, S=S)
        y = np.asarray(labels); te = np.asarray(is_test)
        ncls = nsym if task == "channel" else 2
        tr_idx, te_idx = np.where(~te)[0], np.where(te)[0]
        tr_ser = self._series(feats["B"], tr_idx, y[tr_idx], ncls, derive_seed(seed, "gtr"), times)
        best = max(tr_ser, key=lambda r_: r_["linear_acc"])     # t* escolhido apenas no treino (sem p-hacking)
        t_star = best["t"]
        X = feats["B"][t_star]
        yp = ridge_predict(X[tr_idx], y[tr_idx], X[te_idx], ncls)
        acc_te = balanced_accuracy(y[te_idx], yp, ncls)
        k = int((yp == y[te_idx]).sum())
        p_te = binom_p_greater(k, len(te_idx), 1.0 / ncls)
        tr_norm, te_norm = best["norm_acc"], norm_acc(acc_te, 1.0 / ncls)
        if any(info["bad"]):
            status, label = STATUS_ERROR, "ERROR"
        elif not best["sig"]:
            status, label = STATUS_FAIL, "NO_TRAIN_SIGNAL"
        elif p_te < self.th.alpha and te_norm >= self.th.min_norm_acc and (tr_norm - te_norm) <= self.th.generalization_drop:
            status, label = STATUS_PASS, "GENERALIZES"
        else:
            status, label = STATUS_FAIL, "OVERFIT"
        return TestResult(probe="generalization", candidate_id=genome.genome_id, status=status,
                          score=float(max(0.0, te_norm)) if status == STATUS_PASS else 0.0, p_value=float(p_te),
                          effect_size=float(te_norm - 0.0),
                          details={"label": label, "task": task, "t_star": t_star, "train_norm_acc": tr_norm,
                                   "test_norm_acc": te_norm, "test_acc": acc_te, "n_test": int(len(te_idx)),
                                   "unseen": "amplitudes {0.6,1.4}, jitter ±1, sementes novas, ambientes de " + split},
                          protocol={"S": S, "trials": n, "task": task, "split": split, "times": times}, split=split)

    # ------------------------------------------------------------------
    # AUTORREPARAÇÃO / RECUPERAÇÃO
    # ------------------------------------------------------------------
    def _largest_structure(self, U: torch.Tensor) -> Tuple[np.ndarray, bool]:
        mag = U.pow(2).sum(0).sqrt().cpu().numpy()
        thr = mag.mean() + mag.std()
        mask = (mag > thr) & (mag > 1e-3)
        if HAVE_SCIPY:
            lab, nlab = sndi.label(mask)
            if nlab == 0:
                return mask, False
            sizes = sndi.sum(mask, lab, index=np.arange(1, nlab + 1))
            big = int(np.argmax(sizes)) + 1
            comp = lab == big
        else:
            comp = mask
        return comp, bool(comp.sum() >= 4)

    def _similarity(self, Ud: torch.Tensor, Ur: torch.Tensor, region: torch.Tensor) -> Dict[str, float]:
        md = Ud.pow(2).sum(0).sqrt(); mr = Ur.pow(2).sum(0).sqrt()
        thr = mr.mean() + torch.clamp(mr.std(), min=0.05 * float(mr.mean()) + 1e-6)
        a, b = md > thr, mr > thr
        un = (a | b).float().sum().item()
        iou = 1.0 if un == 0 else (a & b).float().sum().item() / un
        reg = region.float()
        vd, vr = (Ud * reg).flatten(), (Ur * reg).flatten()
        cos = float(torch.dot(vd, vr) / (vd.norm() * vr.norm() + 1e-12)) if vr.norm() > 1e-9 else float(vd.norm() < 1e-9)
        traj = 1.0 - float((Ud - Ur).norm() / (Ud.norm() + Ur.norm() + 1e-12))
        return {"geometry": float(iou), "internal": max(0.0, cos), "trajectory": traj,
                "similarity": float((iou + max(0.0, cos) + traj) / 3)}

    def probe_recovery(self, genome: Genome, seed: int, split: str = "evolution", trials: Optional[int] = None,
                       S: Optional[int] = None, allow_fallback: bool = False, levels: Optional[List[float]] = None,
                       envs: Optional[List[Environment]] = None) -> TestResult:
        n = max(2, (trials or self.ps.trials) // 2)
        levels = levels or list(self.ps.damage_levels)
        gb, st, prot = self._setup(genome, n, split, seed, S=S, envs=envs)
        S = gb.S
        st = self.sim.run(gb, st, self.ps.recovery_warmup)
        regions, found = [], []
        for i in range(n):
            comp, ok = self._largest_structure(st.U[i])
            if not ok and allow_fallback:
                comp = np.zeros((S, S), bool); comp[S // 4:3 * S // 4, S // 4:3 * S // 4] = True
                ok = True
            regions.append(comp); found.append(ok)
        if not any(found):
            return TestResult(probe="recovery", candidate_id=genome.genome_id, status=STATUS_INCONCLUSIVE, score=0.0,
                              details={"note": "nenhuma estrutura persistente identificável"}, protocol=prot, split=split)
        L = len(levels)
        idx, groups = [], []
        for i in range(n):
            idx += [i] * (L + 1); groups += [i] * (L + 1)
        gb2 = gb.select(idx)
        st2 = st.select(idx, groups=groups, seed=derive_seed(seed, "branch"))
        rng = np.random.default_rng(derive_seed(seed, "dmg"))
        mask = np.zeros((len(idx), 1, S, S), bool)
        for i in range(n):
            if not found[i]:
                continue
            cells = np.argwhere(regions[i])
            for j, p in enumerate(levels):
                k = int(round(p * len(cells)))
                sel = cells[rng.choice(len(cells), size=k, replace=False)] if k > 0 else np.zeros((0, 2), int)
                mask[i * (L + 1) + j + 1, 0, sel[:, 0], sel[:, 1]] = True
        st2.damage_cells(torch.as_tensor(mask, device=self.sim.device))
        dil = []
        for i in range(n):
            reg = regions[i]
            if HAVE_SCIPY:
                reg = sndi.binary_dilation(reg, iterations=2)
            dil.append(torch.as_tensor(reg, device=self.sim.device).unsqueeze(0))

        def measure():
            out = np.zeros((n, L, 4))
            for i in range(n):
                ref = st2.U[i * (L + 1)]
                for j in range(L):
                    s_ = self._similarity(st2.U[i * (L + 1) + j + 1], ref, dil[i])
                    out[i, j] = [s_["geometry"], s_["internal"], s_["trajectory"], s_["similarity"]]
            return out
        sim0 = measure()
        st2 = self.sim.run(gb2, st2, self.ps.recovery_steps)
        sim1 = measure()
        rec = (sim1[..., 3] - sim0[..., 3]) / (1 - sim0[..., 3] + 1e-6)
        rec = np.clip(rec, -1, 1)
        ok_rows = np.asarray(found)
        curve = [{"damage": float(p), "sim_immediate": float(sim0[ok_rows, j, 3].mean()),
                  "sim_recovered": float(sim1[ok_rows, j, 3].mean()), "recovery": float(rec[ok_rows, j].mean()),
                  "geometry": float(sim1[ok_rows, j, 0].mean()), "internal": float(sim1[ok_rows, j, 1].mean()),
                  "trajectory": float(sim1[ok_rows, j, 2].mean())} for j, p in enumerate(levels)]
        low = [cv for cv in curve if cv["damage"] <= 0.5]
        mean_rec = float(np.mean([cv["recovery"] for cv in low]))
        mean_sim = float(np.mean([cv["sim_recovered"] for cv in low]))
        if bool(st2.bad.any()):
            status = STATUS_ERROR
        elif mean_rec >= self.th.recovery_gain and mean_sim >= self.th.recovery_sim:
            status = STATUS_PASS
        else:
            status = STATUS_FAIL
        lo, hi = bootstrap_ci(rec[ok_rows][:, [j for j, p in enumerate(levels) if p <= 0.5]].ravel())
        return TestResult(probe="recovery", candidate_id=genome.genome_id, status=status, score=max(0.0, mean_rec),
                          ci=(lo, hi), details={"curve": curve, "mean_recovery_le50": mean_rec,
                                                "mean_similarity_le50": mean_sim, "structures_found": int(sum(found)),
                                                "fallback_region": bool(allow_fallback)},
                          protocol={**prot, "levels": levels, "warmup": self.ps.recovery_warmup,
                                    "recovery_steps": self.ps.recovery_steps}, split=split)

    # ------------------------------------------------------------------
    # ROBUSTEZ
    # ------------------------------------------------------------------
    def _variant(self, genome: Genome, fn: Callable[[Dict[str, np.ndarray]], None], tag: str) -> Genome:
        p = {k: np.array(v, copy=True) for k, v in genome.params.items()}
        fn(p)
        return Genome(params=self.space.clamp(p), genome_id=f"{genome.genome_id}~{tag}", parent_ids=[genome.genome_id],
                      generation=genome.generation, origin="test-variant")

    def _desc_runs(self, genomes: List[Genome], envs: List[Environment], seed: int, S: Optional[int] = None,
                   hook: Optional[Callable] = None) -> Tuple[np.ndarray, List[str]]:
        rows = []
        for i, g in enumerate(genomes):
            env = envs[i % len(envs)]
            rows.append((g, env, derive_seed(seed, "rob", i)))
        gl = [r_[0] for r_ in rows]
        # reutiliza o pipeline de triagem com sementes fixas por réplica
        out = []
        per = max(1, self.cfg.batch_size // 2)
        for c0 in range(0, len(rows), per):
            chunk = rows[c0:c0 + per]
            jobs = [(j, e, s) for j, (_, e, s) in enumerate(chunk)]
            res = self.screener._screen_chunk([g for g, _, _ in chunk], jobs, self.ps.robust_steps,
                                              S or self.cfg.world_size, derive_seed(seed, "rob-run", c0), hook)
            out.extend(res)
        vecs = np.stack([descriptor_vector(r_) for r_ in out])
        via = [classify_viability(r_, self.th, r_["bad"]) for r_ in out]
        return vecs, via

    def probe_robustness(self, genome: Genome, seed: int, split: str = "evolution", trials: Optional[int] = None) -> TestResult:
        n = trials or self.ps.robust_trials
        envs = self.envs_for(split)
        base_v, base_via = self._desc_runs([genome] * n, envs, seed)
        v0 = base_v.mean(0)
        dim = math.sqrt(len(v0))

        grand = self.sim._grand

        def noise_hook(level):
            def h(st):
                st.U = st.U + level * grand(st, st.U.shape[1:])
            return h

        def sp_hook(frac):
            def h(st):
                if st.t % 8 == 0:
                    m = grand(st, (1,) + tuple(st.U.shape[2:]), "uniform") < frac
                    sgn = torch.sign(grand(st, st.U.shape[1:], "uniform") - 0.5)
                    st.U = torch.where(m, sgn * 0.9 * self.cfg.state_bound / 3, st.U)
            return h

        def del_hook(frac):
            def h(st):
                if st.t % 8 == 0:
                    st.damage_cells(grand(st, (1,) + tuple(st.U.shape[2:]), "uniform") < frac)
            return h

        drift_rng = np.random.default_rng(derive_seed(seed, "drift"))

        def drift(p):
            for nme, s in self.space.specs.items():
                if s.kind in ("float", "logfloat") and s.group != "mutation_parameters":
                    p[nme] = np.asarray(p[nme]) * (1 + drift_rng.normal(0, 0.05, size=np.shape(p[nme])))

        conds = []
        for lvl in [l for l in self.ps.noise_levels if l > 0]:
            conds.append(("gaussian_noise", lvl, [genome] * n, envs, None, noise_hook(lvl)))
        conds.append(("salt_pepper", 0.01, [genome] * n, envs, None, sp_hook(0.01)))
        conds.append(("cell_deletion", 0.02, [genome] * n, envs, None, del_hook(0.02)))
        conds.append(("parameter_drift", 0.05, [self._variant(genome, drift, "drift")] * n, envs, None, None))
        for f_ in [0.8, 1.25]:
            conds.append(("time_step_variation", f_, [self._variant(genome, lambda p, f_=f_: p.__setitem__("time_scale", np.float32(float(p["time_scale"]) * f_)), f"dt{f_}")] * n, envs, None, None))
        for b in range(len(BOUNDARIES)):
            if b != int(genome.params["boundary"]):
                conds.append(("boundary_variation", BOUNDARIES[b], [self._variant(genome, lambda p, b=b: p.__setitem__("boundary", np.int64(b)), f"b{b}")] * n, envs, None, None))
        other = self.envs_for("validation") if split == "evolution" else self.envs_for("evolution")
        conds.append(("initial_state_variation", other[0].env_id, [genome] * n, other, None, None))
        S2 = int(round(self.cfg.world_size * 1.5 / 2) * 2)
        conds.append(("world_size_variation", S2, [genome] * n, envs, S2, None))
        rows = []
        for name, level, gl, ev, S_, hook in conds:
            v, via = self._desc_runs(gl, ev, seed, S=S_, hook=hook)
            sim_ = float(1 - np.linalg.norm(v.mean(0) - v0) / dim)
            viable = float(np.mean([x == "VIABLE" for x in via]))
            rows.append({"condition": name, "level": level, "similarity": sim_, "viable_fraction": viable,
                         "ok": bool(viable >= 0.5 and sim_ >= self.th.robust_sim)})
        score = float(np.mean([r_["ok"] for r_ in rows]))
        status = STATUS_PASS if score >= self.th.robustness else STATUS_FAIL
        curve = [{"noise": 0.0, "similarity": 1.0}] + [{"noise": r_["level"], "similarity": r_["similarity"]}
                                                       for r_ in rows if r_["condition"] == "gaussian_noise"]
        return TestResult(probe="robustness", candidate_id=genome.genome_id, status=status, score=score,
                          details={"conditions": rows, "noise_curve": curve,
                                   "baseline_viable": float(np.mean([x == "VIABLE" for x in base_via]))},
                          protocol={"trials": n, "split": split, "steps": self.ps.robust_steps}, split=split)

    # ------------------------------------------------------------------
    # LEVEL 8 — ADAPTAÇÃO
    # ------------------------------------------------------------------
    def probe_adaptation(self, genome: Genome, seed: int, split: str = "validation", trials: Optional[int] = None) -> TestResult:
        n = trials or self.ps.trials
        gb, st, prot = self._setup(genome, n, split, seed, repeat=2)
        S = gb.S
        m = min(self.cfg.metric_res, S)
        W = max(16, self.ps.adaptation_steps // 2)
        st = self.sim.run(gb, st, self.ps.warmup)

        def window(st):
            frames = []
            for t in range(W):
                st = self.sim.step(gb, st)
                if t % 2 == 1:
                    frames.append(F.adaptive_avg_pool2d(st.U, m))
            Fr = torch.stack(frames)
            d = compute_descriptors(Fr, None, None, gb.chm.view(gb.B, -1), gb.sat.view(gb.B, -1), S, 2, W)
            vec = np.stack([descriptor_vector({k: float(v[i]) for k, v in d.items()} | {"lyapunov": 0.0, "causal_spread": 0.0})
                            for i in range(gb.B)])
            return st, vec
        st, pre = window(st)
        units = [{"changed": i >= n} for i in range(2 * n)]
        self._apply_env_change(st, {"resource_mult": 0.5, "noise_add": 0.01}, units)
        st = self.sim.run(gb, st, self.ps.adaptation_steps)
        st, post = window(st)
        change = np.linalg.norm(post - pre, axis=1)
        p_change = mann_whitney_greater(change[n:], change[:n])
        func = self.probe_channel(genome, derive_seed(seed, "adapt-func"), split=split, trials=trials,
                                  env_change={"t": -1, "resource_mult": 0.5, "noise_add": 0.01})
        behav_ok = p_change < self.th.adaptation_p
        func_ok = func.status == STATUS_PASS
        status = STATUS_PASS if (behav_ok and func_ok) else STATUS_FAIL
        return TestResult(probe="adaptation", candidate_id=genome.genome_id, status=status,
                          score=float(func.score if status == STATUS_PASS else 0.0), p_value=float(p_change),
                          effect_size=cohens_d(change[n:], change[:n]),
                          details={"behavior_change_changed": float(change[n:].mean()),
                                   "behavior_change_control": float(change[:n].mean()), "behavior_p": p_change,
                                   "function_status": func.status, "function_score": func.score,
                                   "interpretation": "mudança de comportamento acima da deriva natural mantendo canal funcional; "
                                                     "não implica aprendizado, vida ou agência"},
                          protocol={**prot, "window": W, "change": {"resource_mult": 0.5, "noise_add": 0.01}}, split=split)

    # ------------------------------------------------------------------
    # REPLICAÇÃO / ABLAÇÃO / TRIAGEM LONGA
    # ------------------------------------------------------------------
    def probe_replication(self, genome: Genome, seed: int, target: str = "memory", n_seeds: Optional[int] = None,
                          split: str = "validation") -> TestResult:
        n_seeds = n_seeds or self.ps.replication_seeds
        runs = [self.run_probe(target, genome, derive_seed(seed, "rep", i), split=split) for i in range(n_seeds)]
        scores = np.asarray([r_.score if np.isfinite(r_.score) else 0.0 for r_ in runs])
        passes = np.asarray([r_.status == STATUS_PASS for r_ in runs])
        rate = float(passes.mean())
        lo, hi = bootstrap_ci(scores)
        status = STATUS_PASS if rate >= self.th.replication_pass_rate else (STATUS_INCONCLUSIVE if rate >= 0.5 else STATUS_FAIL)
        return TestResult(probe="replication", candidate_id=genome.genome_id, status=status, score=rate,
                          p_value=binom_p_greater(int(passes.sum()), n_seeds, self.th.alpha),
                          effect_size=cohens_d(scores, np.zeros_like(scores) + 1e-9), ci=(lo, hi),
                          details={"target": target, "pass_rate": rate, "scores": scores.tolist(),
                                   "statuses": [r_.status for r_ in runs], "stats": summary_stats(scores)},
                          protocol={"target": target, "n_seeds": n_seeds, "split": split}, split=split)

    ABLATIONS = {
        "remove_diffusion": lambda p: p.__setitem__("diff", np.zeros_like(p["diff"])),
        "remove_long_range": lambda p: p.__setitem__("link_active", np.zeros_like(p["link_active"])),
        "remove_conservation": lambda p: p.__setitem__("conserve", np.zeros_like(p["conserve"])),
        "remove_reaction": lambda p: p.__setitem__("Q", np.zeros_like(p["Q"])),
        "remove_inertia": lambda p: p.__setitem__("inertia", np.zeros_like(p["inertia"])),
        "remove_delay": lambda p: p.__setitem__("delay_w", np.zeros_like(p["delay_w"])),
        "remove_hysteresis": lambda p: p.__setitem__("hyst_w", np.zeros_like(p["hyst_w"])),
        "remove_resource_limit": lambda p: p.__setitem__("res_enabled", np.zeros_like(p["res_enabled"])),
        "remove_plasticity": lambda p: p.__setitem__("link_eta", np.zeros_like(p["link_eta"])),
        "remove_noise": lambda p: p.__setitem__("noise", np.zeros_like(p["noise"])),
        "remove_advection": lambda p: p.__setitem__("vel", np.zeros_like(p["vel"])),
        "remove_nonlinear_growth": lambda p: p.__setitem__("g_gain", np.zeros_like(p["g_gain"])),
    }

    def probe_ablation(self, genome: Genome, seed: int, target: str = "memory", split: str = "validation") -> TestResult:
        base = self.run_probe(target, genome, seed, split=split)
        variants = []
        for name, fn in self.ABLATIONS.items():
            v = self._variant(genome, fn, name)
            if v.hash() != self._variant(genome, lambda p: None, "id").hash():
                variants.append((name, v))
        act = np.where(genome.params["ch_active"])[0]
        if len(act) > 1:
            for c in act:
                variants.append((f"remove_channel_{c}", self._variant(genome, lambda p, c=c: p["ch_active"].__setitem__(c, False), f"rc{c}")))
        kact = np.where(genome.params["k_active"])[0]
        if len(kact) > 1:
            for k in kact:
                variants.append((f"remove_kernel_{k}", self._variant(genome, lambda p, k=k: p["k_active"].__setitem__(k, False), f"rk{k}")))
        rows = []
        bscore = base.score if np.isfinite(base.score) else 0.0
        for name, v in variants:
            r_ = self.run_probe(target, v, seed, split=split)
            sc = r_.score if np.isfinite(r_.score) else 0.0
            rows.append({"ablation": name, "status": r_.status, "score": sc, "importance": float(bscore - sc)})
        rows.sort(key=lambda x: -x["importance"])
        return TestResult(probe="ablation", candidate_id=genome.genome_id, status=STATUS_PASS if rows else STATUS_INCONCLUSIVE,
                          score=float(max([r_["importance"] for r_ in rows], default=0.0)),
                          details={"target": target, "base_status": base.status, "base_score": bscore, "ablations": rows,
                                   "essential": [r_["ablation"] for r_ in rows if r_["importance"] > 0.5 * max(bscore, 1e-9) and bscore > 0]},
                          protocol={"target": target, "split": split}, split=split)

    def probe_long_screen(self, genome: Genome, seed: int, split: str = "evolution") -> TestResult:
        """Avaliação longa para sobreviventes (multi-fidelidade): mais passos, mundo maior."""
        S2 = int(round(self.cfg.world_size * 1.5 / 2) * 2)
        d = self.screener.screen([genome], self.envs_for(split), seed, steps=self.cfg.screen_steps * 2, S=S2,
                                 n_envs=len(self.envs_for(split)))[0]
        ok = d["viability"] == "VIABLE"
        return TestResult(probe="long_screen", candidate_id=genome.genome_id, status=STATUS_PASS if ok else STATUS_FAIL,
                          score=float(d["persistence"]), details={k: v for k, v in d.items() if not isinstance(v, list)},
                          protocol={"S": S2, "steps": self.cfg.screen_steps * 2, "split": split}, split=split)

### 6b. Análise avançada (somente candidatos fortes / ChampionLab)
Instrumentos externos de caracterização: transfer entropy aproximada e teste tipo Granger entre regiões
(correlação preditiva **não** é causalidade física — a intervenção tem prioridade), lei efetiva macroscópica por
coarse-graining (AR linear em blocos), variáveis emergentes (PCA de canais), recorrência (atratores / ciclos-limite),
rastreamento de objetos (tempo de vida, trajetórias), hierarquia multiescala, varredura de fase (transições abruptas),
histerese ambiental, localização da memória e mapa de importância causal por dano em blocos.

In [ ]:
# @title 6b. Análise avançada (somente candidatos fortes / ChampionLab) { display-mode: "form" }
def _transfer_entropy(x: np.ndarray, y: np.ndarray, bins: int = 3) -> float:
    """TE_{x->y} com histórico 1, discretização por quantis (estimador plug-in; viesado para séries curtas)."""
    if len(x) < 10 or np.std(x) < 1e-9 or np.std(y) < 1e-9:
        return 0.0
    q = lambda s: np.clip(np.searchsorted(np.quantile(s, np.linspace(0, 1, bins + 1)[1:-1]), s), 0, bins - 1)
    xd, yd = q(x), q(y)
    y1, y0, x0 = yd[1:], yd[:-1], xd[:-1]
    joint = np.zeros((bins, bins, bins))
    for a, b, c in zip(y1, y0, x0):
        joint[a, b, c] += 1
    joint /= joint.sum()
    p_y0x0 = joint.sum(0); p_y1y0 = joint.sum(2); p_y0 = joint.sum((0, 2))
    te = 0.0
    for a in range(bins):
        for b in range(bins):
            for c in range(bins):
                p = joint[a, b, c]
                if p > 0 and p_y0x0[b, c] > 0 and p_y1y0[a, b] > 0 and p_y0[b] > 0:
                    te += p * math.log2(p * p_y0[b] / (p_y0x0[b, c] * p_y1y0[a, b]))
    return float(max(0.0, te))


def _granger_gain(x: np.ndarray, y: np.ndarray) -> float:
    if len(x) < 10:
        return 0.0
    Y = y[1:]
    A1 = np.stack([y[:-1], np.ones(len(Y))], 1)
    A2 = np.stack([y[:-1], x[:-1], np.ones(len(Y))], 1)
    r1 = Y - A1 @ np.linalg.lstsq(A1, Y, rcond=None)[0]
    r2 = Y - A2 @ np.linalg.lstsq(A2, Y, rcond=None)[0]
    return float(max(0.0, 1 - (r2 @ r2) / (r1 @ r1 + 1e-12)))


def probe_advanced(self: ValidationEngine, genome: Genome, seed: int, split: str = "holdout", steps: int = 256) -> TestResult:
    out: Dict[str, Any] = {}
    n = 4
    gb, st, prot = self._setup(genome, n, split, seed)
    S = gb.S
    m = min(self.cfg.metric_res, S)
    d, (ay, ax), (by, bx) = self._channel_geometry(S)
    r = self.ps.obs_radius
    st = self.sim.run(gb, st, self.ps.warmup)
    frames, sa, sb = [], [], []
    for t in range(steps):
        st = self.sim.step(gb, st)
        U = st.U
        sa.append(U[:, :, ay - r:ay + r + 1, ax - r:ax + r + 1].sum(1).mean((1, 2)).cpu().numpy())
        sb.append(U[:, :, by - r:by + r + 1, bx - r:bx + r + 1].sum(1).mean((1, 2)).cpu().numpy())
        if t % 2 == 1:
            frames.append(F.adaptive_avg_pool2d(U, m).cpu().numpy())
    sa, sb = np.asarray(sa), np.asarray(sb)
    Fr = np.asarray(frames)                       # [T, n, C, m, m]
    out["transfer_entropy_A_to_B"] = float(np.mean([_transfer_entropy(sa[:, i], sb[:, i]) for i in range(n)]))
    out["transfer_entropy_B_to_A"] = float(np.mean([_transfer_entropy(sb[:, i], sa[:, i]) for i in range(n)]))
    out["granger_gain_A_to_B"] = float(np.mean([_granger_gain(sa[:, i], sb[:, i]) for i in range(n)]))
    out["granger_gain_B_to_A"] = float(np.mean([_granger_gain(sb[:, i], sa[:, i]) for i in range(n)]))
    # lei efetiva por coarse-graining (blocos 4x4) — AR(1) linear com ridge, R² fora da amostra vs persistência
    T = Fr.shape[0]
    blocks = max(2, m // 8)
    Xc = np.stack([F.adaptive_avg_pool2d(torch.as_tensor(Fr[:, i]), blocks).flatten(1).numpy() for i in range(n)])
    tr = int(T * 0.7)
    r2s, r2p = [], []
    for i in range(n):
        X0, X1 = Xc[i, :-1], Xc[i, 1:]
        A = np.hstack([X0[:tr], np.ones((tr, 1))])
        Wt = np.linalg.solve(A.T @ A + 1e-2 * np.eye(A.shape[1]), A.T @ X1[:tr])
        pred = np.hstack([X0[tr:], np.ones((len(X0) - tr, 1))]) @ Wt
        ss = ((X1[tr:] - X1[tr:].mean(0)) ** 2).sum() + 1e-12
        r2s.append(1 - ((X1[tr:] - pred) ** 2).sum() / ss)
        r2p.append(1 - ((X1[tr:] - X0[tr:]) ** 2).sum() / ss)
    out["effective_law_R2"] = float(np.mean(r2s)); out["persistence_baseline_R2"] = float(np.mean(r2p))
    out["macroscopic_law_detected"] = bool(np.mean(r2s) > max(0.5, np.mean(r2p) + 0.05))
    # variáveis emergentes (PCA de canais)
    Xch = Fr.transpose(0, 1, 3, 4, 2).reshape(-1, Fr.shape[2])
    Xch = Xch - Xch.mean(0)
    try:
        sv = np.linalg.svd(Xch[:: max(1, len(Xch) // 20000)], compute_uv=False)
        ev = (sv ** 2) / ((sv ** 2).sum() + 1e-12)
    except Exception:
        ev = np.zeros(Fr.shape[2])
    out["channel_pca_explained"] = ev[:4].round(4).tolist()
    # recorrência: distância mínima a estados anteriores (atrator fixo / ciclo-limite / caos)
    Z = Fr[:, 0].reshape(T, -1)
    Zn = Z / (np.linalg.norm(Z, axis=1, keepdims=True) + 1e-9)
    late = Zn[T // 2:]
    D = 1 - late @ Zn[: T // 2].T
    rec = float(D.min(1).mean())
    lag_best = int(np.argmax([np.mean(np.sum(Zn[k:] * Zn[:-k], 1)) for k in range(2, max(3, T // 3))]) + 2) if T > 6 else 0
    out["recurrence_distance"] = rec
    out["attractor_type"] = ("fixed_point" if np.mean(np.abs(np.diff(Z[-10:], axis=0))) < 1e-5 else
                             ("limit_cycle_candidate" if rec < 0.01 else "aperiodic"))
    out["dominant_period_frames"] = lag_best
    # rastreamento de objetos
    lifetimes, speeds = [], []
    if HAVE_SCIPY:
        prev = None
        alive: Dict[int, Dict[str, Any]] = {}
        nid = 0
        for t in range(T):
            mag = np.sqrt((Fr[t, 0] ** 2).sum(0))
            mask = mag > mag.mean() + mag.std()
            lab, nl = sndi.label(mask)
            cents = sndi.center_of_mass(mask, lab, range(1, nl + 1)) if nl else []
            new_alive = {}
            for c in cents:
                best, bd = None, 3.0
                for k_, o in alive.items():
                    dd = math.hypot(c[0] - o["pos"][0], c[1] - o["pos"][1])
                    if dd < bd and k_ not in new_alive:
                        best, bd = k_, dd
                if best is None:
                    new_alive[nid] = {"pos": c, "born": t, "path": [c]}; nid += 1
                else:
                    o = alive[best]; o["path"].append(c); o["pos"] = c; new_alive[best] = o
            for k_, o in alive.items():
                if k_ not in new_alive:
                    lifetimes.append(t - o["born"])
                    if len(o["path"]) > 2:
                        p = np.asarray(o["path"]); speeds.append(float(np.linalg.norm(p[-1] - p[0]) / len(p)))
            alive = new_alive
        for o in alive.values():
            lifetimes.append(T - o["born"])
    out["object_lifetimes"] = summary_stats(lifetimes) if lifetimes else {"n": 0}
    out["object_speed_cells_per_frame"] = summary_stats(speeds) if speeds else {"n": 0}
    # hierarquia multiescala: estruturas em resoluções m, m/2, m/4
    hier = {}
    last = torch.as_tensor(Fr[-1])
    for res_ in [m, max(4, m // 2), max(2, m // 4)]:
        mg = F.adaptive_avg_pool2d(last, res_).pow(2).sum(1).sqrt()
        mk = mg > mg.mean((1, 2), keepdim=True) + mg.std((1, 2), keepdim=True)
        hier[str(res_)] = float(count_components(mk, 2 * res_).mean().item())
    out["structures_by_scale"] = hier
    # varredura de fase: ganho de interação × difusão
    gains, diffs = [0.5, 1.0, 1.5], [0.0, 1.0, 2.0]
    variants = []
    for gg in gains:
        for dd in diffs:
            variants.append(self._variant(genome, lambda p, gg=gg, dd=dd: (p.__setitem__("W", p["W"] * gg),
                                                                          p.__setitem__("diff", p["diff"] * dd)), f"ph{gg}_{dd}"))
    vecs, via = self._desc_runs(variants, self.envs_for(split), derive_seed(seed, "phase"))
    grid = vecs[:, DESCRIPTOR_NAMES.index("persistence")].reshape(len(gains), len(diffs))
    act = vecs[:, DESCRIPTOR_NAMES.index("rel_activity")].reshape(len(gains), len(diffs))
    jumps = max(float(np.abs(np.diff(grid, axis=0)).max(initial=0)), float(np.abs(np.diff(grid, axis=1)).max(initial=0)))
    out["phase_diagram"] = {"gains": gains, "diffusion_scales": diffs, "persistence": grid.round(3).tolist(),
                            "activity": act.round(3).tolist(), "viability": via, "max_jump": jumps,
                            "abrupt_transition_suspected": bool(jumps > 0.4)}
    # histerese ambiental (recurso sobe e desce)
    gb1, st1, _ = self._setup(genome, 1, split, derive_seed(seed, "hyst"))
    st1 = self.sim.run(gb1, st1, self.ps.warmup)
    levels = [0.4, 0.7, 1.0, 1.3, 1.6]
    up, down = [], []
    for seq, store in [(levels, up), (levels[::-1], down)]:
        for lv in seq:
            st1.R0 = torch.full_like(st1.R0, lv)
            U0 = st1.U.clone()
            st1 = self.sim.run(gb1, st1, 32)
            store.append(float((st1.U - U0).abs().mean().item()))
    out["hysteresis"] = {"levels": levels, "activity_up": up, "activity_down": down[::-1],
                         "area": float(np.abs(np.asarray(up) - np.asarray(down[::-1])).sum())}
    # mapa de importância causal: dano em blocos 4x4 e divergência resultante (pares com ruído idêntico)
    nb = 4
    bs = S // nb
    gbm, stm, _ = self._setup(genome, 1, split, derive_seed(seed, "imp"), repeat=nb * nb + 1)
    stm = self.sim.run(gbm, stm, self.ps.warmup)
    mask = torch.zeros(nb * nb + 1, 1, S, S, dtype=torch.bool, device=self.sim.device)
    for k in range(nb * nb):
        i, j = divmod(k, nb)
        mask[k + 1, 0, i * bs:(i + 1) * bs, j * bs:(j + 1) * bs] = True
    stm.damage_cells(mask)
    stm = self.sim.run(gbm, stm, 64)
    ref = stm.U[0]
    imp = ((stm.U[1:] - ref).abs().mean((1, 2, 3)) / (ref.abs().mean() + 1e-9)).cpu().numpy().reshape(nb, nb)
    out["causal_importance_map"] = imp.round(4).tolist()
    out["importance_concentration"] = float(imp.max() / (imp.mean() + 1e-9))
    # localização da memória: decodificação local vs distante vs global
    pats = [self._pattern(np.random.default_rng(seed), self.cfg.channels_max, self.ps.stim_radius + 1) for _ in range(2)]
    c = S // 2
    pool = self.envs_for(split)
    units = []
    k_ = 2 * self.ps.trials
    for i in range(k_):
        units.append({"env": pool[(i // 2) % len(pool)], "seed": derive_seed(seed, "memloc", i), "group": i,
                      "stims": [(h, pats[i % 2], c, c, self.ps.stim_amp * 0.5) for h in range(self.ps.memory_history_steps)]})
    tmem = self.ps.memory_history_steps + self.th.memory_min_delay
    far = (c + S // 4) % S
    feats, _ = self._run_protocol(genome, units, [
        {"name": "local", "kind": "region", "cy": c, "cx": c, "r": self.ps.stim_radius + 2, "times": [tmem]},
        {"name": "far", "kind": "region", "cy": far, "cx": far, "r": self.ps.stim_radius + 2, "times": [tmem]},
        {"name": "global", "kind": "global", "cy": 0, "cx": 0, "r": 0, "times": [tmem]}], tmem, derive_seed(seed, "ml"))
    y = np.arange(k_) % 2
    loc = {nm: decode_report(feats[nm][tmem], y, 2, self.ps.permutations, self.ps.folds, seed)["linear_acc"] for nm in feats}
    out["memory_location_acc"] = loc
    out["memory_distribution"] = ("localized" if loc["local"] - loc["far"] > 0.2 else
                                  ("distributed" if loc["far"] > 0.7 else "weak/none"))
    return TestResult(probe="advanced", candidate_id=genome.genome_id, status=STATUS_PASS, score=float("nan"),
                      details=out, protocol={**prot, "steps": steps}, split=split)


ValidationEngine.probe_advanced = probe_advanced

## 7. Quality-Diversity e arquivos permanentes
* **MAP-Elites** sobre eixos comportamentais (padrão: persistência × propagação causal × complexidade temporal).
  Elites substituídas nunca são apagadas (histórico de elites).
* **Novelty archive** (kNN no espaço de descritores) — um universo pode ser "ruim" e muito diferente: é preservado.
* **Genesis Zoo** (categorias descritivas externas, múltiplas por universo; `UNKNOWN` e `ANOMALOUS` são cidadãos de primeira classe).
* **Champion / Anomaly / Failure archives**, **linhagem** completa, **especiação** por genótipo + comportamento,
  **modelo de surpresa** (z-score frente ao histórico), **bandit UCB** para operadores de mutação.

In [ ]:
# @title 7. Quality-Diversity e arquivos permanentes { display-mode: "form" }
ZOO_CATEGORIES = ["STATIC", "OSCILLATORY", "PROPAGATING", "PERSISTENT", "MOBILE", "RECOVERING", "MEMORY_CANDIDATE",
                  "CHANNEL_CANDIDATE", "TRANSFORMER_CANDIDATE", "UNKNOWN", "ANOMALOUS"]


def pack_params(space: ParamSpace, params: Dict[str, np.ndarray]) -> np.ndarray:
    return np.concatenate([np.asarray(params[n], np.float32).ravel() for n in space.specs]).astype(np.float32)


def unpack_params(space: ParamSpace, vec: np.ndarray) -> Dict[str, np.ndarray]:
    out, i = {}, 0
    for n, s in space.specs.items():
        k = int(np.prod(s.shape)) if s.shape else 1
        v = np.asarray(vec[i:i + k], np.float64).reshape(s.shape)
        i += k
        if s.kind == "bool":
            v = v > 0.5
        elif s.kind in ("int", "cat"):
            v = np.rint(v).astype(np.int64)
        else:
            v = v.astype(np.float32)
        out[n] = v
    return space.clamp(out)


def genome_to_record(space: ParamSpace, g: Genome) -> dict:
    d = g.to_dict(); d["packed"] = pack_params(space, g.params); del d["params"]
    return d


def genome_from_record(space: ParamSpace, d: dict) -> Genome:
    d = dict(d); d["params"] = unpack_params(space, d.pop("packed"))
    return Genome.from_dict(d)


class MapElites:
    def __init__(self, axes: List[str], bins: List[int]):
        self.axes, self.bins = list(axes), list(bins)
        self.cells: Dict[Tuple[int, ...], Dict[str, Any]] = {}
        self.history: List[Dict[str, Any]] = []
        self.insertions = 0

    def index(self, d: Dict[str, float]) -> Tuple[int, ...]:
        return tuple(int(min(b - 1, max(0, math.floor(float(np.clip(d.get(a, 0.0), 0, 1)) * b))))
                     for a, b in zip(self.axes, self.bins))

    def try_insert(self, gid: str, d: Dict[str, float], quality: float, gen: int) -> Tuple[bool, bool]:
        key = self.index(d)
        cur = self.cells.get(key)
        if cur is None or quality > cur["quality"]:
            if cur is not None:
                self.history.append({**cur, "cell": list(key), "replaced_at": gen})
            self.cells[key] = {"genome_id": gid, "quality": float(quality), "generation": gen}
            self.insertions += 1
            return True, cur is None
        return False, False

    def coverage(self) -> float:
        return len(self.cells) / float(np.prod(self.bins))

    def heat(self) -> np.ndarray:
        H = np.full(self.bins[:2], np.nan)
        for k, v in self.cells.items():
            i, j = k[0], k[1]
            H[i, j] = v["quality"] if np.isnan(H[i, j]) else max(H[i, j], v["quality"])
        return H

    def ids(self) -> List[str]:
        return [v["genome_id"] for v in self.cells.values()]

    def sample(self, rng: np.random.Generator, k: int) -> List[str]:
        ids = self.ids()
        if not ids:
            return []
        return [ids[i] for i in rng.integers(0, len(ids), size=k)]

    def state_dict(self):
        return {"axes": self.axes, "bins": self.bins, "cells": {",".join(map(str, k)): v for k, v in self.cells.items()},
                "history": self.history, "insertions": self.insertions}

    def load_state_dict(self, s):
        self.axes, self.bins = s["axes"], s["bins"]
        self.cells = {tuple(int(x) for x in k.split(",")): v for k, v in s["cells"].items()}
        self.history, self.insertions = s["history"], s["insertions"]


class NoveltyArchive:
    def __init__(self, k: int, max_size: int, target_add_rate: float = 0.05):
        self.k, self.max_size = k, max_size
        self.vecs = np.zeros((0, len(DESCRIPTOR_NAMES)), np.float32)
        self.ids: List[str] = []
        self.threshold = 0.15
        self.target = target_add_rate
        self.recent_adds: List[int] = []

    def novelty(self, V: np.ndarray, others: Optional[np.ndarray] = None) -> np.ndarray:
        ref = self.vecs if others is None else (np.vstack([self.vecs, others]) if len(self.vecs) else others)
        if len(ref) == 0:
            return np.ones(len(V), np.float32)
        Vt, Rt = torch.as_tensor(np.asarray(V, np.float32)), torch.as_tensor(np.asarray(ref, np.float32))
        d = torch.cdist(Vt, Rt)
        d = torch.where(d < 1e-9, torch.full_like(d, 1e6), d)     # exclui a auto-correspondência
        k = min(self.k, d.shape[1])
        nn = torch.topk(d, k, dim=1, largest=False).values
        valid = (nn < 1e6).float()
        out = (nn * valid).sum(1) / valid.sum(1).clamp(min=1)
        out = torch.where(valid.sum(1) > 0, out, torch.ones_like(out))
        return out.numpy().astype(np.float32)

    def maybe_add(self, ids: List[str], V: np.ndarray, nov: np.ndarray, rng: np.random.Generator) -> List[str]:
        added = []
        for gid, v, n in zip(ids, V, nov):
            if n > self.threshold or rng.random() < 0.01:
                self.vecs = np.vstack([self.vecs, v[None]]); self.ids.append(gid); added.append(gid)
        self.recent_adds.append(len(added))
        self.recent_adds = self.recent_adds[-10:]
        rate = sum(self.recent_adds) / max(1, len(ids) * len(self.recent_adds))
        self.threshold *= 1.05 if rate > self.target else 0.95
        self.threshold = float(np.clip(self.threshold, 0.01, 2.0))
        if len(self.ids) > self.max_size:   # mantém os mais isolados (diversidade), remove os mais redundantes
            nv = self.novelty(self.vecs, None)
            keep = np.argsort(-nv)[: self.max_size]
            keep.sort()
            self.vecs = self.vecs[keep]; self.ids = [self.ids[i] for i in keep]
        return added

    def state_dict(self):
        return {"vecs": self.vecs, "ids": self.ids, "threshold": self.threshold, "recent": self.recent_adds}

    def load_state_dict(self, s):
        self.vecs, self.ids, self.threshold, self.recent_adds = np.asarray(s["vecs"], np.float32), list(s["ids"]), s["threshold"], s["recent"]


class Zoo:
    def __init__(self, cap: int = 300):
        self.cap = cap
        self.cats: Dict[str, List[Dict[str, Any]]] = {c: [] for c in ZOO_CATEGORIES}

    @staticmethod
    def classify(d: Dict[str, float], statuses: Dict[str, str], th: Thresholds, anomalous: bool) -> List[str]:
        if d.get("viability") not in ("VIABLE", "FROZEN"):
            return []
        cats = []
        frozen = d["rel_activity"] < th.frozen_rel_activity * 10
        if frozen and d["persistence"] > 0.5 and d["structures"] >= 1:
            cats.append("STATIC")
        if d["periodicity"] > 0.5 and not frozen:
            cats.append("OSCILLATORY")
        if d["causal_spread"] > 0.3 and d["lyapunov"] < th.chaos_lyap:
            cats.append("PROPAGATING")
        if d["persistence"] > th.persistence and not frozen:
            cats.append("PERSISTENT")
        if d["mobility"] > 0.2:
            cats.append("MOBILE")
        if statuses.get("recovery") == STATUS_PASS:
            cats.append("RECOVERING")
        if statuses.get("memory") == STATUS_PASS:
            cats.append("MEMORY_CANDIDATE")
        if statuses.get("channel") == STATUS_PASS:
            cats.append("CHANNEL_CANDIDATE")
        if statuses.get("transformation") == STATUS_PASS:
            cats.append("TRANSFORMER_CANDIDATE")
        if anomalous:
            cats.append("ANOMALOUS")
        if not cats and d.get("viability") == "VIABLE":
            cats.append("UNKNOWN")
        return cats

    def add(self, gid: str, cats: List[str], novelty: float, gen: int):
        for c in cats:
            lst = self.cats[c]
            if any(e["genome_id"] == gid for e in lst):
                continue
            lst.append({"genome_id": gid, "novelty": float(novelty), "generation": gen})
            if len(lst) > self.cap:
                lst.sort(key=lambda e: -e["novelty"])
                del lst[self.cap:]

    def ids(self) -> List[str]:
        return sorted(set(e["genome_id"] for l in self.cats.values() for e in l))

    def counts(self) -> Dict[str, int]:
        return {c: len(v) for c, v in self.cats.items()}

    def state_dict(self):
        return {"cap": self.cap, "cats": self.cats}

    def load_state_dict(self, s):
        self.cap = s["cap"]; self.cats = {c: list(s["cats"].get(c, [])) for c in ZOO_CATEGORIES}


class SimpleArchive:
    def __init__(self, name: str, cap: int = 2000):
        self.name, self.cap = name, cap
        self.entries: List[Dict[str, Any]] = []

    def add(self, gid: str, reason: str, gen: int, **extra) -> bool:
        if any(e["genome_id"] == gid and e["reason"] == reason for e in self.entries):
            return False
        self.entries.append({"genome_id": gid, "reason": reason, "generation": gen, "ts": iso_now(), **to_jsonable(extra)})
        if len(self.entries) > self.cap:
            self.entries = self.entries[-self.cap:]
        return True

    def ids(self) -> List[str]:
        return sorted(set(e["genome_id"] for e in self.entries))

    def state_dict(self):
        return {"name": self.name, "cap": self.cap, "entries": self.entries}

    def load_state_dict(self, s):
        self.name, self.cap, self.entries = s["name"], s["cap"], list(s["entries"])


class Lineage:
    def __init__(self):
        self.records: Dict[str, Dict[str, Any]] = {}
        self.children: Dict[str, List[str]] = {}

    def add(self, g: Genome):
        self.records[g.genome_id] = {"parent_ids": list(g.parent_ids), "generation": g.generation,
                                     "mutation_record": list(g.mutation_record), "birth_timestamp": g.birth_timestamp,
                                     "operator": g.operator, "origin": g.origin, "species_id": g.species_id,
                                     "quality": None, "level": 0}
        for p in g.parent_ids:
            self.children.setdefault(p, []).append(g.genome_id)

    def update(self, gid: str, **kw):
        if gid in self.records:
            self.records[gid].update(to_jsonable(kw))

    def ancestors(self, gid: str, max_depth: int = 200) -> List[str]:
        out, cur, seen = [], gid, set()
        for _ in range(max_depth):
            r = self.records.get(cur)
            if r is None or not r["parent_ids"]:
                break
            cur = r["parent_ids"][0]
            if cur in seen:
                break
            seen.add(cur); out.append(cur)
        return out

    def tree(self, gid: str, depth: int = 3, max_children: int = 6) -> str:
        lines = []

        def rec(node, prefix, d, last):
            r = self.records.get(node, {})
            tag = f"{node} (gen {r.get('generation', '?')}, L{r.get('level', 0)}, {r.get('operator', '')})"
            lines.append(prefix + ("└─ " if last else "├─ ") + tag if prefix or d < depth else tag)
            if d <= 0:
                return
            kids = self.children.get(node, [])[:max_children]
            for i, k in enumerate(kids):
                rec(k, prefix + ("   " if last else "│  "), d - 1, i == len(kids) - 1)
        anc = self.ancestors(gid, 3)
        root = anc[-1] if anc else gid
        lines.append(f"ancestor chain: {' <- '.join([gid] + anc)}")
        rec(root, "", depth, True)
        return "\n".join(lines)

    def prune(self, keep: set, current_gen: int, horizon: int = 30):
        needed = set()
        for gid in keep:
            needed.add(gid); needed.update(self.ancestors(gid))
        for gid in list(self.records.keys()):
            if gid not in needed and current_gen - self.records[gid]["generation"] > horizon:
                del self.records[gid]
        for p in list(self.children.keys()):
            self.children[p] = [c for c in self.children[p] if c in self.records]
            if p not in self.records and not self.children[p]:
                del self.children[p]

    def state_dict(self):
        return {"records": self.records, "children": self.children}

    def load_state_dict(self, s):
        self.records, self.children = dict(s["records"]), {k: list(v) for k, v in s["children"].items()}


class Speciation:
    """Espécies por distância combinada de genótipo e comportamento (fenótipo)."""

    def __init__(self, threshold: float):
        self.threshold = threshold
        self.reps: Dict[int, Dict[str, Any]] = {}
        self.next_id = 0
        self.extinct: List[Dict[str, Any]] = []
        self.births = 0

    def _matrix(self):
        if getattr(self, "_cache", None) is None or self._cache[0] != len(self.reps):
            ids = list(self.reps.keys())
            G = np.stack([self.reps[i]["g"] for i in ids]) if ids else None
            Bm = np.stack([self.reps[i]["b"] for i in ids]) if ids else None
            self._cache = (len(self.reps), ids, G, Bm)
        return self._cache[1], self._cache[2], self._cache[3]

    def assign(self, gvec: np.ndarray, bvec: np.ndarray, gen: int) -> int:
        best, bd = None, 1e9
        ids, G, Bm = self._matrix()
        if ids:
            dg = np.sqrt(np.mean((G - gvec[None]) ** 2, axis=1)) if G.shape[1] == gvec.shape[0] else np.ones(len(ids))
            db = np.sqrt(np.mean((Bm - bvec[None]) ** 2, axis=1))
            d = 0.5 * dg + 0.5 * db
            j = int(np.argmin(d))
            best, bd = ids[j], float(d[j])
        if best is not None and bd < self.threshold:
            self.reps[best]["last_seen"] = gen
            self.reps[best]["members_gen"] = self.reps[best].get("members_gen", 0) + 1
            return best
        sid = self.next_id; self.next_id += 1; self.births += 1
        self.reps[sid] = {"g": gvec.copy(), "b": bvec.copy(), "born": gen, "last_seen": gen, "members_gen": 1}
        return sid

    def end_generation(self, gen: int, alive: set, patience: int = 10):
        for sid in list(self.reps.keys()):
            if sid not in alive and gen - self.reps[sid]["last_seen"] > patience:
                r = self.reps.pop(sid)
                self._cache = None
                self.extinct.append({"species_id": sid, "born": r["born"], "extinct_at": gen})
        for r in self.reps.values():
            r["members_gen"] = 0

    def state_dict(self):
        return {"threshold": self.threshold, "reps": self.reps, "next_id": self.next_id, "extinct": self.extinct[-5000:],
                "births": self.births}

    def load_state_dict(self, s):
        self.threshold, self.next_id, self.births = s["threshold"], s["next_id"], s["births"]
        self.reps = {int(k): v for k, v in s["reps"].items()}
        self.extinct = list(s["extinct"])
        self._cache = None


class SurpriseModel:
    """Modelo histórico do comportamento (média/variância de Welford). Surpresa = RMS dos z-scores."""

    def __init__(self, dim: int):
        self.n = 0
        self.mean = np.zeros(dim); self.m2 = np.zeros(dim)

    def score(self, V: np.ndarray) -> np.ndarray:
        if self.n < 20:
            return np.zeros(len(V))
        sd = np.sqrt(self.m2 / max(1, self.n - 1)) + 0.02
        return np.sqrt(np.mean(((V - self.mean) / sd) ** 2, axis=1))

    def update(self, V: np.ndarray):
        for v in V:
            self.n += 1
            d = v - self.mean
            self.mean += d / self.n
            self.m2 += d * (v - self.mean)

    def state_dict(self):
        return {"n": self.n, "mean": self.mean, "m2": self.m2}

    def load_state_dict(self, s):
        self.n, self.mean, self.m2 = s["n"], np.asarray(s["mean"]), np.asarray(s["m2"])


class OperatorBandit:
    """UCB1 sobre operadores de mutação (mede quais produzem melhorias e ajusta probabilidades)."""

    def __init__(self, ops: List[str]):
        self.ops = list(ops)
        self.n = {o: 1.0 for o in ops}
        self.r = {o: 0.5 for o in ops}
        self.total = float(len(ops))

    def select(self, rng: np.random.Generator) -> str:
        if rng.random() < 0.15:
            return str(rng.choice(self.ops))
        ucb = {o: self.r[o] / self.n[o] + math.sqrt(2 * math.log(self.total) / self.n[o]) for o in self.ops}
        return max(ucb, key=ucb.get)

    def update(self, op: str, reward: float):
        if op not in self.n:
            return
        self.n[op] += 1; self.r[op] += float(reward); self.total += 1
        if self.total > 5000:   # esquecimento lento (não-estacionário)
            for o in self.ops:
                self.n[o] *= 0.5; self.r[o] *= 0.5
            self.total = sum(self.n.values())

    def probs(self) -> Dict[str, float]:
        return {o: round(self.r[o] / self.n[o], 3) for o in self.ops}

    def state_dict(self):
        return {"ops": self.ops, "n": self.n, "r": self.r, "total": self.total}

    def load_state_dict(self, s):
        self.ops, self.n, self.r, self.total = s["ops"], s["n"], s["r"], s["total"]

## 8. Escada de descoberta, scorecard, evidência e GENESIS SCIENTIST
**Fitness ≠ evidência.** Fitness conduz a seleção; evidência mede quanto sabemos (p-valores, replicação, controles).
A escada é sequencial (um nível só conta se todos os anteriores foram demonstrados) e cada promoção guarda o
`result_id` que a sustenta (auditabilidade).

O **Scientist** é baseado em regras + valor de informação: prioriza experimentos cujo resultado é mais incerto e mais
informativo por unidade de custo, nunca repete o mesmo teste de seleção no mesmo candidato (anti p-hacking),
usa cache por (genoma, probe, versão), evita re-testar comportamentos quase idênticos a falhas já conhecidas
e dá prioridade a anomalias.

In [ ]:
# @title 8. Escada de descoberta, scorecard, evidência e GENESIS SCIENTIST { display-mode: "form" }
LEVEL_NAMES = {0: "ACTIVITY", 1: "STRUCTURE", 2: "CAUSALITY", 3: "MEMORY", 4: "CHANNEL", 5: "TRANSFORMATION",
               6: "COMPOSITION", 7: "GENERALIZATION", 8: "ADAPTATION", 9: "EMERGENT COMPUTATIONAL ARCHITECTURE",
               10: "RESERVED"}
SCORECARD_FIELDS = ["activity", "persistence", "mobility", "complexity", "causality", "memory", "channel",
                    "transformation", "composition", "generalization", "recovery", "robustness", "adaptation",
                    "replication", "novelty"]
PROBE_LEVEL = {"causality": 2, "memory": 3, "channel": 4, "transformation": 5, "composition": 6, "generalization": 7,
               "adaptation": 8}
PROBE_COST_DEFAULT = {"causality": 1.0, "memory": 2.0, "channel": 2.0, "transformation": 2.0, "composition": 3.0,
                      "generalization": 3.0, "recovery": 1.5, "robustness": 4.0, "adaptation": 4.0, "replication": 12.0,
                      "ablation": 15.0, "long_screen": 2.0, "advanced": 10.0, "functional_recovery": 2.5,
                      "channel_noise": 2.0, "champion_lab": 60.0}


class DiscoveryLadder:
    def __init__(self, th: Thresholds):
        self.th = th

    def evaluate(self, d: Optional[Dict[str, float]], status: Dict[str, str], replication_ok: bool) -> Tuple[int, List[str]]:
        ev: List[str] = []
        if d is None or d.get("viability") in ("DEAD", "NUMERICALLY_UNSTABLE", "EXPLOSIVE"):
            return -1, ev
        level = -1
        if d.get("rel_activity", 0) >= self.th.frozen_rel_activity or d.get("viability") == "VIABLE":
            level = 0; ev.append("L0: atividade dinâmica na triagem multi-ambiente")
        else:
            return level, ev
        if d.get("persistence", 0) >= self.th.persistence and d.get("structures", 0) >= 1:
            level = 1; ev.append(f"L1: persistência={d['persistence']:.2f}, estruturas≈{d['structures']:.1f}")
        else:
            return level, ev
        for probe, lv in sorted(PROBE_LEVEL.items(), key=lambda x: x[1]):
            if status.get(probe) == STATUS_PASS:
                level = lv; ev.append(f"L{lv}: {probe} PASS")
            else:
                return level, ev
        core = all(status.get(p) == STATUS_PASS for p in ["memory", "channel", "transformation", "composition", "generalization"])
        if core and status.get("robustness") == STATUS_PASS and replication_ok:
            level = 9; ev.append("L9: armazenamento+transmissão+transformação+composição+robustez+generalização+replicação")
        return level, ev


def evidence_score(results: List[TestResult]) -> float:
    """Separado do fitness: soma de evidência estatística ponderada por nível e replicação."""
    s = 0.0
    rep_rate = {r.details.get("target"): r.score for r in results if r.probe == "replication"}
    for r in results:
        if r.status != STATUS_PASS or r.probe not in PROBE_LEVEL:
            continue
        p = r.p_value if (r.p_value is not None and np.isfinite(r.p_value) and r.p_value > 0) else 1e-6
        s += min(10.0, -math.log10(max(p, 1e-10))) * (1 + PROBE_LEVEL[r.probe] / 3) * (0.5 + 0.5 * rep_rate.get(r.probe, 0.0))
    return float(s)


@dataclass
class Experiment:
    candidate_id: str
    probe: str
    kwargs: Dict[str, Any]
    priority: float
    reason: str
    created_gen: int
    exp_id: str = ""

    def __post_init__(self):
        if not self.exp_id:
            self.exp_id = new_id("E")


class GenesisScientist:
    """Decide o próximo experimento. Pluggable: `self.advisor` pode ser trocado por outro pesquisador (ex.: LLM),
    mas nada aqui depende disso."""

    def __init__(self, cfg: GenesisConfig):
        self.cfg, self.th = cfg, cfg.thresholds
        self.queue: List[Experiment] = []
        self.done: Dict[str, str] = {}             # chave (cand|probe|args) -> result_id
        self.cache: Dict[str, str] = {}            # (genome_hash|probe|version|args) -> result_id
        self.stats: Dict[str, Dict[str, float]] = {}
        self.failed_behaviors: Dict[str, List[np.ndarray]] = {}
        self.queued: set = set()                   # chaves já na fila (deduplicação O(1))
        self._fail_cache: Dict[str, Tuple[int, np.ndarray, Any]] = {}
        self.advisor: Optional[Callable] = None

    @staticmethod
    def key(cid: str, probe: str, kwargs: Dict[str, Any]) -> str:
        return f"{cid}|{probe}|{json.dumps(to_jsonable(kwargs), sort_keys=True)}"

    def p_pass(self, probe: str) -> float:
        s = self.stats.get(probe, {"n": 0, "pass": 0})
        return (s["pass"] + 1) / (s["n"] + 2)

    def cost(self, probe: str) -> float:
        s = self.stats.get(probe)
        if s and s.get("n", 0) > 0:
            return max(0.05, s["time"] / s["n"])
        return PROBE_COST_DEFAULT.get(probe, 2.0)

    @staticmethod
    def probe_level(probe: str, kwargs: Dict[str, Any], level: int) -> float:
        """Degrau equivalente: testes de validação valem o nível que validam (replicação/ablação/robustez)."""
        if probe in PROBE_LEVEL:
            return PROBE_LEVEL[probe]
        if probe == "replication":
            return PROBE_LEVEL.get(kwargs.get("target", "memory"), 3)
        if probe in ("robustness", "channel_noise", "functional_recovery"):
            return max(2.0, level)
        if probe == "ablation":
            return 0.7 * PROBE_LEVEL.get(kwargs.get("target", "memory"), 3)
        if probe == "champion_lab":
            return 10.0
        return 1.5

    def value_of_information(self, probe: str, bonus: float, level: int = 0, kwargs: Optional[Dict[str, Any]] = None) -> float:
        """Ganho esperado de conhecimento por custo: incerteza do resultado (p(1-p)), altura do degrau na escada e
        quão promissor o candidato já é (candidatos mais altos na escada recebem os experimentos caros)."""
        p = self.p_pass(probe)
        gain = (1.0 + self.probe_level(probe, kwargs or {}, level)) ** 2
        depth = 1.0 + max(0, level)
        return (gain * (0.25 + p * (1 - p)) * depth + bonus) / (1.0 + self.cost(probe))

    def similar_to_failure(self, probe: str, vec: Optional[np.ndarray]) -> bool:
        lst = self.failed_behaviors.get(probe)
        if vec is None or not lst:
            return False
        ent = self._fail_cache.get(probe)
        if ent is None or ent[0] != len(lst) or ent[2] is not lst[-1]:
            ent = (len(lst), np.asarray(lst[-500:], np.float32), lst[-1])
            self._fail_cache[probe] = ent
        d = np.sqrt(np.mean((ent[1] - np.asarray(vec, np.float32)[None]) ** 2, axis=1))
        return bool((d < 0.015).any())

    def propose(self, cid: str, d: Dict[str, float], status: Dict[str, str], level: int, anomalous: bool, gen: int,
                vec: Optional[np.ndarray] = None, is_top: bool = False) -> List[Experiment]:
        th = self.th
        out: List[Experiment] = []
        bonus = (1.0 if anomalous else 0.0) + 0.25 * max(0, level) + 0.3 * sum(v == STATUS_PASS for v in status.values())

        def want(probe, reason, extra=0.0, **kw):
            k = self.key(cid, probe, kw)
            if k in self.done or k in self.queued or any(self.key(e.candidate_id, e.probe, e.kwargs) == k for e in out):
                return
            if probe in PROBE_LEVEL and not anomalous and self.similar_to_failure(probe, vec):
                return
            out.append(Experiment(cid, probe, kw, self.value_of_information(probe, bonus + extra, level, kw), reason, gen))
        if d.get("viability") not in ("VIABLE", "FROZEN"):
            return out
        if is_top or anomalous:
            want("long_screen", "sobrevivente de triagem (multi-fidelidade)" if is_top else "anomalia", 0.5)
        if "causality" not in status and d.get("causal_spread", 0) > th.causal_spread:
            want("causality", f"causal_spread={d['causal_spread']:.2f}>thr")
        elif "causality" not in status and (level >= 1 and any(v == STATUS_PASS for v in status.values())):
            want("causality", "pré-requisito da escada (outro probe já passou)", 1.0)
        if d.get("persistence", 0) > th.persistence and "recovery" not in status:
            want("recovery", f"persistence={d['persistence']:.2f}>thr")
        if status.get("causality") == STATUS_PASS and "memory" not in status:
            want("memory", "causality PASS -> memory probe", 0.5)
        if d.get("persistence", 0) > th.persistence and "memory" not in status and level >= 1:
            want("memory", "estrutura persistente -> memory probe")
        if status.get("memory") == STATUS_PASS and "channel" not in status:
            want("channel", "memory PASS -> channel probe", 1.0)
        if status.get("channel") == STATUS_PASS:
            if "transformation" not in status:
                want("transformation", "channel PASS -> transformation probe", 1.0)
            if "robustness" not in status:
                want("robustness", "candidato forte -> bateria de robustez")
            if "channel_noise" not in status:
                want("channel_noise", "robustez do canal ao ruído", 0.0, noise_level=self.cfg.probe.channel_noise * 3)
            if status.get("recovery") == STATUS_PASS and "functional_recovery" not in status:
                want("functional_recovery", "recuperação estrutural -> teste funcional pós-dano", 0.5, pre_damage=0.5)
            if "generalization" not in status and status.get("transformation") != STATUS_PASS:
                want("generalization", "channel PASS -> entradas/ambientes não vistos", 0.5, task="channel")
        if status.get("transformation") == STATUS_PASS:
            if "composition" not in status:
                want("composition", "transformation PASS -> composition", 1.0)
            if "generalization" not in status:
                want("generalization", "transformation PASS -> generalization", 1.0, task="xor")
        if status.get("generalization") == STATUS_PASS:
            if "adaptation" not in status:
                want("adaptation", "generalization PASS -> adaptation", 1.0)
            if "replication_channel" not in status:
                want("replication", "descoberta relevante -> replicação independente", 1.0, target="channel")
        if status.get("memory") == STATUS_PASS and "replication_memory" not in status:
            want("replication", "memória -> replicação independente (sementes novas, ambientes de validação)", 0.3,
                 target="memory", n_seeds=self.cfg.probe.replication_seeds)
        passed = [p for p in PROBE_LEVEL if status.get(p) == STATUS_PASS]
        if passed and "ablation" not in status:
            top = max(passed, key=lambda p: PROBE_LEVEL[p])
            if PROBE_LEVEL[top] >= 3:
                want("ablation", f"descoberta {top} -> ablação (o que realmente importa?)", 0.2, target=top)
        if level >= th.champion_level and "champion_lab" not in status:
            want("champion_lab", f"LEVEL {level} -> ChampionLab (bateria final em holdout)", 5.0)
        if self.advisor is not None:
            try:
                out.extend(self.advisor(cid, d, status, level) or [])
            except Exception as e:
                print(f"[scientist] advisor falhou: {e}")
        return out

    EXPLORATION_PROBES = ("causality", "long_screen", "recovery")

    def _eff(self, e: Experiment, gen: int) -> float:
        return e.priority * (1 + 0.15 * max(0, gen - e.created_gen))      # envelhecimento: sem inanição

    def _rebuild_queued(self):
        self.queued = {self.key(e.candidate_id, e.probe, e.kwargs) for e in self.queue}

    def push(self, exps: List[Experiment], gen: int = 0):
        for e in exps:
            k = self.key(e.candidate_id, e.probe, e.kwargs)
            if k in self.queued or k in self.done:
                continue
            self.queue.append(e); self.queued.add(k)
        self.queue.sort(key=lambda e: -self._eff(e, gen))
        if len(self.queue) > 2000:
            self.queue = self.queue[:2000]
            self._rebuild_queued()

    def pop(self, gen: Optional[int] = None) -> Optional[Experiment]:
        """Balanço exploração/exploração: 1 em cada 3 experimentos vai para triagem ampla (exploração),
        os demais para o experimento de maior valor de informação (aprofundamento)."""
        gen = 0 if gen is None else gen
        self._pops = getattr(self, "_pops", 0) + 1
        self.queue.sort(key=lambda e: -self._eff(e, gen))
        explore = self._pops % 3 == 0
        order = ([i for i, e in enumerate(self.queue) if e.probe in self.EXPLORATION_PROBES] if explore else []) + \
            list(range(len(self.queue)))
        for i in order:
            if i >= len(self.queue):
                continue
            e = self.queue[i]
            if self.key(e.candidate_id, e.probe, e.kwargs) in self.done:
                continue
            self.queue.pop(i)
            self.queued.discard(self.key(e.candidate_id, e.probe, e.kwargs))
            return e
        self.queue = [e for e in self.queue if self.key(e.candidate_id, e.probe, e.kwargs) not in self.done]
        self._rebuild_queued()
        return None

    def record(self, e: Experiment, result_id: str, status: str, duration: float, vec: Optional[np.ndarray]):
        self.done[self.key(e.candidate_id, e.probe, e.kwargs)] = result_id
        s = self.stats.setdefault(e.probe, {"n": 0, "pass": 0, "time": 0.0})
        s["n"] += 1; s["time"] += duration; s["pass"] += int(status == STATUS_PASS)
        if status == STATUS_FAIL and vec is not None and e.probe in PROBE_LEVEL:
            self.failed_behaviors.setdefault(e.probe, []).append(vec)
            self.failed_behaviors[e.probe] = self.failed_behaviors[e.probe][-500:]

    def drop_candidates(self, alive: set):
        self.queue = [e for e in self.queue if e.candidate_id in alive]
        self._rebuild_queued()

    def state_dict(self):
        return {"queue": [asdict(e) for e in self.queue], "done": self.done, "cache": self.cache, "stats": self.stats,
                "failed_behaviors": {k: np.asarray(v) for k, v in self.failed_behaviors.items()}}

    def load_state_dict(self, s):
        self.queue = [Experiment(**e) for e in s["queue"]]
        self.done, self.cache, self.stats = dict(s["done"]), dict(s["cache"]), dict(s["stats"])
        self.failed_behaviors = {k: [np.asarray(x) for x in v] for k, v in s["failed_behaviors"].items()}
        self._fail_cache = {}
        self._rebuild_queued()

## 9. Checkpoint atômico, validação por hash, migração e backups
`write checkpoint.tmp → validate → checkpoint_current → (anterior vira checkpoint_previous)`.
Corrupção detectada por hash → fallback automático para o anterior, depois para backups.
Carregamento sempre com `map_location="cpu"` (independente de hardware).

In [ ]:
# @title 9. Checkpoint atômico, validação por hash, migração e backups { display-mode: "form" }
CHECKPOINT_REQUIRED_KEYS = ["version", "checkpoint_version", "genesis_version", "config", "counters", "population",
                            "store", "map_elites", "novelty", "zoo", "anomalies", "failures", "champions", "lineage",
                            "species", "scientist", "rng", "hashes", "history", "results", "tests"]


def _ckpt_hashes(state: dict) -> Dict[str, str]:
    return {"population": hash_obj(state["population"]),
            "store": hash_obj({k: v["packed"] for k, v in state["store"].items()}),
            "map_elites": hash_obj(state["map_elites"]["cells"]),
            "counters": hash_obj(state["counters"])}


def migrate_checkpoint(state: dict) -> dict:
    """Estrutura de migração: aplica conversores sucessivos v -> v+1."""
    MIGRATIONS: Dict[int, Callable[[dict], dict]] = {
        # exemplo para versões futuras: 1: lambda s: {**s, "novo_campo": default, "checkpoint_version": 2},
    }
    v = int(state.get("checkpoint_version", 1))
    while v < CHECKPOINT_VERSION:
        if v not in MIGRATIONS:
            raise RuntimeError(f"sem migração de checkpoint v{v} -> v{v + 1}")
        state = MIGRATIONS[v](state)
        v = int(state["checkpoint_version"])
    return state


def _torch_load(path: str):
    try:
        return torch.load(path, map_location="cpu", weights_only=False)
    except TypeError:
        return torch.load(path, map_location="cpu")


class CheckpointManager:
    def __init__(self, paths: GenesisPaths, logger: EventLogger, keep_backups: int = 3, backup_every: int = 5):
        self.paths, self.logger = paths, logger
        self.current = os.path.join(paths.checkpoints, "checkpoint_current.pt")
        self.previous = os.path.join(paths.checkpoints, "checkpoint_previous.pt")
        self.tmp = os.path.join(paths.checkpoints, "checkpoint.tmp")
        self.keep_backups, self.backup_every = keep_backups, backup_every
        self.saves = 0
        self.last_save_ts: Optional[float] = None
        self.last_path: Optional[str] = None

    def validate(self, path: str) -> Tuple[bool, Optional[dict], str]:
        try:
            st = _torch_load(path)
        except Exception as e:
            return False, None, f"load failed: {e}"
        missing = [k for k in CHECKPOINT_REQUIRED_KEYS if k not in st]
        if missing:
            return False, None, f"missing keys: {missing}"
        try:
            h = _ckpt_hashes(st)
        except Exception as e:
            return False, None, f"hash failed: {e}"
        if h != st["hashes"]:
            bad = [k for k in h if h[k] != st["hashes"].get(k)]
            return False, None, f"hash mismatch: {bad}"
        return True, st, "ok"

    def save(self, state: dict) -> str:
        state = dict(state)
        state["hashes"] = _ckpt_hashes(state)
        t0 = time.time()
        torch.save(state, self.tmp)
        ok, _, msg = self.validate(self.tmp)
        if not ok:
            self.logger.event("checkpoint_invalid", echo=True, reason=msg)
            raise RuntimeError(f"checkpoint temporário inválido: {msg}")
        if os.path.exists(self.current):
            os.replace(self.current, self.previous)
        os.replace(self.tmp, self.current)
        self.saves += 1
        if self.saves % self.backup_every == 0:
            bk = os.path.join(self.paths.backups, f"checkpoint_gen{state['counters']['generation']:07d}.pt")
            shutil.copy2(self.current, bk)
            bks = sorted(glob.glob(os.path.join(self.paths.backups, "checkpoint_gen*.pt")))
            for old in bks[:-self.keep_backups]:
                os.remove(old)
        self.last_save_ts = time.time()
        self.last_path = self.current
        size = os.path.getsize(self.current)
        self.logger.event("checkpoint_saved", generation=state["counters"]["generation"], size=size,
                          seconds=round(time.time() - t0, 2))
        return self.current

    def load_latest(self) -> Tuple[Optional[dict], str]:
        candidates = [self.current, self.previous] + sorted(glob.glob(os.path.join(self.paths.backups, "checkpoint_gen*.pt")), reverse=True)
        for p in candidates:
            if not os.path.exists(p):
                continue
            ok, st, msg = self.validate(p)
            if ok:
                try:
                    st = migrate_checkpoint(st)
                except Exception as e:
                    self.logger.event("checkpoint_migration_failed", echo=True, path=p, error=repr(e))
                    continue
                if p != self.current:
                    self.logger.event("checkpoint_fallback", echo=True, used=p)
                return st, p
            self.logger.event("checkpoint_corrupt", echo=True, path=p, reason=msg)
        return None, ""

## 10. Visualização (somente observacional — nunca participa do fitness) e relatórios
Projeções: PCA dos canais → RGB, seleção de canais, mapa de magnitude. Animações GIF somente para descobertas importantes.

In [ ]:
# @title 10. Visualização (somente observacional — nunca participa do fitness) e relatórios { display-mode: "form" }
def rgb_projection(U: np.ndarray, mode: str = "pca", channels: Tuple[int, int, int] = (0, 1, 2),
                   basis: Optional[np.ndarray] = None) -> np.ndarray:
    """U [C,S,S] -> RGB [S,S,3] em [0,1]. Com C>3, três canais nunca são tratados como representação total."""
    C, S, _ = U.shape
    if mode == "magnitude":
        m = np.sqrt((U ** 2).sum(0)); m = m / (m.max() + 1e-9)
        return plt.cm.magma(m)[..., :3]
    if mode == "channels":
        ch = [c if c < C else 0 for c in channels]
        X = np.stack([U[c] for c in ch], -1)
    else:
        Xf = U.reshape(C, -1).T
        Xf = Xf - Xf.mean(0, keepdims=True)
        if basis is None:
            try:
                _, _, Vt = np.linalg.svd(Xf, full_matrices=False)
                basis = Vt[:3]
            except Exception:
                basis = np.eye(C)[:3]
        if basis.shape[0] < 3:
            basis = np.vstack([basis, np.zeros((3 - basis.shape[0], C))])
        X = (Xf @ basis.T).reshape(S, S, 3)
    lo, hi = np.percentile(X, 1), np.percentile(X, 99)
    return np.clip((X - lo) / (hi - lo + 1e-9), 0, 1)


def simulate_frames(lab: "GenesisLab", genome: Genome, env: Optional[Environment] = None, steps: int = 200,
                    every: int = 4, seed: int = 0, S: Optional[int] = None) -> Tuple[List[np.ndarray], np.ndarray]:
    env = env or lab.envs[0]
    gb = lab.sim.prepare([genome], S or lab.cfg.world_size)
    st = lab.sim.init_state(gb, [env], [derive_seed(seed, genome.genome_id)], run_seed=seed)
    init = st.U[0].cpu().numpy()
    frames = []
    for t in range(steps):
        st = lab.sim.step(gb, st)
        if t % every == 0:
            frames.append(st.U[0].cpu().numpy())
    return frames, init


def save_animation(frames: List[np.ndarray], path: str, mode: str = "pca", fps: int = 12) -> Optional[str]:
    try:
        from PIL import Image
        Xall = np.concatenate([f.reshape(f.shape[0], -1).T for f in frames[:: max(1, len(frames) // 10)]])
        Xall = Xall - Xall.mean(0, keepdims=True)
        _, _, Vt = np.linalg.svd(Xall, full_matrices=False)
        imgs = []
        for f in frames:
            rgb = (rgb_projection(f, mode, basis=Vt[:3]) * 255).astype(np.uint8)
            im = Image.fromarray(rgb).resize((rgb.shape[1] * 4, rgb.shape[0] * 4), Image.NEAREST)
            imgs.append(im)
        imgs[0].save(path, save_all=True, append_images=imgs[1:], duration=int(1000 / fps), loop=0)
        return path
    except Exception as e:
        print(f"[viz] animação não gerada: {e}")
        return None


def _finish(fig, path: Optional[str], show: bool):
    if path:
        fig.savefig(path, dpi=110, bbox_inches="tight")
    if show and IN_NOTEBOOK:
        plt.show()
    plt.close(fig)


def plot_frames(frames: List[np.ndarray], title: str = "", path: Optional[str] = None, show: bool = True, n: int = 6):
    idx = np.linspace(0, len(frames) - 1, min(n, len(frames))).astype(int)
    fig, ax = plt.subplots(2, len(idx), figsize=(2.2 * len(idx), 4.6))
    ax = np.atleast_2d(ax)
    for j, i in enumerate(idx):
        ax[0, j].imshow(rgb_projection(frames[i], "pca")); ax[0, j].set_title(f"t≈{i}", fontsize=8)
        ax[1, j].imshow(rgb_projection(frames[i], "magnitude"))
        ax[0, j].axis("off"); ax[1, j].axis("off")
    ax[0, 0].set_ylabel("PCA→RGB"); ax[1, 0].set_ylabel("|U|")
    fig.suptitle(title, fontsize=10)
    _finish(fig, path, show)


def plot_causal_map(res: TestResult, path: Optional[str] = None, show: bool = True):
    maps, times = res.details.get("causal_map", []), res.details.get("map_times", [])
    if not maps:
        return
    fig, ax = plt.subplots(1, len(maps), figsize=(2 * len(maps), 2.3))
    ax = np.atleast_1d(ax)
    vmax = max(float(np.max(np.asarray(m, np.float32))) for m in maps) + 1e-12
    for a, m, t in zip(ax, maps, times):
        a.imshow(np.log10(np.asarray(m, np.float32) + vmax * 1e-4), cmap="inferno"); a.set_title(f"t={t}", fontsize=8); a.axis("off")
    fig.suptitle(f"Mapa causal log10|U_A-U_B| — {res.candidate_id} ({res.status})", fontsize=9)
    _finish(fig, path, show)


def plot_memory_curve(res: TestResult, path: Optional[str] = None, show: bool = True):
    cv = res.details.get("curve", [])
    if not cv:
        return
    fig, ax = plt.subplots(figsize=(5, 3))
    T = [c["T"] for c in cv]
    ax.plot(T, [c["acc"] for c in cv], "o-", label="história (ridge CV)")
    ax.plot(T, [c["null_acc"] for c in cv], "s--", label="controle nulo")
    ax.scatter([c["T"] for c in cv if c["sig"]], [c["acc"] for c in cv if c["sig"]], s=120, facecolors="none",
               edgecolors="g", label="significativo (Holm)")
    ax.axhline(0.5, color="gray", lw=0.8); ax.set_xscale("log"); ax.set_ylim(0, 1.05)
    ax.set_xlabel("atraso T"); ax.set_ylabel("acurácia balanceada"); ax.set_title(f"M(T) — {res.candidate_id}", fontsize=9)
    ax.legend(fontsize=7)
    _finish(fig, path, show)


def plot_damage_curve(res: TestResult, path: Optional[str] = None, show: bool = True):
    cv = res.details.get("curve", [])
    if not cv:
        return
    fig, ax = plt.subplots(figsize=(5, 3))
    x = [c["damage"] * 100 for c in cv]
    ax.plot(x, [c["sim_immediate"] for c in cv], "o--", label="logo após dano")
    ax.plot(x, [c["sim_recovered"] for c in cv], "o-", label="após recuperação")
    ax.plot(x, [c["recovery"] for c in cv], "s-", label="Recovery (relativa)")
    ax.set_xlabel("dano (%)"); ax.set_ylim(-1.05, 1.05); ax.legend(fontsize=7)
    ax.set_title(f"Recovery(damage%) — {res.candidate_id}", fontsize=9)
    _finish(fig, path, show)


def plot_robustness_curve(res: TestResult, path: Optional[str] = None, show: bool = True):
    cv = res.details.get("noise_curve", [])
    conds = res.details.get("conditions", [])
    if not conds:
        return
    fig, ax = plt.subplots(1, 2, figsize=(10, 3))
    ax[0].plot([c["noise"] for c in cv], [c["similarity"] for c in cv], "o-")
    ax[0].set_xlabel("ruído gaussiano"); ax[0].set_ylabel("similaridade comportamental"); ax[0].set_ylim(0, 1.05)
    labels = [f"{c['condition']}\n{c['level']}" for c in conds]
    ax[1].bar(range(len(conds)), [c["similarity"] for c in conds], color=["tab:green" if c["ok"] else "tab:red" for c in conds])
    ax[1].set_xticks(range(len(conds))); ax[1].set_xticklabels(labels, rotation=70, fontsize=6); ax[1].set_ylim(0, 1.05)
    fig.suptitle(f"Robustez — {res.candidate_id} (score={res.score:.2f})", fontsize=9)
    _finish(fig, path, show)


def plot_map_elites(me: MapElites, path: Optional[str] = None, show: bool = True):
    H = me.heat()
    fig, ax = plt.subplots(figsize=(4.4, 3.6))
    im = ax.imshow(H.T, origin="lower", cmap="viridis", aspect="auto")
    ax.set_xlabel(me.axes[0]); ax.set_ylabel(me.axes[1])
    ax.set_title(f"MAP-Elites cobertura={me.coverage():.1%} (máx sobre {me.axes[2] if len(me.axes) > 2 else '-'})", fontsize=8)
    fig.colorbar(im, ax=ax, label="qualidade")
    _finish(fig, path, show)


def plot_convergence(history: List[Dict[str, Any]], path: Optional[str] = None, show: bool = True):
    if len(history) < 2:
        return
    g = [h["generation"] for h in history]
    fig, ax = plt.subplots(1, 4, figsize=(15, 2.8))
    ax[0].plot(g, [h["best_quality"] for h in history]); ax[0].set_title("melhor qualidade", fontsize=8)
    ax[1].plot(g, [h["map_coverage"] for h in history]); ax[1].set_title("cobertura MAP", fontsize=8)
    ax[2].plot(g, [h["species"] for h in history]); ax[2].set_title("espécies", fontsize=8)
    ax[3].plot(g, [h["best_level"] for h in history]); ax[3].set_title("melhor DiscoveryLevel", fontsize=8)
    for a in ax:
        a.set_xlabel("geração", fontsize=7)
    _finish(fig, path, show)

## 11. GENESIS LAB — orquestração do ciclo `CREATE → EVOLVE → OBSERVE → PERTURB → MEASURE → REJECT → PRESERVE → REPLICATE → GENERALIZE → CONTINUE`
* Universe Evolution Engine (este módulo) e Validation Engine (módulo 6) são separados: a evolução **nunca** altera
  probes, thresholds, checkpoint ou segurança (core imutável; meta-evolução só toca parâmetros de mutação do genoma).
* Seleção: MAP-Elites + novelty + frentes de Pareto multiobjetivo + limite por espécie + imigrantes aleatórios.
* Orçamento: triagem barata para todos; experimentos caros só para candidatos promovidos pelo Scientist.
* Estagnação: medidas graduais (mutação ↑, pressão de novidade ↑, novas espécies, reinício parcial, mudança ambiental).
  Elites históricas nunca são apagadas.

In [ ]:
# @title 11. GENESIS LAB — orquestração do ciclo CREATE → EVOLVE → OBSERVE → PERTURB → MEASURE → REJECT → PRESERVE → REPLICATE → GENERALIZE → CONTINUE { display-mode: "form" }
def pareto_ranks(Obj: np.ndarray) -> np.ndarray:
    n = len(Obj)
    ranks = np.zeros(n, int)
    if n == 0:
        return ranks
    dom = (Obj[:, None, :] >= Obj[None, :, :]).all(-1) & (Obj[:, None, :] > Obj[None, :, :]).any(-1)
    remaining = np.ones(n, bool)
    r = 0
    while remaining.any():
        dominated = (dom[remaining][:, remaining]).any(0)
        idx = np.where(remaining)[0]
        front = idx[~dominated]
        if front.size == 0:
            front = idx
        ranks[front] = r
        remaining[front] = False
        r += 1
    return ranks


PROBE_ALIAS = {"channel_noise": "channel", "functional_recovery": "channel"}


class GenesisLab:
    def __init__(self, cfg: GenesisConfig, hw: Dict[str, Any], paths: GenesisPaths, logger: EventLogger,
                 device: torch.device):
        self.cfg, self.hw, self.paths, self.logger, self.device = cfg, hw, paths, logger, device
        self.space = ParamSpace(cfg)
        self.sim = Simulator(cfg, device)
        self.envs = make_environment_suite(cfg.master_seed)
        self.screener = Screener(self.sim, cfg)
        self.curriculum: Dict[str, float] = {"perturbation_mult": 1.0, "memory_delay_mult": 1.0, "channel_distance_add": 0}
        self.ve = ValidationEngine(self.sim, cfg, self.envs, self.screener, self.space, self.curriculum)
        self.ladder = DiscoveryLadder(cfg.thresholds)
        self.scientist = GenesisScientist(cfg)
        self.ckpt = CheckpointManager(paths, logger)
        self.store: Dict[str, Genome] = {}
        self.population: List[str] = []
        self.desc: Dict[str, Dict[str, Any]] = {}
        self.vec: Dict[str, np.ndarray] = {}
        self.quality: Dict[str, float] = {}
        self.objectives: Dict[str, np.ndarray] = {}
        self.novelty_score: Dict[str, float] = {}
        self.results: Dict[str, TestResult] = {}
        self.tests: Dict[str, Dict[str, str]] = {}
        self.levels: Dict[str, int] = {}
        self.evidence: Dict[str, float] = {}
        self.audit: List[Dict[str, Any]] = []
        self.map = MapElites(cfg.map_axes, cfg.map_bins)
        self.novelty = NoveltyArchive(cfg.novelty_k, cfg.novelty_archive_max)
        self.zoo = Zoo()
        self.anomalies = SimpleArchive("anomaly")
        self.failures = SimpleArchive("failure")
        self.champions = SimpleArchive("champion_candidates")
        self.pantheon: List[Dict[str, Any]] = []
        self.lineage = Lineage()
        self.species = Speciation(cfg.species_threshold)
        self.surprise = SurpriseModel(len(DESCRIPTOR_NAMES))
        self.bandit = OperatorBandit(MUTATION_OPERATORS)
        self.discoveries: Dict[str, Dict[str, Any]] = {}
        self.baselines: Dict[str, Dict[str, Any]] = {}
        self.counters = {"generation": 0, "evaluations": 0, "sim_steps": 0, "total_runtime": 0.0, "sessions": 0,
                         "experiments": 0, "oom_events": 0, "discovery_events": 0}
        self.history: List[Dict[str, Any]] = []
        self.stagnation = {"stage": 0, "best_quality": -1.0, "best_coverage": 0.0, "best_level": -1, "since": 0,
                           "sigma_mult": 1.0, "novelty_weight": 1.0}
        self.rng = np.random.default_rng(cfg.master_seed)
        self.champion: Optional[Dict[str, Any]] = None
        self.champion_lab_runs = 0
        self.metric_flags: List[Dict[str, Any]] = []
        self.session_start = time.time()
        self.runtime_at_start = 0.0
        self.last_checkpoint_ts = time.time()
        self.last_dashboard_ts = 0.0
        self.best_level_global = -1
        self.throughput = {"universe_steps_per_s": 0.0, "evals_per_hour_est": 0.0}
        self.stop_reason = ""
        self.last_viability: Dict[str, int] = {}

    # ==================================================================
    # utilidades de estado
    # ==================================================================
    def status_of(self, cid: str) -> Dict[str, str]:
        return {k: self.results[r].status for k, r in self.tests.get(cid, {}).items() if r in self.results}

    def result_of(self, cid: str, key: str) -> Optional[TestResult]:
        rid = self.tests.get(cid, {}).get(key)
        return self.results.get(rid) if rid else None

    def scorecard(self, cid: str) -> Dict[str, Any]:
        d = self.desc.get(cid)
        sc: Dict[str, Any] = {"candidate_id": cid}
        if d is not None:
            sc["activity"] = round(d["rel_activity"], 5)
            sc["persistence"] = round(d["persistence"], 4)
            sc["mobility"] = round(d["mobility"], 4)
            sc["complexity"] = round(0.5 * (d["temporal_complexity"] + d["spatial_complexity"]), 4)
            sc["novelty"] = round(float(self.novelty_score.get(cid, 0.0)), 4)
        for f in SCORECARD_FIELDS:
            if f in sc:
                continue
            key = "replication_channel" if f == "replication" and "replication_channel" in self.tests.get(cid, {}) else (
                "replication_memory" if f == "replication" else f)
            r = self.result_of(cid, key)
            sc[f] = STATUS_NOT_TESTED if r is None else (r.status if not np.isfinite(r.score) else f"{r.status}:{r.score:.3f}")
        sc["level"] = self.levels.get(cid, -1)
        sc["evidence"] = round(self.evidence.get(cid, 0.0), 3)
        sc["quality(fitness)"] = round(self.quality.get(cid, 0.0), 4)
        return sc

    def say(self, text: str):
        """Mensagens importantes: no painel ao vivo viram anúncios (sem poluir a saída); fora dele, print."""
        m = LIVE.monitor
        if m is not None and getattr(m, "mode", "text") == "html":
            m.announce(text)
        else:
            print(text)

    def runtime_total(self) -> float:
        return self.runtime_at_start + (time.time() - self.session_start)

    def probe_bonus(self, cid: str) -> float:
        st = self.status_of(cid)
        vals = [self.results[self.tests[cid][k]].score for k in ["memory", "channel", "causality"]
                if st.get(k) == STATUS_PASS and np.isfinite(self.results[self.tests[cid][k]].score)]
        return float(np.mean(vals)) if vals else 0.0

    # ==================================================================
    # CREATE
    # ==================================================================
    def initialize(self):
        n = self.cfg.population_size
        gs = [Genome(self.space.random_params(self.rng), generation=0, origin="random") for _ in range(n)]
        self.logger.event("genesis_created", echo=True, population=n, world_size=self.cfg.world_size,
                          channels_max=self.cfg.channels_max, profile=self.cfg.profile)
        self.evaluate_and_integrate(gs, parents={})
        self.population = [g.genome_id for g in gs]
        self.evaluate_baselines()

    def evaluate_baselines(self):
        """Baselines: lei nula, lei simples (reação-difusão), difusão pura, ruído, ancestral aleatório."""
        kinds = ["identity", "noise_replace", "diffusion", "reaction_diffusion"]
        gs = [control_genome(self.space, k, self.ve.channel_distance(self.cfg.world_size), self.cfg.world_size) for k in kinds]
        gs.append(Genome(self.space.random_params(np.random.default_rng(derive_seed(self.cfg.master_seed, "baseline-random"))),
                         origin="baseline:random"))
        ds = self.screener.screen(gs, self.ve.envs_for("evolution"), derive_seed(self.cfg.master_seed, "baselines"))
        for k, g, d in zip(kinds + ["random"], gs, ds):
            self.baselines[k] = {"descriptor": {kk: v for kk, v in d.items() if not isinstance(v, list)},
                                 "genome": genome_to_record(self.space, g), "version": VALIDATION_ENGINE_VERSION}

    # ==================================================================
    # EVOLVE
    # ==================================================================
    def select_parents(self, n: int) -> List[str]:
        rng = self.rng
        pop = [p for p in self.population if p in self.store]
        out: List[str] = []
        n_map = int(n * 0.35); n_nov = int(n * 0.2 * self.stagnation["novelty_weight"]); n_nov = min(n_nov, n // 2)
        out += [g for g in self.map.sample(rng, n_map) if g in self.store]
        nov_ids = [g for g in self.novelty.ids[-500:] if g in self.store]
        if nov_ids:
            out += [nov_ids[i] for i in rng.integers(0, len(nov_ids), size=n_nov)]
        if pop:
            obj = np.stack([self.objectives.get(p, np.zeros(7)) for p in pop])
            ranks = pareto_ranks(obj)
            nov = np.asarray([self.novelty_score.get(p, 0.0) for p in pop])
            while len(out) < n:
                a, b = rng.integers(0, len(pop), size=2)
                if ranks[a] != ranks[b]:
                    w = a if ranks[a] < ranks[b] else b
                else:
                    w = a if nov[a] >= nov[b] else b
                out.append(pop[w])
        while len(out) < n:
            out.append(str(rng.choice(list(self.store.keys()))))
        return out[:n]

    def make_offspring(self, n: int) -> Tuple[List[Genome], Dict[str, Tuple[str, str]]]:
        gen = self.counters["generation"]
        n_imm = max(0, int(round(n * self.cfg.immigrant_rate * (3 if self.stagnation["stage"] >= 3 else 1))))
        parents = self.select_parents(n - n_imm)
        kids: List[Genome] = []
        meta: Dict[str, Tuple[str, str]] = {}
        for pid in parents:
            p = self.store[pid]
            op = self.bandit.select(self.rng)
            other = None
            if op == "crossover":
                mate = self.store.get(str(self.rng.choice(self.population))) if self.population else None
                other = mate.params if mate is not None else None
                if other is None:
                    op = "gauss_small"
            params, rec = self.space.mutate(p.params, self.rng, op, self.stagnation["sigma_mult"], other)
            g = Genome(params=params, parent_ids=[pid] + ([mate.genome_id] if other is not None else []),
                       generation=gen, mutation_record=rec, origin="mutation", operator=op)
            kids.append(g); meta[g.genome_id] = (pid, op)
        for _ in range(n_imm):
            g = Genome(self.space.random_params(self.rng), generation=gen, origin="immigrant", operator="random_immigrant")
            kids.append(g)
        return kids, meta

    # ==================================================================
    # OBSERVE / MEASURE / PRESERVE
    # ==================================================================
    def evaluate_and_integrate(self, genomes: List[Genome], parents: Dict[str, Tuple[str, str]],
                               label: Optional[str] = None) -> Dict[str, Any]:
        gen = self.counters["generation"]
        t0 = time.time()
        steps0 = self.sim.steps_done
        descs = self.screener.screen(genomes, self.ve.envs_for("evolution"), derive_seed(self.cfg.master_seed, "screen", gen),
                                     label=label or f"Geração {gen}: triagem de {len(genomes)} universos")
        dt = time.time() - t0
        self.counters["evaluations"] += len(genomes)
        ds = self.sim.steps_done - steps0
        self.counters["sim_steps"] += ds
        if dt > 0:
            self.throughput["universe_steps_per_s"] = ds / dt
            self.throughput["evals_per_hour_est"] = len(genomes) / dt * 3600
        self.last_viability = dict(collections.Counter(d["viability"] for d in descs))
        V = np.stack([descriptor_vector(d) for d in descs])
        pop_vecs = np.stack([self.vec[p] for p in self.population if p in self.vec]) if self.population else None
        others = np.vstack([V, pop_vecs]) if pop_vecs is not None and len(pop_vecs) else V
        nov = self.novelty.novelty(V, others)
        surp = self.surprise.score(V)
        stats = {"new_cells": 0, "map_inserts": 0, "anomalies": 0, "viable": 0}
        viable_ids, viable_vecs, viable_nov = [], [], []
        live_progress(phase=f"Geração {gen}: arquivos, novidade, espécies", done=0, total=len(genomes),
                      detail="MAP-Elites, novelty archive, zoo, linhagem, surpresa")
        for i_, (g, d, v, nv, sp) in enumerate(zip(genomes, descs, V, nov, surp)):
            if i_ % 64 == 0:
                live_progress(done=i_)
            gid = g.genome_id
            self.store[gid] = g
            self.lineage.add(g)
            self.desc[gid] = d
            self.vec[gid] = v
            self.novelty_score[gid] = float(nv)
            obj = quality_objectives(d, self.cfg.thresholds, self.probe_bonus(gid))
            viable = d["viability"] == "VIABLE"
            q = float((obj * QUALITY_WEIGHTS).sum()) if viable else -1.0
            self.objectives[gid] = obj if viable else obj * 0 - 1
            self.quality[gid] = q
            reward = 0.0
            if viable:
                stats["viable"] += 1
                ins, new = self.map.try_insert(gid, d, q, gen)
                stats["map_inserts"] += int(ins); stats["new_cells"] += int(new)
                reward += 1.0 if new else (0.5 if ins else 0.0)
                viable_ids.append(gid); viable_vecs.append(v); viable_nov.append(nv)
                anomalous = bool(sp > self.cfg.thresholds.anomaly_surprise)
                if anomalous:
                    stats["anomalies"] += 1
                    self.anomalies.add(gid, "surprise", gen, surprise=float(sp))
                    self.logger.event("anomaly", genome_id=gid, surprise=float(sp))
                cats = Zoo.classify(d, {}, self.cfg.thresholds, anomalous)
                self.zoo.add(gid, cats, nv, gen)
                if d["causal_spread"] > 0.5 and d["lyapunov"] > self.cfg.thresholds.chaos_lyap * 2:
                    self.failures.add(gid, "high-causality chaos (triagem)", gen, lyapunov=d["lyapunov"])
                g.species_id = self.species.assign(self.space.vector(g.params), v, gen)
                level, ev = self.ladder.evaluate(d, {}, False)
                self.levels[gid] = level
                self.lineage.update(gid, quality=q, level=level, species_id=g.species_id)
            else:
                g.species_id = -1
                self.levels[gid] = -1
                self.lineage.update(gid, quality=q, level=-1)
            if gid in parents:
                pid, op = parents[gid]
                if self.quality.get(pid, -1) < q:
                    reward += 0.5
                self.bandit.update(op, min(1.0, reward))
        added = self.novelty.maybe_add(viable_ids, np.asarray(viable_vecs) if viable_vecs else np.zeros((0, V.shape[1])),
                                       np.asarray(viable_nov), self.rng)
        if viable_vecs:
            self.surprise.update(np.asarray(viable_vecs))
        stats.update({"novelty_added": len(added), "screen_seconds": dt, "n": len(genomes)})
        return stats

    def select_survivors(self, candidates: List[str]):
        cfg = self.cfg
        N = cfg.population_size
        pool = list(dict.fromkeys([c for c in candidates if c in self.store]))
        viable = [c for c in pool if self.quality.get(c, -1) >= 0]
        if len(viable) < max(4, N // 4):
            viable = sorted(pool, key=lambda c: -self.quality.get(c, -1))[:max(4, N // 4)]
        obj = np.stack([np.concatenate([self.objectives[c], [self.novelty_score.get(c, 0.0) * self.stagnation["novelty_weight"]]])
                        for c in viable])
        ranks = pareto_ranks(obj)
        nov = np.asarray([self.novelty_score.get(c, 0.0) for c in viable])
        order = np.lexsort((-nov, ranks))
        cap = max(2, int(N * cfg.species_cap_fraction))
        per_species: Dict[int, int] = {}
        chosen = []
        for i in order:
            c = viable[i]
            sid = self.store[c].species_id
            if per_species.get(sid, 0) >= cap:
                continue
            per_species[sid] = per_species.get(sid, 0) + 1
            chosen.append(c)
            if len(chosen) >= N:
                break
        if len(chosen) < N:
            rest = [viable[i] for i in order if viable[i] not in set(chosen)]
            chosen += rest[: N - len(chosen)]
        self.population = chosen

    # ==================================================================
    # PERTURB / MEASURE (experimentos)
    # ==================================================================
    def schedule_experiments(self, new_ids: List[str]):
        gen = self.counters["generation"]
        top = sorted([c for c in new_ids if self.quality.get(c, -1) >= 0], key=lambda c: -self.quality[c])[:3]
        exps = []
        cand = set(new_ids) | set(self.population[: min(len(self.population), 64)])
        for cid in cand:
            d = self.desc.get(cid)
            if d is None:
                continue
            anomalous = any(e["genome_id"] == cid for e in self.anomalies.entries[-200:])
            exps += self.scientist.propose(cid, d, self.status_of(cid), self.levels.get(cid, -1), anomalous, gen,
                                           self.vec.get(cid), is_top=cid in top)
        # replicação independente obrigatória para as descobertas mais relevantes (nível ≥ 3)
        ranked = sorted([c for c, l in self.levels.items() if l >= 3 and c in self.store],
                        key=lambda c: (-self.levels[c], -self.evidence.get(c, 0.0)))[:3]
        for cid in ranked:
            st = self.status_of(cid)
            target = max([p for p in ("memory", "channel", "transformation") if st.get(p) == STATUS_PASS],
                         key=lambda p: PROBE_LEVEL[p], default=None)
            if target is None or f"replication_{target}" in st:
                continue
            kw = {"target": target, "n_seeds": self.cfg.probe.replication_seeds}
            k_ = self.scientist.key(cid, "replication", kw)
            if k_ in self.scientist.done or k_ in self.scientist.queued:
                continue
            exps.append(Experiment(cid, "replication", kw, 100.0 + self.levels[cid],
                                   f"descoberta L{self.levels[cid]} -> replicação independente obrigatória", gen))
        self.scientist.push(exps, gen)

    def run_experiments(self, budget_s: float, max_n: int) -> int:
        t0 = time.time()
        n = 0
        while n < max_n and (time.time() - t0 < budget_s or n == 0):
            e = self.scientist.pop(self.counters["generation"])
            if e is None:
                break
            if e.candidate_id not in self.store:
                continue
            live_progress(phase=f"Geração {self.counters['generation']}: experimento {n + 1}/{max_n}", done=n, total=max_n,
                          detail=f"{e.probe} em {e.candidate_id} — {e.reason}")
            self.run_one_experiment(e)
            n += 1
        return n

    def status_key(self, e: Experiment) -> str:
        if e.probe == "replication":
            return f"replication_{e.kwargs.get('target', 'memory')}"
        return e.probe

    def run_one_experiment(self, e: Experiment):
        g = self.store[e.candidate_id]
        if e.probe == "champion_lab":
            self.champion_lab(e.candidate_id)
            self.scientist.record(e, "champion_lab", STATUS_PASS, 0.0, None)
            return
        probe = PROBE_ALIAS.get(e.probe, e.probe)
        cache_key = f"{g.hash()}|{probe}|{self.ve.PROBE_VERSIONS.get(probe)}|{json.dumps(to_jsonable(e.kwargs), sort_keys=True)}"
        seed = derive_seed(self.cfg.master_seed, "exp", e.candidate_id, e.probe, json.dumps(to_jsonable(e.kwargs), sort_keys=True))
        if cache_key in self.scientist.cache and self.scientist.cache[cache_key] in self.results:
            res = self.results[self.scientist.cache[cache_key]]
        else:
            kw = dict(e.kwargs)
            if probe not in ("replication", "ablation", "generalization", "adaptation"):
                kw.setdefault("split", "evolution")
            res = self.ve.run_probe(probe, g, seed, **kw)
            self.counters["experiments"] += 1
            self.scientist.cache[cache_key] = res.result_id
        res.details["experiment_reason"] = e.reason
        self.record_result(e.candidate_id, self.status_key(e), res)
        self.scientist.record(e, res.result_id, res.status, res.duration_s, self.vec.get(e.candidate_id))
        # experimentação ativa: o resultado gera imediatamente o próximo experimento mais informativo
        cid = e.candidate_id
        if cid in self.desc:
            anomalous = any(x["genome_id"] == cid for x in self.anomalies.entries[-200:])
            self.scientist.push(self.scientist.propose(cid, self.desc[cid], self.status_of(cid), self.levels.get(cid, -1),
                                                       anomalous, self.counters["generation"], self.vec.get(cid)),
                                self.counters["generation"])
        self.logger.event("experiment", candidate=e.candidate_id, probe=e.probe, status=res.status,
                          score=res.score, seconds=round(res.duration_s, 2), reason=e.reason)

    def record_result(self, cid: str, key: str, res: TestResult):
        if res.probe == "causality" and res.status != STATUS_PASS:
            res.details.pop("causal_map", None)      # política de armazenamento: mapas só para causalidade confirmada
        self.results[res.result_id] = res
        self.tests.setdefault(cid, {})[key] = res.result_id
        st = self.status_of(cid)
        gen = self.counters["generation"]
        # arquivo de falhas interessantes (evita redescobrir becos sem saída)
        if key == "causality" and res.details.get("chaos_suspected"):
            self.failures.add(cid, "high-causality chaos", gen, spread=res.details.get("spread_mean"))
        if key == "replication_memory" and res.status != STATUS_PASS and st.get("memory") == STATUS_PASS:
            self.failures.add(cid, "memory-like but non-reproducible", gen, pass_rate=res.score)
        if key == "channel" and res.status == STATUS_FAIL and st.get("recovery") == STATUS_PASS:
            self.failures.add(cid, "self-recovery without function", gen)
        if key == "generalization" and res.details.get("label") == "OVERFIT":
            self.failures.add(cid, "OVERFIT (falhou em entradas não vistas)", gen)
        if key == "memory" and res.details.get("leakage_detected"):
            self.failures.add(cid, "memory probe: vazamento no controle nulo (inconclusivo)", gen)
        # escada
        old = self.levels.get(cid, -1)
        level, ev = self.ladder.evaluate(self.desc.get(cid), st, st.get("replication_channel") == STATUS_PASS)
        self.levels[cid] = level
        self.evidence[cid] = evidence_score([self.results[r] for r in self.tests[cid].values() if r in self.results])
        self.lineage.update(cid, level=level)
        anomalous = any(e["genome_id"] == cid for e in self.anomalies.entries[-500:])
        if cid in self.desc:
            self.zoo.add(cid, Zoo.classify(self.desc[cid], st, self.cfg.thresholds, anomalous), self.novelty_score.get(cid, 0.0), gen)
            if self.quality.get(cid, -1) >= 0:
                obj = quality_objectives(self.desc[cid], self.cfg.thresholds, self.probe_bonus(cid))
                self.objectives[cid] = obj
                self.quality[cid] = float((obj * QUALITY_WEIGHTS).sum())
                self.map.try_insert(cid, self.desc[cid], self.quality[cid], gen)
        if level > old:
            self.audit.append({"candidate_id": cid, "from": old, "to": level, "evidence": ev, "result_id": res.result_id,
                               "generation": gen, "ts": iso_now(), "config_thresholds_hash": self.cfg.thresholds_hash()})
            if level >= 3 or (level >= 2 and level > self.best_level_global):
                self.register_discovery(cid, old, level, res, ev)
        if level >= self.cfg.thresholds.champion_level - 2:
            self.champions.add(cid, f"level>={level}", gen, level=level)

    # ==================================================================
    # DISCOVERY
    # ==================================================================
    def register_discovery(self, cid: str, old: int, new: int, res: TestResult, ev: List[str]):
        did = new_id("D")
        g = self.store[cid]
        rec = {"discovery_id": did, "candidate_id": cid, "previous_level": old, "new_level": new,
               "level_name": LEVEL_NAMES.get(new), "probe": res.probe, "result_id": res.result_id,
               "status": res.status, "score": res.score, "p_value": res.p_value, "protocol": res.protocol,
               "genome": genome_to_record(self.space, g), "evidence": ev, "generation": self.counters["generation"],
               "ts": iso_now(), "config": self.cfg.to_dict(), "engine": VALIDATION_ENGINE_VERSION,
               "global_record": new > self.best_level_global}
        self.discoveries[did] = rec
        self.counters["discovery_events"] += 1
        self.logger.event("discovery", echo=False, discovery_id=did, candidate=cid, level=new, probe=res.probe)
        if new > self.best_level_global:
            self.best_level_global = new
            self.say(discovery_report_text(rec))
            try:
                rec["bundle"] = write_evidence_bundle(self, rec)
            except Exception as e:
                self.logger.event("bundle_failed", echo=True, error=repr(e), tb=traceback.format_exc()[-1500:])

    # ==================================================================
    # CHAMPION LAB
    # ==================================================================
    def champion_lab(self, cid: str):
        if self.champion_lab_runs >= self.cfg.champion_lab_max_per_session:
            self.logger.event("champion_lab_deferred", echo=True, candidate=cid)
            return
        self.champion_lab_runs += 1
        th, ps = self.cfg.thresholds, self.cfg.probe
        g = self.store[cid]
        self.say(f"[ChampionLab] bateria final em HOLDOUT para {cid} (genoma congelado, zero evolução)")
        seed = derive_seed(self.cfg.master_seed, "champion", cid)
        battery: Dict[str, TestResult] = {}
        live_progress(phase=f"ChampionLab: {cid}", done=0, total=10, detail="bateria final em holdout")
        for i_, probe in enumerate(["causality", "memory", "channel", "transformation", "composition"]):
            live_progress(done=i_, detail=f"holdout: {probe}")
            battery[probe] = self.ve.run_probe(probe, g, derive_seed(seed, probe), split="holdout")
        live_progress(done=5, detail="holdout: generalização, robustez, replicação, ablação, análise avançada")
        battery["generalization"] = self.ve.run_probe("generalization", g, derive_seed(seed, "gen"), split="holdout", task="xor")
        battery["robustness"] = self.ve.run_probe("robustness", g, derive_seed(seed, "rob"), split="holdout")
        battery["replication"] = self.ve.run_probe("replication", g, derive_seed(seed, "rep"), target="channel",
                                                   n_seeds=ps.critical_seeds, split="holdout")
        battery["ablation"] = self.ve.run_probe("ablation", g, derive_seed(seed, "abl"), target="channel", split="holdout")
        battery["advanced"] = self.ve.run_probe("advanced", g, derive_seed(seed, "adv"), split="holdout")
        for k, r in battery.items():
            self.record_result(cid, f"holdout_{k}", r)
        gate = {
            "level": self.levels.get(cid, -1) >= th.champion_level,
            "reproducibility": battery["replication"].score >= th.champion_reproducibility,
            "robustness": battery["robustness"].score >= th.champion_robustness,
            "generalization": battery["generalization"].status == STATUS_PASS,
            "causality": battery["causality"].status == STATUS_PASS,
            "memory": battery["memory"].status == STATUS_PASS,
        }
        # dominância contra o melhor concorrente
        rivals = sorted([c for c in self.levels if c != cid and c in self.store],
                        key=lambda c: (-self.levels[c], -self.evidence.get(c, 0)))
        rival = rivals[0] if rivals else None
        dom = {"rival": rival}
        if rival is not None:
            rr = self.ve.run_probe("replication", self.store[rival], derive_seed(seed, "rival"), target="channel",
                                   n_seeds=ps.critical_seeds, split="holdout")
            a, b = np.asarray(battery["replication"].details["scores"]), np.asarray(rr.details["scores"])
            dom.update({"p": mann_whitney_greater(a, b), "effect_size": cohens_d(a, b), "rival_pass_rate": rr.score})
            dominant = (dom["p"] < th.dominance_p) and (dom["effect_size"] > th.dominance_effect)
        else:
            dominant = True
            dom.update({"p": 0.0, "effect_size": float("inf"), "note": "sem concorrentes avaliados"})
        passed = all(gate.values()) and dominant
        entry = {"candidate_id": cid, "gate": gate, "dominance": to_jsonable(dom), "passed": passed,
                 "battery": {k: r.result_id for k, r in battery.items()}, "ts": iso_now(),
                 "generation": self.counters["generation"]}
        self.champions.add(cid, "champion_lab", self.counters["generation"], **entry)
        self.logger.event("champion_lab", echo=True, candidate=cid, passed=passed, gate=gate)
        if passed:
            if self.champion is not None and self.champion["candidate_id"] != cid:
                self.champion["revoked_by"] = cid
                self.pantheon.append(self.champion)
            self.champion = entry
            self.pantheon.append(entry)
            write_champion_outputs(self, cid, battery, entry)
        else:
            reason = "NO UNIQUE CHAMPION" if all(gate.values()) and not dominant else "GATE NOT PASSED"
            self.say(f"[ChampionLab] {cid}: {reason} — {gate} | dominância: {to_jsonable(dom)}")

    # ==================================================================
    # GERAÇÃO
    # ==================================================================
    def generation_step(self) -> Dict[str, Any]:
        t0 = time.time()
        self.counters["generation"] += 1
        gen = self.counters["generation"]
        live_progress(phase=f"Geração {gen}: mutação/crossover", done=0, total=1,
                      detail=f"{self.cfg.population_size} filhos (operadores escolhidos pelo bandit)")
        kids, meta = self.make_offspring(self.cfg.population_size)
        stats = self.evaluate_and_integrate(kids, meta)
        live_progress(phase=f"Geração {gen}: seleção e Scientist", done=0, total=1, detail="Pareto + novidade + espécies")
        self.select_survivors(self.population + [k.genome_id for k in kids])
        self.species.end_generation(gen, set(self.store[c].species_id for c in self.population))
        self.schedule_experiments([k.genome_id for k in kids])
        frac = self.cfg.experiment_time_fraction
        budget = frac / max(1e-6, 1 - frac) * stats["screen_seconds"]
        stats["experiments"] = self.run_experiments(budget, self.cfg.max_experiments_per_generation)
        self.check_stagnation_and_curriculum()
        self.check_metric_gaming()
        if gen % 5 == 0:
            self.garbage_collect()
        row = self.metrics_row(stats, time.time() - t0)
        self.history.append(row)
        self.history = self.history[-20000:]
        self.logger.metrics(row)
        return row

    def best_candidate(self) -> Optional[str]:
        if not self.levels:
            return None
        return max(self.levels.keys(), key=lambda c: (self.levels[c], self.evidence.get(c, 0.0), self.quality.get(c, -1)))

    def metrics_row(self, stats: Dict[str, Any], gen_seconds: float) -> Dict[str, Any]:
        best = self.best_candidate()
        bq = max([self.quality.get(c, -1) for c in self.population], default=-1)
        pop_d = [self.desc[c] for c in self.population if c in self.desc]
        mean = lambda k: float(np.mean([d[k] for d in pop_d])) if pop_d else 0.0
        return {"generation": self.counters["generation"], "ts": iso_now(), "evaluations": self.counters["evaluations"],
                "sim_steps": self.counters["sim_steps"], "total_runtime": round(self.runtime_total(), 1),
                "gen_seconds": round(gen_seconds, 2), "best_quality": round(bq, 4),
                "map_coverage": round(self.map.coverage(), 4), "species": len(self.species.reps),
                "novelty_archive": len(self.novelty.ids), "anomalies": len(self.anomalies.entries),
                "best_level": self.levels.get(best, -1) if best else -1, "best_candidate": best,
                "viable_fraction": round(stats.get("viable", 0) / max(1, stats.get("n", 1)), 3),
                "new_cells": stats.get("new_cells", 0), "experiments": stats.get("experiments", 0),
                "mean_persistence": round(mean("persistence"), 4), "mean_causal": round(mean("causal_spread"), 4),
                "mean_tc": round(mean("temporal_complexity"), 4), "stagnation_stage": self.stagnation["stage"],
                "batch_size": self.cfg.batch_size, "universe_steps_per_s": round(self.throughput["universe_steps_per_s"], 1)}

    def check_stagnation_and_curriculum(self):
        s = self.stagnation
        bq = max([self.quality.get(c, -1) for c in self.population], default=-1)
        cov = self.map.coverage()
        bl = max(self.levels.values(), default=-1)
        improved = bq > s["best_quality"] + 1e-3 or cov > s["best_coverage"] + 1e-9 or bl > s["best_level"]
        if improved:
            s.update({"best_quality": max(bq, s["best_quality"]), "best_coverage": max(cov, s["best_coverage"]),
                      "best_level": max(bl, s["best_level"]), "since": 0})
            if s["stage"] > 0:
                s["stage"] -= 1
            s["sigma_mult"] = max(1.0, s["sigma_mult"] * 0.9)
            s["novelty_weight"] = max(1.0, s["novelty_weight"] * 0.95)
        else:
            s["since"] += 1
        if s["since"] >= self.cfg.stagnation_window:
            s["since"] = 0
            s["stage"] = min(5, s["stage"] + 1)
            st = s["stage"]
            action = {1: "aumentar mutação", 2: "aumentar pressão de novidade", 3: "criar novas espécies (imigrantes)",
                      4: "reinicializar parte da população", 5: "modificar condições ambientais"}[st]
            if st >= 1:
                s["sigma_mult"] = min(4.0, s["sigma_mult"] * 1.5)
            if st >= 2:
                s["novelty_weight"] = min(3.0, s["novelty_weight"] * 1.3)
            if st >= 4:
                k = len(self.population) // 3
                fresh = [Genome(self.space.random_params(self.rng), generation=self.counters["generation"], origin="restart")
                         for _ in range(k)]
                self.evaluate_and_integrate(fresh, {})
                keep = self.population[: len(self.population) - k]
                self.population = keep + [g.genome_id for g in fresh]
            if st >= 5:
                self.curriculum["perturbation_mult"] = min(4.0, self.curriculum["perturbation_mult"] * 1.2)
                for e in self.envs:
                    if e.split == "evolution":
                        e.ext_noise = float(min(0.02, e.ext_noise * 1.2 + 0.0005))
            self.logger.event("stagnation", echo=True, stage=st, action=action)
        # currículo: capacidades encontradas -> ambientes/probes mais difíceis
        levels = list(self.levels.values())
        if any(l >= 3 for l in levels) and self.curriculum["memory_delay_mult"] < 2.0:
            n3 = sum(1 for l in levels if l >= 3)
            if n3 >= 5:
                self.curriculum["memory_delay_mult"] = min(2.0, self.curriculum["memory_delay_mult"] * 1.25)
                self.logger.event("curriculum", echo=True, change="memory delay ↑", value=self.curriculum["memory_delay_mult"])
        if sum(1 for l in levels if l >= 4) >= 5 and self.curriculum["channel_distance_add"] < self.cfg.world_size // 4:
            self.curriculum["channel_distance_add"] += 2
            self.logger.event("curriculum", echo=True, change="channel distance ↑", value=self.curriculum["channel_distance_add"])

    def check_metric_gaming(self):
        W = self.cfg.thresholds.metric_exploit_window
        if len(self.history) < W + 1:
            return
        a, b = self.history[-W - 1], self.history[-1]
        gain = (b["best_quality"] - a["best_quality"]) / (abs(a["best_quality"]) + 1e-6)
        if gain > self.cfg.thresholds.metric_exploit_gain and b["best_level"] <= a["best_level"] and b["experiments"] >= 0:
            passes_a = sum(1 for h in self.history[-2 * W - 1:-W - 1] for _ in [0] if h.get("best_level", -1) >= 2)
            passes_b = sum(1 for h in self.history[-W:] if h.get("best_level", -1) >= 2)
            if passes_b <= passes_a:
                flag = {"generation": b["generation"], "fitness_gain": gain, "level": b["best_level"]}
                if not self.metric_flags or self.metric_flags[-1]["generation"] < b["generation"] - W:
                    self.metric_flags.append(flag)
                    self.logger.event("METRIC_EXPLOIT_SUSPECTED", echo=True, **flag)

    def garbage_collect(self):
        # elites históricas: metadados são permanentes; genomas completos para as 5000 mais recentes
        keep = set(self.population) | set(self.map.ids()) | set(h["genome_id"] for h in self.map.history[-5000:])
        keep |= set(self.novelty.ids) | set(self.zoo.ids()) | set(self.anomalies.ids()) | set(self.failures.ids())
        keep |= set(self.champions.ids()) | set(d["candidate_id"] for d in self.discoveries.values())
        keep |= set(self.tests.keys())
        keep |= set(e.candidate_id for e in self.scientist.queue)
        for gid in list(self.store.keys()):
            if gid not in keep:
                for dct in (self.store, self.desc, self.vec, self.quality, self.objectives, self.novelty_score, self.levels):
                    dct.pop(gid, None)
        self.lineage.prune(keep, self.counters["generation"])

    # ==================================================================
    # CHECKPOINT
    # ==================================================================
    def state_dict(self) -> dict:
        def rng_state():
            s = {"python": random.getstate(), "numpy_global": np.random.get_state(),
                 "lab_rng": self.rng.bit_generator.state, "torch_cpu": torch.get_rng_state()}
            if torch.cuda.is_available():
                try:
                    s["torch_cuda"] = torch.cuda.get_rng_state_all()
                except Exception as e:
                    s["torch_cuda_error"] = repr(e)
            return s
        self.counters["total_runtime"] = self.runtime_total()
        return {
            "version": GENESIS_VERSION, "checkpoint_version": CHECKPOINT_VERSION, "genesis_version": GENESIS_VERSION,
            "engine_version": VALIDATION_ENGINE_VERSION, "saved_at": iso_now(),
            "config": self.cfg.to_dict(), "hardware": self.hw, "counters": dict(self.counters),
            "population": list(self.population),
            "store": {k: genome_to_record(self.space, g) for k, g in self.store.items()},
            "desc": {k: {kk: (vv if not isinstance(vv, np.ndarray) else vv.tolist()) for kk, vv in v.items()} for k, v in self.desc.items()},
            "vec": self.vec, "quality": self.quality, "objectives": self.objectives, "novelty_score": self.novelty_score,
            "results": {k: r.to_dict() for k, r in self.results.items()}, "tests": self.tests, "levels": self.levels,
            "evidence": self.evidence, "audit": self.audit,
            "map_elites": self.map.state_dict(), "novelty": self.novelty.state_dict(), "zoo": self.zoo.state_dict(),
            "anomalies": self.anomalies.state_dict(), "failures": self.failures.state_dict(),
            "champions": self.champions.state_dict(), "pantheon": self.pantheon, "champion": self.champion,
            "lineage": self.lineage.state_dict(), "species": self.species.state_dict(),
            "surprise": self.surprise.state_dict(), "bandit": self.bandit.state_dict(),
            "scientist": self.scientist.state_dict(), "discoveries": self.discoveries, "baselines": self.baselines,
            "history": self.history[-5000:], "stagnation": self.stagnation, "curriculum": dict(self.curriculum),
            "envs": [e.to_dict() for e in self.envs], "metric_flags": self.metric_flags,
            "best_level_global": self.best_level_global, "rng": rng_state(), "changelog": GENESIS_CHANGELOG,
        }

    def load_state_dict(self, s: dict):
        self.counters = dict(s["counters"])
        self.population = list(s["population"])
        self.store = {k: genome_from_record(self.space, v) for k, v in s["store"].items()}
        self.desc = s["desc"]; self.vec = {k: np.asarray(v, np.float32) for k, v in s["vec"].items()}
        self.quality, self.objectives, self.novelty_score = s["quality"], s["objectives"], s["novelty_score"]
        self.results = {k: TestResult.from_dict(v) for k, v in s["results"].items()}
        self.tests, self.levels, self.evidence, self.audit = s["tests"], s["levels"], s["evidence"], s["audit"]
        self.map.load_state_dict(s["map_elites"]); self.novelty.load_state_dict(s["novelty"]); self.zoo.load_state_dict(s["zoo"])
        self.anomalies.load_state_dict(s["anomalies"]); self.failures.load_state_dict(s["failures"])
        self.champions.load_state_dict(s["champions"]); self.pantheon = list(s["pantheon"]); self.champion = s["champion"]
        self.lineage.load_state_dict(s["lineage"]); self.species.load_state_dict(s["species"])
        self.surprise.load_state_dict(s["surprise"]); self.bandit.load_state_dict(s["bandit"])
        self.scientist.load_state_dict(s["scientist"]); self.discoveries = s["discoveries"]; self.baselines = s["baselines"]
        self.history = list(s["history"]); self.stagnation = s["stagnation"]
        self.curriculum.clear(); self.curriculum.update(s["curriculum"])
        self.envs[:] = [Environment(**e) for e in s["envs"]]
        self.metric_flags = s.get("metric_flags", []); self.best_level_global = s.get("best_level_global", -1)
        self.runtime_at_start = float(self.counters.get("total_runtime", 0.0))
        self.session_start = time.time()
        r = s["rng"]
        random.setstate(r["python"]); np.random.set_state(r["numpy_global"])
        self.rng = np.random.default_rng(); self.rng.bit_generator.state = r["lab_rng"]
        torch.set_rng_state(r["torch_cpu"])
        if torch.cuda.is_available() and "torch_cuda" in r:
            try:
                torch.cuda.set_rng_state_all(r["torch_cuda"])
            except Exception as e:
                self.logger.event("cuda_rng_restore_failed", echo=True, error=repr(e))

    def save_checkpoint(self, reason: str = "periodic") -> str:
        live_progress(phase="Salvando checkpoint", done=0, total=1, detail=f"{reason} → {self.paths.checkpoints}")
        p = self.ckpt.save(self.state_dict())
        live_progress(done=1)
        self.last_checkpoint_ts = time.time()
        self.logger.event("checkpoint", reason=reason, path=p, generation=self.counters["generation"])
        return p

## 12. Relatórios, evidence bundles, saída de campeão e dashboard
Linguagem obrigatoriamente contida: nada é chamado de vida, organismo, inteligência, consciência ou AGI.
Todo relatório lista limitações.

In [ ]:
# @title 12. Relatórios, evidence bundles, saída de campeão e dashboard { display-mode: "form" }
LIMITATIONS = [
    "Universo artificial: nada aqui simula a física real; resultados valem dentro destas regras.",
    "Decodificadores externos (ridge/kNN) medem informação acessível a um observador; não provam que o universo 'usa' essa informação.",
    "Transformação/composição são definidas operacionalmente (XOR/paridade linearmente decodificáveis); outras formas de computação podem passar despercebidas.",
    "Estimativas de MI/TE são plug-in com correção simples de viés; amostras pequenas.",
    "Tempo de GPU não é energia física; nenhum resultado implica viabilidade de hardware.",
    "Adaptação = mudança de comportamento com manutenção de função; não implica aprendizado, vida ou agência.",
    "Um único notebook/Colab limita tamanho de mundo, horizonte temporal e número de sementes.",
]


def discovery_report_text(rec: Dict[str, Any]) -> str:
    ev = "\n".join(f"  - {e}" for e in rec.get("evidence", []))
    return (f"\n╔══════════════════════════════════════╗\n║           DISCOVERY EVENT            ║\n╚══════════════════════════════════════╝\n"
            f"Discovery: {rec['discovery_id']}\nCandidate: {rec['candidate_id']}\n"
            f"Previous level: {rec['previous_level']}\nNew level: {rec['new_level']} ({rec['level_name']})\n"
            f"Probe: {rec['probe']}  status={rec['status']}  score={rec['score']:.3f}  p={rec['p_value']:.2e}\n"
            f"Evidence:\n{ev}\nPrimeira reação: não comemorar — tentar quebrar (replicação, ablação, holdout).\n")


def write_evidence_bundle(lab: "GenesisLab", rec: Dict[str, Any]) -> str:
    d = os.path.join(lab.paths.discoveries, f"{rec['discovery_id']}_L{rec['new_level']}_{rec['candidate_id']}")
    os.makedirs(d, exist_ok=True)
    g = lab.store[rec["candidate_id"]]
    np.savez_compressed(os.path.join(d, "genome.npz"), **{k: np.asarray(v) for k, v in g.params.items()})
    env = lab.envs[0]
    frames, init = simulate_frames(lab, g, env, steps=160, every=2, seed=derive_seed(rec["discovery_id"], "viz"))
    np.savez_compressed(os.path.join(d, "initial_state.npz"), U0=init, env=json.dumps(env.to_dict()),
                        seed=derive_seed(rec["discovery_id"], "viz"))
    res = lab.results.get(rec["result_id"])
    with open(os.path.join(d, "result.json"), "w") as f:
        json.dump(to_jsonable({"discovery": {k: v for k, v in rec.items() if k != "genome"},
                               "result": res.to_dict() if res else None,
                               "scorecard": lab.scorecard(rec["candidate_id"]),
                               "descriptor": {k: v for k, v in lab.desc.get(rec["candidate_id"], {}).items()}}), f, indent=1)
    with open(os.path.join(d, "protocol.json"), "w") as f:
        json.dump(to_jsonable(rec["protocol"]), f, indent=1)
    plot_frames(frames, f"{rec['candidate_id']} — L{rec['new_level']}", os.path.join(d, "frames.png"), show=False)
    if res is not None:
        if res.probe == "memory":
            plot_memory_curve(res, os.path.join(d, "memory_curve.png"), show=False)
        if res.probe == "causality":
            plot_causal_map(res, os.path.join(d, "causal_map.png"), show=False)
        if res.probe == "recovery":
            plot_damage_curve(res, os.path.join(d, "damage_curve.png"), show=False)
    if rec["new_level"] >= 2:
        save_animation(frames, os.path.join(d, "animation.gif"))
    with open(os.path.join(d, "REPORT.md"), "w") as f:
        f.write(f"# DISCOVERY {rec['discovery_id']}\n\n```\n{discovery_report_text(rec)}\n```\n\n## Scorecard\n\n")
        for k, v in lab.scorecard(rec["candidate_id"]).items():
            f.write(f"- **{k}**: {v}\n")
        f.write("\n## Linhagem\n\n```\n" + lab.lineage.tree(rec["candidate_id"], depth=2) + "\n```\n\n## Limitações\n\n")
        f.writelines(f"- {l}\n" for l in LIMITATIONS)
    return d


def write_champion_outputs(lab: "GenesisLab", cid: str, battery: Dict[str, TestResult], entry: Dict[str, Any]):
    d = lab.paths.champion
    g = lab.store[cid]
    lab.say("╔══════════════════════════════════════╗\n║       GENESIS CHAMPION DETECTED      ║\n╚══════════════════════════════════════╝")
    torch.save({"genome": genome_to_record(lab.space, g), "config": lab.cfg.to_dict(), "entry": entry,
                "version": GENESIS_VERSION}, os.path.join(d, "GENESIS_CHAMPION.pt"))
    with open(os.path.join(d, "GENESIS_CHAMPION_EXPERIMENTS.json"), "w") as f:
        json.dump(to_jsonable({k: r.to_dict() for k, r in battery.items()} |
                              {"all_results": {k: lab.results[r].to_dict() for k, r in lab.tests.get(cid, {}).items() if r in lab.results}}), f, indent=1)
    anc = [cid] + lab.lineage.ancestors(cid, 500)
    with open(os.path.join(d, "GENESIS_CHAMPION_LINEAGE.json"), "w") as f:
        json.dump(to_jsonable({a: lab.lineage.records.get(a) for a in anc}), f, indent=1)
    with open(os.path.join(d, "GENESIS_CHAMPION_METRICS.csv"), "w", newline="") as f:
        w = csv.writer(f); w.writerow(["field", "value"])
        for k, v in lab.scorecard(cid).items():
            w.writerow([k, v])
        for k, v in lab.desc.get(cid, {}).items():
            if not isinstance(v, list):
                w.writerow([f"descriptor.{k}", v])
    runner = entry["dominance"].get("rival")
    with open(os.path.join(d, "GENESIS_CHAMPION_REPORT.md"), "w") as f:
        f.write(f"# GENESIS CHAMPION — {cid}\n\nGerado em {iso_now()} · Genesis {GENESIS_VERSION} · {VALIDATION_ENGINE_VERSION}\n\n")
        f.write("## Gate\n\n" + "\n".join(f"- {k}: {'PASS' if v else 'FAIL'}" for k, v in entry["gate"].items()))
        f.write(f"\n\n## Dominância\n\n```\n{json.dumps(to_jsonable(entry['dominance']), indent=1)}\n```\n")
        f.write(f"\nRunner-up preservado: {runner}\n\n## Scorecard\n\n")
        f.writelines(f"- **{k}**: {v}\n" for k, v in lab.scorecard(cid).items())
        f.write("\n## Genoma\n\n```\n" + g.summary(lab.space) + "\n```\n\n## Limitações obrigatórias\n\n")
        f.writelines(f"- {l}\n" for l in LIMITATIONS)
    lab.logger.event("GENESIS_CHAMPION", echo=True, candidate=cid, runner_up=runner)


def dashboard_text(lab: "GenesisLab", status: str = "SEARCHING") -> str:
    best = lab.best_candidate()
    sc = lab.scorecard(best) if best else {}
    gpu = lab.hw.get("gpu_name", "CPU")
    if torch.cuda.is_available():
        used = torch.cuda.memory_allocated() / 1e9; tot = lab.hw.get("vram_total", 0) / 1e9
        vram = f"{used:.2f} / {tot:.1f} GB"
    else:
        vram = "n/a"
    nxt = max(0.0, lab.cfg.checkpoint_every_minutes * 60 - (time.time() - lab.last_checkpoint_ts))
    last = lab.ckpt.last_save_ts
    val = lambda k: sc.get(k, STATUS_NOT_TESTED)
    lines = [
        "╔══════════════════════════════════════╗",
        "║          PROJECT GENESIS Ω           ║",
        "║ UNIVERSAL DISCOVERY ENGINE — ONLINE  ║",
        "╚══════════════════════════════════════╝",
        f"Session Runtime:      {fmt_hms(time.time() - lab.session_start)}   (sessão #{lab.counters['sessions']})",
        f"Total Runtime:        {fmt_hms(lab.runtime_total())}",
        f"Generation:           {lab.counters['generation']}",
        f"Universes Evaluated:  {lab.counters['evaluations']:,}",
        f"Simulation Steps:     {lab.counters['sim_steps']:,} universo·passos   (~{lab.throughput['universe_steps_per_s']:,.0f}/s, ~{lab.throughput['evals_per_hour_est']:,.0f} avaliações/h — estimativa)",
        f"GPU:                  {gpu}",
        f"VRAM:                 {vram}",
        f"Population:           {len(lab.population)}   batch={lab.cfg.batch_size}  world={lab.cfg.world_size}²×{lab.cfg.channels_max}",
        f"Species:              {len(lab.species.reps)} vivas / {len(lab.species.extinct)} extintas",
        f"Novelty Zoo:          {len(lab.novelty.ids)} | zoo: " + ", ".join(f"{k[:5]}={v}" for k, v in lab.zoo.counts().items() if v),
        f"MAP-Elites Coverage:  {lab.map.coverage():.1%} ({len(lab.map.cells)} células)",
        f"Best Discovery Level: {sc.get('level', -1)} ({LEVEL_NAMES.get(sc.get('level', -1), '-')})",
        f"Current Best Candidate: {best}",
        f"Best Fitness:         {max([lab.quality.get(c, -1) for c in lab.population], default=-1):.4f}   (fitness ≠ evidência; evidência={sc.get('evidence', 0)})",
        f"Persistence:          {val('persistence')}",
        f"Complexity:           {val('complexity')}",
        f"Causality:            {val('causality')}",
        f"Memory:               {val('memory')}",
        f"Channel:              {val('channel')}",
        f"Transformation:       {val('transformation')}",
        f"Composition:          {val('composition')}",
        f"Generalization:       {val('generalization')}",
        f"Recovery:             {val('recovery')}",
        f"Robustness:           {val('robustness')}",
        f"Novel Discoveries:    {lab.counters['discovery_events']}   Experiments: {lab.counters['experiments']}  (fila: {len(lab.scientist.queue)})",
        f"Anomalies:            {len(lab.anomalies.entries)}   Failures archived: {len(lab.failures.entries)}",
        f"Provisional Champions: {len(lab.champions.entries)}   Pantheon: {len(lab.pantheon)}",
        f"Checkpoint:           {lab.paths.checkpoints}  ({'Drive' if lab.paths.on_drive else 'local'})",
        f"Last Checkpoint:      {datetime.datetime.fromtimestamp(last).strftime('%H:%M:%S') if last else '-'}   Next: ~{fmt_hms(nxt)}",
        f"Stagnation stage:     {lab.stagnation['stage']}  sigma×{lab.stagnation['sigma_mult']:.2f}  novelty×{lab.stagnation['novelty_weight']:.2f}",
        f"STATUS:               {status}",
    ]
    return "\n".join(lines)


def show_dashboard(lab: "GenesisLab", status: str = "SEARCHING", plots: bool = False):
    if IN_NOTEBOOK:
        clear_output(wait=True)
    print(dashboard_text(lab, status))
    if plots and IN_NOTEBOOK:
        plot_map_elites(lab.map, None, True)
        plot_convergence(lab.history, None, True)


def session_report(lab: "GenesisLab", reason: str) -> str:
    path = os.path.join(lab.paths.reports, f"session_{lab.counters['sessions']:04d}_{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}.md")
    best = lab.best_candidate()
    lvl_counts: Dict[int, int] = {}
    for l in lab.levels.values():
        lvl_counts[l] = lvl_counts.get(l, 0) + 1
    with open(path, "w") as f:
        f.write(f"# Genesis Ω — relatório de sessão #{lab.counters['sessions']}\n\nMotivo do encerramento: **{reason}**\n\n")
        f.write("```\n" + dashboard_text(lab, "SESSION END") + "\n```\n\n")
        f.write("## Distribuição de DiscoveryLevel (candidatos retidos)\n\n")
        f.writelines(f"- L{k} {LEVEL_NAMES.get(k, '')}: {v}\n" for k, v in sorted(lvl_counts.items()))
        f.write("\n## Melhor candidato\n\n")
        if best:
            f.writelines(f"- **{k}**: {v}\n" for k, v in lab.scorecard(best).items())
            f.write("\n```\n" + lab.lineage.tree(best, depth=2) + "\n```\n")
        f.write("\n## Promoções (auditoria)\n\n")
        f.writelines(f"- gen {a['generation']}: {a['candidate_id']} L{a['from']}→L{a['to']} (result {a['result_id']})\n" for a in lab.audit[-50:])
        f.write("\n## Failure archive (últimos)\n\n")
        f.writelines(f"- {e['genome_id']}: {e['reason']}\n" for e in lab.failures.entries[-20:])
        f.write("\n## Bandit de operadores\n\n" + json.dumps(lab.bandit.probs()) + "\n")
        f.write("\n## Sinais de METRIC EXPLOIT\n\n" + (json.dumps(to_jsonable(lab.metric_flags[-10:])) or "nenhum") + "\n")
        f.write("\n## Limitações\n\n")
        f.writelines(f"- {l}\n" for l in LIMITATIONS)
        if lab.champion is None:
            f.write("\n**Nenhum GENESIS CHAMPION nesta sessão. Isso não é erro: a pesquisa continua na próxima sessão.**\n")
    return path

## 12b. Painel AO VIVO — atualiza a cada segundo, sem parar a pesquisa
Enquanto a pesquisa roda, a saída da célula de pesquisa vira um painel que se atualiza sozinho:
* **agora no simulador**: mosaico de universos do lote que está sendo simulado neste instante (PCA dos canais → cor);
* **fase atual** com barra de progresso (triagem, arquivos, experimento do Scientist, ChampionLab, checkpoint…);
* contadores, velocidade, **melhor candidato** com scorecard e nível na escada, anúncios de descoberta, eventos recentes;
* **gráficos por geração**: convergência, MAP-Elites, escada de descoberta, zoológico, operadores de mutação.

Não é preciso interromper nada para ver resultados. O painel é só observação: lê o estado, nunca participa do
fitness, da seleção ou dos testes, e não altera nenhum gerador aleatório da pesquisa.
Fora de notebook (script), o painel vira uma linha de status periódica.

In [ ]:
# @title 12b. Painel AO VIVO — atualiza a cada segundo, sem parar a pesquisa { display-mode: "form" }
LIVE_STATUS_COLORS = {STATUS_PASS: "#3fb950", STATUS_FAIL: "#f85149", STATUS_INCONCLUSIVE: "#d29922",
                      STATUS_ERROR: "#db6d28", STATUS_NOT_TESTED: "#6e7681", STATUS_RUNNING: "#58a6ff"}
LEVEL_COLORS = ["#6e7681", "#8b949e", "#58a6ff", "#a371f7", "#db61a2", "#f0883e", "#d29922", "#3fb950", "#56d364",
                "#ffd33d", "#ffffff"]
PT_FIELD = {"activity": "atividade", "persistence": "persistência", "mobility": "mobilidade", "complexity": "complexidade",
            "causality": "causalidade", "memory": "memória", "channel": "canal", "transformation": "transformação",
            "composition": "composição", "generalization": "generalização", "recovery": "recuperação",
            "robustness": "robustez", "adaptation": "adaptação", "replication": "replicação", "novelty": "novidade"}
_ROOT = ("background:#0d1117;color:#e6edf3;border:1px solid #30363d;border-radius:12px;padding:12px 14px;margin:4px 0;"
         "font-family:ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;font-size:12.5px;line-height:1.45;")
_MUTED = "color:#8b949e;"


def _esc(x) -> str:
    return _html.escape(str(x))


def _short(gid: Optional[str]) -> str:
    return "-" if not gid else str(gid)


def _human(n: float) -> str:
    n = float(n)
    for unit, div in (("G", 1e9), ("M", 1e6), ("k", 1e3)):
        if abs(n) >= div:
            return f"{n / div:.2f}{unit}"
    return f"{n:.0f}"


def rgb_tile(U: np.ndarray) -> np.ndarray:
    """[C,S,S] -> RGB uint8 por PCA dos canais (sinal fixado para as cores não piscarem entre quadros)."""
    C, S, _ = U.shape
    X = np.nan_to_num(U.reshape(C, -1).T.astype(np.float64))
    X = X - X.mean(0, keepdims=True)
    if np.abs(X).max() < 1e-9:
        return np.zeros((S, S, 3), np.uint8)
    k = min(3, C)
    try:
        _, _, Vt = np.linalg.svd(X, full_matrices=False)
        V = Vt[:k]
        V = V * np.sign(V[np.arange(k), np.abs(V).argmax(1)])[:, None]
        Y = X @ V.T
    except np.linalg.LinAlgError:
        Y = X[:, :k]
    if Y.shape[1] < 3:
        Y = np.hstack([Y, np.zeros((Y.shape[0], 3 - Y.shape[1]))])
    lo, hi = np.percentile(Y, 1), np.percentile(Y, 99)
    Y = np.clip((Y - lo) / (hi - lo + 1e-9), 0, 1)
    return (Y.reshape(S, S, 3) * 255).astype(np.uint8)


def mosaic_jpeg(frames: List[np.ndarray], cols: int = 4, tile: int = 112, quality: int = 78) -> bytes:
    from PIL import Image
    n = max(1, len(frames))
    cols = max(1, min(cols, n))
    rows = int(math.ceil(n / cols))
    gap = 3
    canvas = Image.new("RGB", (cols * tile + (cols - 1) * gap, rows * tile + (rows - 1) * gap), (13, 17, 23))
    for i, f in enumerate(frames):
        im = Image.fromarray(rgb_tile(f)).resize((tile, tile), Image.NEAREST)
        canvas.paste(im, ((i % cols) * (tile + gap), (i // cols) * (tile + gap)))
    buf = io.BytesIO()
    canvas.save(buf, format="JPEG", quality=quality)
    return buf.getvalue()


def svg_spark(vals: Sequence[float], label: str, color: str = "#58a6ff", fmt: str = "{:.3f}", w: int = 200,
              h: int = 44) -> str:
    v = [float(x) for x in vals if x is not None and np.isfinite(x)][-300:]
    head = f'<div style="font-size:11px;{_MUTED}">{_esc(label)} <b style="color:#e6edf3">{fmt.format(v[-1]) if v else "–"}</b></div>'
    if len(v) < 2:
        return (f'<div style="margin:4px 10px 4px 0">{head}<div style="height:{h}px;width:{w}px;background:#161b22;'
                f'border-radius:4px;color:#6e7681;font-size:10.5px;padding:4px">aguardando dados…</div></div>')
    lo, hi = min(v), max(v)
    if hi - lo < 1e-12:
        lo, hi = lo - 0.5, hi + 0.5
    n = len(v)
    pts = " ".join(f"{1 + i * (w - 2) / (n - 1):.1f},{h - 3 - (x - lo) / (hi - lo) * (h - 6):.1f}" for i, x in enumerate(v))
    return (f'<div style="margin:4px 10px 4px 0">{head}<svg width="{w}" height="{h}" style="background:#161b22;border-radius:4px">'
            f'<polyline points="{pts}" fill="none" stroke="{color}" stroke-width="1.6"/></svg></div>')


def svg_map(me: MapElites, cell: int = 15) -> str:
    import matplotlib.colors as mcolors
    H = me.heat()
    nx, ny = H.shape
    vals = H[np.isfinite(H)]
    lo, hi = (float(vals.min()), float(vals.max())) if vals.size else (0.0, 1.0)
    rects = []
    for i in range(nx):
        for j in range(ny):
            v = H[i, j]
            col = "#161b22" if not np.isfinite(v) else mcolors.to_hex(plt.cm.viridis((v - lo) / (hi - lo + 1e-9)))
            rects.append(f'<rect x="{i * cell}" y="{(ny - 1 - j) * cell}" width="{cell - 1}" height="{cell - 1}" fill="{col}"/>')
    return (f'<div><div style="font-size:11px;{_MUTED}">MAP-Elites — {_esc(me.axes[0])} → / {_esc(me.axes[1])} ↑ · '
            f'cobertura <b style="color:#e6edf3">{me.coverage():.1%}</b></div>'
            f'<svg width="{nx * cell}" height="{ny * cell}" style="background:#0d1117">{"".join(rects)}</svg></div>')


def _chip(text: str, color: str) -> str:
    return (f'<span style="display:inline-block;margin:2px 3px 2px 0;padding:1px 7px;border-radius:10px;font-size:11px;'
            f'border:1px solid {color};color:{color}">{text}</span>')


def _level_badge(level: int) -> str:
    lv = max(-1, min(10, int(level)))
    name = LEVEL_NAMES.get(lv, "inviável") if lv >= 0 else "inviável"
    col = LEVEL_COLORS[lv] if lv >= 0 else "#f85149"
    return (f'<span style="padding:1px 8px;border-radius:10px;background:{col};color:#0d1117;font-weight:700;'
            f'font-size:11px">L{lv} {_esc(name)}</span>')


class LiveMonitor:
    """Painel que se redesenha a cada `interval` segundos, alimentado pelos próprios passos da simulação."""

    def __init__(self, lab: "GenesisLab", interval: float = 1.0, refresh_every: float = 90.0, text_every: float = 15.0):
        self.lab = lab
        self.interval = max(0.25, float(interval))
        self.refresh_every = refresh_every
        self.text_every = text_every
        self.mode = "html" if IN_NOTEBOOK else "text"
        self.status = "SEARCHING"
        self.phase, self.detail, self.done, self.total = "iniciando", "", 0.0, 0.0
        self.phase_t0 = time.time()
        self.phase_steps0 = lab.sim.steps_done
        self.last = 0.0
        self.last_text = 0.0
        self.last_refresh = 0.0
        self.frames: Optional[np.ndarray] = None
        self.frames_info = ""
        self.frames_jpeg_b64: Optional[str] = None
        self.frames_dirty = False
        self.best_cache: Tuple[Optional[str], Optional[str], float] = (None, None, 0.0)
        self.view_ts = 0.0
        self.charts_key: Any = None
        self.charts_ts = 0.0
        self.speed = 0.0
        self.speed_hist: "collections.deque" = collections.deque(maxlen=240)
        self._steps_prev, self._steps_t = lab.sim.steps_done, time.time()
        self.banners: List[Tuple[float, str]] = []
        self.handles: Dict[str, Any] = {}
        self.viz_sim: Optional[Simulator] = None
        self.errors = 0
        self._busy = False

    # ------------------------------------------------------------------ entradas
    def start(self):
        self.render(force=True)

    def on_step(self, sim: "Simulator", gb: "GenomeBatch", st: "SimState"):
        if self._busy or sim is self.viz_sim:
            return
        now = time.time()
        if now - self.last < self.interval:
            return
        self._capture(st)
        self.render()

    def progress(self, phase=None, done=None, total=None, detail=None):
        if phase is not None and phase != self.phase:
            self.phase, self.phase_t0 = phase, time.time()
            self.phase_steps0 = self.lab.sim.steps_done
            self.done, self.total, self.detail = 0.0, 0.0, ""
        if detail is not None:
            self.detail = detail
        if total is not None:
            self.total = float(total)
        if done is not None:
            self.done = float(done)
        if not self._busy and time.time() - self.last >= self.interval:
            self.render()

    def announce(self, text: str):
        self.banners.append((time.time(), text))
        self.banners = self.banners[-4:]

    def finish(self, status: str):
        self.status = status
        self.phase, self.detail, self.done, self.total = status, "", 0.0, 0.0
        if self.mode == "html":
            self._busy = True
            try:
                self._open_handles(time.time())      # painel final sempre visível (saídas novas)
            except Exception as e:
                print(f"[painel] não foi possível desenhar o estado final: {e!r}")
            finally:
                self._busy = False
        else:
            self.render(force=True)

    # ------------------------------------------------------------------ captura
    def _capture(self, st: "SimState"):
        try:
            B = int(st.U.shape[0])
            pool = B // 2 if B > 16 else B
            k = min(8, pool)
            idx = np.unique(np.linspace(0, pool - 1, k).astype(int))
            self.frames = st.U[torch.as_tensor(idx, device=st.U.device)].float().cpu().numpy()
            self.frames_info = f"{len(idx)} de {B} universos do lote atual · passo interno t={st.t}"
            self.frames_dirty = True
        except Exception as e:
            self.frames_info = f"captura indisponível ({type(e).__name__})"

    # ------------------------------------------------------------------ desenho
    def render(self, force: bool = False):
        if self._busy:
            return
        self._busy = True
        try:
            now = time.time()
            self.last = now
            self._update_speed(now)
            if self.mode == "text":
                if force or now - self.last_text >= self.text_every:
                    print(self._text_line(), flush=True)
                    self.last_text = now
                return
            if not self.handles or now - self.last_refresh >= self.refresh_every:
                self._open_handles(now)
                return
            self.handles["head"].update(HTML(self._head_html(now)))
            best = self.lab.best_candidate()
            if force or self.frames_dirty or best != self.best_cache[0] or now - self.view_ts > 5:
                self.handles["view"].update(HTML(self._view_html(now, best)))
                self.view_ts = now
            key = (self.lab.counters["generation"], len(self.lab.results), self.lab.counters["discovery_events"],
                   len(self.lab.map.cells))
            if force or key != self.charts_key or now - self.charts_ts > 20:
                self.handles["charts"].update(HTML(self._charts_html()))
                self.charts_key, self.charts_ts = key, now
        except KeyboardInterrupt:
            raise
        except Exception as e:
            self.errors += 1
            if self.errors <= 3:
                self.lab.logger.event("live_render_error", error=repr(e), tb=traceback.format_exc()[-1200:])
            if self.errors >= 25:
                self.mode = "text"     # degrada para linha de status, sem nunca derrubar a pesquisa
        finally:
            self._busy = False

    def _open_handles(self, now: float):
        """(Re)cria as saídas. Periodicamente refeito: se o navegador recarregar a página, o painel volta sozinho."""
        if self.handles:
            clear_output(wait=True)
        best = self.lab.best_candidate()
        self.handles = {"head": display(HTML(self._head_html(now)), display_id=True),
                        "view": display(HTML(self._view_html(now, best)), display_id=True),
                        "charts": display(HTML(self._charts_html()), display_id=True)}
        self.last_refresh = self.view_ts = self.charts_ts = now

    def _update_speed(self, now: float):
        s = self.lab.sim.steps_done
        dt = now - self._steps_t
        if dt >= 0.5:
            self.speed = (s - self._steps_prev) / dt
            self._steps_prev, self._steps_t = s, now
            self.speed_hist.append(self.speed)

    def _text_line(self) -> str:
        lab = self.lab
        c = lab.counters
        best = lab.best_candidate()
        pct = f" {100 * self.done / self.total:.0f}%" if self.total else ""
        return (f"[{fmt_hms(time.time() - lab.session_start)}] gen {c['generation']} | {c['evaluations']:,} universos | "
                f"{self.speed:,.0f} u·passos/s | {self.phase}{pct} | melhor L{lab.levels.get(best, -1) if best else -1} {best} | "
                f"exp {c['experiments']} | descobertas {c['discovery_events']}")

    def _tile(self, label: str, value: str, sub: str = "") -> str:
        return (f'<div style="background:#161b22;border:1px solid #21262d;border-radius:8px;padding:6px 9px">'
                f'<div style="{_MUTED}font-size:10px;text-transform:uppercase;letter-spacing:.05em">{_esc(label)}</div>'
                f'<div style="font-size:16px;font-weight:600">{value}</div>'
                f'<div style="{_MUTED}font-size:10.5px">{sub}</div></div>')

    def _head_html(self, now: float) -> str:
        lab = self.lab
        c = lab.counters
        best = lab.best_candidate()
        best_lv = lab.levels.get(best, -1) if best else -1
        last_gen = lab.history[-1] if lab.history else {}
        st_col = "#3fb950" if self.status == "SEARCHING" else "#d29922"
        last_ck = lab.ckpt.last_save_ts
        nxt = max(0.0, lab.cfg.checkpoint_every_minutes * 60 - (now - lab.last_checkpoint_ts))
        avg_speed = float(np.mean(self.speed_hist)) if self.speed_hist else 0.0
        tiles = "".join([
            self._tile("Geração", f"{c['generation']:,}", f"{last_gen.get('gen_seconds', 0):.0f} s na última"),
            self._tile("Universos avaliados", f"{c['evaluations']:,}", f"{lab.throughput['evals_per_hour_est']:,.0f}/h (estim.)"),
            self._tile("Passos simulados", _human(c["sim_steps"]), "universo·passos"),
            self._tile("Velocidade", f"{_human(self.speed)}/s", f"média {_human(avg_speed)}/s"),
            self._tile("Melhor nível", _level_badge(best_lv), f"recorde L{lab.best_level_global}"),
            self._tile("Descobertas", f"{c['discovery_events']:,}", f"experimentos {c['experiments']:,}"),
            self._tile("Fila do Scientist", f"{len(lab.scientist.queue):,}", "experimentos pendentes"),
            self._tile("Espécies", f"{len(lab.species.reps):,}", f"{len(lab.species.extinct):,} extintas"),
            self._tile("MAP-Elites", f"{lab.map.coverage():.1%}", f"{len(lab.map.cells)} células"),
            self._tile("Anomalias", f"{len(lab.anomalies.entries):,}", f"falhas arquivadas {len(lab.failures.entries):,}"),
            self._tile("Novelty zoo", f"{len(lab.novelty.ids):,}", f"campeões provisórios {len(lab.champions.entries)}"),
            self._tile("Checkpoint", datetime.datetime.fromtimestamp(last_ck).strftime("%H:%M:%S") if last_ck else "-",
                       f"próximo em {fmt_hms(nxt)}"),
        ])
        if self.status != "SEARCHING":
            bar, prog = "", "pesquisa parada — estado salvo"
        elif self.total > 0:
            pct = max(0.0, min(100.0, 100.0 * self.done / self.total))
            bar = (f'<div style="background:#21262d;border-radius:6px;height:10px;margin:4px 0">'
                   f'<div style="width:{pct:.1f}%;background:#58a6ff;height:10px;border-radius:6px"></div></div>')
            prog = f"{pct:.0f}%"
        else:
            bar = ('<div style="background:#21262d;border-radius:6px;height:10px;margin:4px 0">'
                   '<div style="width:100%;background:#1f6feb55;height:10px;border-radius:6px"></div></div>')
            prog = "em andamento"
        phase_steps = self.lab.sim.steps_done - self.phase_steps0
        events = self._events_html()
        where = f"{_esc(lab.paths.checkpoints)} ({'Google Drive' if lab.paths.on_drive else 'disco local do Colab'})"
        return (f'<div style="{_ROOT}">'
                f'<div style="display:flex;justify-content:space-between;flex-wrap:wrap;gap:6px;align-items:baseline">'
                f'<div style="font-size:15px;font-weight:700"><span style="color:{st_col}">●</span> PROJECT GENESIS Ω '
                f'<span style="color:{st_col}">AO VIVO</span> <span style="{_MUTED}font-weight:400;font-size:11px">'
                f'atualiza a cada {self.interval:g} s · {datetime.datetime.now().strftime("%H:%M:%S")}</span></div>'
                f'<div style="{_MUTED}">sessão {fmt_hms(now - lab.session_start)} · tempo total {fmt_hms(lab.runtime_total())} · '
                f'sessão nº {c["sessions"]} · <b style="color:{st_col}">{_esc(self.status)}</b></div></div>'
                f'<div style="display:grid;grid-template-columns:repeat(auto-fill,minmax(138px,1fr));gap:6px;margin:10px 0">{tiles}</div>'
                f'<div>▶ <b>{_esc(self.phase)}</b> <span style="{_MUTED}">{_esc(self.detail)}</span></div>{bar}'
                f'<div style="{_MUTED}font-size:11px">{prog} · {fmt_hms(now - self.phase_t0)} nesta fase · '
                f'{_human(phase_steps)} universo·passos nesta fase</div>'
                f'<div style="margin-top:8px">{events}</div>'
                f'<div style="{_MUTED}font-size:11px;margin-top:8px;border-top:1px solid #21262d;padding-top:6px">'
                f'💾 {where} · ⏸ pausar: botão ■ da célula (salva tudo) · ▶ continuar: rode esta célula de novo '
                f'(ou Executar tudo) — a pesquisa retoma do último checkpoint</div></div>')

    def _events_html(self) -> str:
        rows = []
        for e in reversed(self.lab.logger.recent[-80:]):
            line = self._fmt_event(e)
            if line:
                rows.append(f'<div style="font-size:11.5px;white-space:nowrap;overflow:hidden;text-overflow:ellipsis">'
                            f'<span style="{_MUTED}">{_esc(e.get("ts", "")[11:19])}</span> {line}</div>')
            if len(rows) >= 9:
                break
        if not rows:
            rows = [f'<div style="{_MUTED}font-size:11.5px">aguardando os primeiros eventos…</div>']
        return f'<div style="{_MUTED}font-size:11px;margin-bottom:2px">EVENTOS RECENTES</div>' + "".join(rows)

    @staticmethod
    def _fmt_event(e: Dict[str, Any]) -> str:
        k = e.get("kind")
        if k == "experiment":
            col = LIVE_STATUS_COLORS.get(e.get("status"), "#e6edf3")
            sc = e.get("score")
            sc_s = f" {sc:.2f}" if isinstance(sc, (int, float)) and sc is not None and np.isfinite(sc) else ""
            return (f'🧪 {_esc(e.get("probe"))} → <b style="color:{col}">{_esc(e.get("status"))}{sc_s}</b> em '
                    f'{_esc(e.get("candidate"))} <span style="{_MUTED}">({e.get("seconds")} s)</span>')
        if k == "discovery":
            return (f'⭐ <b style="color:#ffd33d">descoberta L{e.get("level")} {_esc(LEVEL_NAMES.get(e.get("level"), ""))}</b> '
                    f'via {_esc(e.get("probe"))} em {_esc(e.get("candidate"))}')
        if k == "anomaly":
            return f'⚠️ anomalia {_esc(e.get("genome_id"))} (surpresa {e.get("surprise", 0):.1f})'
        if k == "checkpoint":
            return f'💾 checkpoint salvo (geração {e.get("generation")}, {_esc(e.get("reason"))})'
        if k == "stagnation":
            return f'🌀 estagnação, estágio {e.get("stage")}: {_esc(e.get("action"))}'
        if k == "curriculum":
            return f'📈 currículo: {_esc(e.get("change"))} → {e.get("value")}'
        if k == "champion_lab":
            return f'🏆 ChampionLab {_esc(e.get("candidate"))}: {"PASSOU" if e.get("passed") else "não passou"}'
        if k == "GENESIS_CHAMPION":
            return f'🏆 <b style="color:#ffd33d">GENESIS CHAMPION {_esc(e.get("candidate"))}</b>'
        if k == "METRIC_EXPLOIT_SUSPECTED":
            return f'🚩 suspeita de exploit de métrica (geração {e.get("generation")})'
        if k == "OOM":
            return f'🧯 falta de memória na GPU: batch {e.get("old_batch")} → {e.get("new_batch")} (recuperado)'
        if k in ("resume", "research_start"):
            return f'▶ pesquisa iniciada/retomada na geração {e.get("generation")}'
        if k == "crash":
            return f'💥 erro: {_esc(e.get("error"))}'
        return ""

    def _render_best(self, cid: str) -> Optional[str]:
        g = self.lab.store.get(cid)
        if g is None:
            return None
        if self.viz_sim is None:
            self.viz_sim = Simulator(self.lab.cfg, self.lab.device)
        gb = self.viz_sim.prepare([g])
        st = self.viz_sim.init_state(gb, [self.lab.envs[0]], [derive_seed(cid, "thumb")], run_seed=1)
        frames = []
        for t in range(96):
            st = self.viz_sim.step(gb, st)
            if t in (31, 63, 95):
                frames.append(st.U[0].float().cpu().numpy())
        return base64.b64encode(mosaic_jpeg(frames, cols=3, tile=96)).decode("ascii")

    def _view_html(self, now: float, best: Optional[str]) -> str:
        lab = self.lab
        if self.frames is not None and (self.frames_dirty or self.frames_jpeg_b64 is None):
            self.frames_jpeg_b64 = base64.b64encode(mosaic_jpeg(list(self.frames))).decode("ascii")
            self.frames_dirty = False
        live = (f'<img src="data:image/jpeg;base64,{self.frames_jpeg_b64}" style="width:100%;max-width:470px;'
                f'image-rendering:pixelated;border-radius:6px"/>' if self.frames_jpeg_b64 else
                f'<div style="{_MUTED}height:120px">aguardando o primeiro lote…</div>')
        if best and best != self.best_cache[0] and (self.best_cache[0] is None or now - self.best_cache[2] > 20):
            try:
                self.best_cache = (best, self._render_best(best), now)
            except Exception as e:
                self.best_cache = (best, None, now)
                lab.logger.event("live_thumb_error", error=repr(e))
        best_html = f'<div style="{_MUTED}">ainda sem candidatos avaliados</div>'
        if best:
            sc = lab.scorecard(best)
            g = lab.store.get(best)
            chips = []
            for f in ["persistence", "complexity", "causality", "memory", "channel", "transformation", "composition",
                      "generalization", "recovery", "robustness", "adaptation", "replication"]:
                v = sc.get(f, STATUS_NOT_TESTED)
                if isinstance(v, (int, float)):
                    chips.append(_chip(f"{PT_FIELD.get(f, f)} {v:.2f}", "#58a6ff"))
                else:
                    stt, _, num = str(v).partition(":")
                    mark = {"PASS": "✓", "FAIL": "✗", "INCONCLUSIVE": "?", "ERROR": "!", "NOT_TESTED": "·"}.get(stt, "")
                    chips.append(_chip(f"{PT_FIELD.get(f, f)} {mark}{(' ' + num[:4]) if num else ''}",
                                       LIVE_STATUS_COLORS.get(stt, "#6e7681")))
            thumb = (f'<img src="data:image/jpeg;base64,{self.best_cache[1]}" style="width:100%;max-width:300px;'
                     f'image-rendering:pixelated;border-radius:6px;margin-top:6px"/>'
                     f'<div style="{_MUTED}font-size:10.5px">o melhor universo em t = 32, 64, 96</div>'
                     if self.best_cache[0] == best and self.best_cache[1] else "")
            best_html = (f'<div style="font-size:13px"><b>{_esc(best)}</b> {_level_badge(sc.get("level", -1))}</div>'
                         f'<div style="{_MUTED}font-size:11px">fitness {sc.get("quality(fitness)", 0):.3f} · evidência '
                         f'{sc.get("evidence", 0):.2f} · espécie {g.species_id if g else "-"} · geração {g.generation if g else "-"} · '
                         f'{_esc(g.operator if g else "")}</div>{thumb}<div style="margin-top:6px">{"".join(chips)}</div>')
        banners = ""
        for ts, text in reversed(self.banners):
            if now - ts < 300:
                lines = [l.strip().strip("║").strip() for l in text.splitlines()]
                body = "\n".join(l for l in lines if l and not set(l) <= set("╔╗╚╝═║ "))
                banners += (f'<div style="margin-top:8px;border:1px solid #d29922;border-radius:8px;padding:6px 8px;'
                            f'background:#2d2205;white-space:pre-wrap;font-size:11px">{_esc(body[:900])}</div>')
        return (f'<div style="{_ROOT}display:flex;flex-wrap:wrap;gap:14px">'
                f'<div style="flex:1 1 440px;min-width:280px"><div style="{_MUTED}font-size:11px;margin-bottom:4px">'
                f'🔬 AGORA NO SIMULADOR — {_esc(self.frames_info)}</div>{live}'
                f'<div style="{_MUTED}font-size:10.5px">cada quadrado é um universo artificial do lote atual '
                f'(PCA dos canais → cor). A imagem muda a cada segundo.</div></div>'
                f'<div style="flex:1 1 300px;min-width:260px"><div style="{_MUTED}font-size:11px;margin-bottom:4px">'
                f'🏅 MELHOR CANDIDATO ATÉ AGORA</div>{best_html}{banners}</div></div>')

    def _charts_html(self) -> str:
        lab = self.lab
        h = lab.history
        col = lambda k: [r.get(k) for r in h]
        sparks = "".join([
            svg_spark(col("best_quality"), "melhor fitness", "#58a6ff"),
            svg_spark(col("map_coverage"), "cobertura MAP-Elites", "#3fb950", "{:.1%}"),
            svg_spark(col("best_level"), "melhor nível", "#ffd33d", "{:.0f}"),
            svg_spark(col("species"), "espécies", "#a371f7", "{:.0f}"),
            svg_spark(col("viable_fraction"), "fração viável", "#f0883e", "{:.1%}"),
            svg_spark(list(self.speed_hist), "velocidade (u·passos/s)", "#8b949e", "{:,.0f}"),
        ])
        counts = {L: 0 for L in range(10)}
        for l in lab.levels.values():
            if 0 <= l <= 9:
                counts[l] += 1
        mx = max(1, max(counts.values()))
        ladder = "".join(
            f'<div style="display:flex;align-items:center;gap:6px;font-size:11px">'
            f'<span style="width:190px;{_MUTED}">L{L} {_esc(LEVEL_NAMES[L][:24])}</span>'
            f'<span style="display:inline-block;height:9px;width:{max(1, int(150 * counts[L] / mx)) if counts[L] else 0}px;'
            f'background:{LEVEL_COLORS[L]};border-radius:3px"></span><span>{counts[L]}</span></div>'
            for L in range(9, -1, -1))
        zoo = "".join(_chip(f"{k.lower()} {v}", "#8b949e") for k, v in lab.zoo.counts().items() if v)
        via = getattr(lab, "last_viability", {}) or {}
        via_cols = {"VIABLE": "#3fb950", "DEAD": "#6e7681", "FROZEN": "#58a6ff", "EXPLOSIVE": "#f85149",
                    "PURE_NOISE": "#d29922", "NUMERICALLY_UNSTABLE": "#db6d28"}
        via_html = "".join(_chip(f"{k.lower()} {v}", via_cols.get(k, "#8b949e")) for k, v in sorted(via.items(), key=lambda x: -x[1]))
        ops = "".join(_chip(f"{k} {v:.2f}", "#58a6ff") for k, v in lab.bandit.probs().items())
        return (f'<div style="{_ROOT}">'
                f'<div style="{_MUTED}font-size:11px">📈 RESULTADOS POR GERAÇÃO (atualiza a cada geração)</div>'
                f'<div style="display:flex;flex-wrap:wrap">{sparks}</div>'
                f'<div style="display:flex;flex-wrap:wrap;gap:22px;margin-top:8px">'
                f'<div>{svg_map(lab.map)}</div>'
                f'<div><div style="{_MUTED}font-size:11px;margin-bottom:2px">ESCADA DE DESCOBERTA (candidatos retidos)</div>{ladder}</div>'
                f'<div style="max-width:380px"><div style="{_MUTED}font-size:11px">ÚLTIMA GERAÇÃO (sobrevivência dinâmica)</div>'
                f'{via_html or "<span style=color:#6e7681>-</span>"}'
                f'<div style="{_MUTED}font-size:11px;margin-top:6px">ZOOLÓGICO</div>{zoo or "<span style=color:#6e7681>-</span>"}'
                f'<div style="{_MUTED}font-size:11px;margin-top:6px">OPERADORES DE MUTAÇÃO (recompensa média, bandit UCB)</div>{ops}'
                f'</div></div></div>')

## 13. Ferramentas do pesquisador
`inspect_candidate(id)`, `compare_candidates(a, b)`, `replay_candidate(id)`, `reproduce_discovery(discovery_id)`,
`export_candidate(id)`, `import_candidate(path)`, `profile_genesis()`, `results_dataframe()`, `list_discoveries()`.

In [ ]:
# @title 13. Ferramentas do pesquisador { display-mode: "form" }
def _lab(lab=None) -> "GenesisLab":
    lab = lab or globals().get("LAB")
    if lab is None:
        raise RuntimeError("LAB ainda não inicializado — rode as células de boot.")
    return lab


def _resolve(lab, cid: Optional[str]) -> str:
    if cid is None:
        cid = lab.best_candidate()
    if cid not in lab.store:
        raise KeyError(f"candidato {cid} não está no store (talvez coletado; use um ID de arquivo/descoberta)")
    return cid


def inspect_candidate(candidate_id: Optional[str] = None, lab=None, show: bool = True, steps: int = 160):
    lab = _lab(lab); cid = _resolve(lab, candidate_id)
    g = lab.store[cid]
    print(g.summary(lab.space))
    print("\nScorecard:")
    for k, v in lab.scorecard(cid).items():
        print(f"  {k:>18}: {v}")
    print("\nDescritores:", {k: round(v, 4) for k, v in lab.desc.get(cid, {}).items() if isinstance(v, float)})
    print("\nLinhagem:\n" + lab.lineage.tree(cid, depth=2))
    print("\nExperimentos:")
    for k, rid in lab.tests.get(cid, {}).items():
        r = lab.results.get(rid)
        if r:
            print(f"  {k:>22}: {r.status:<12} score={r.score:.3f} p={r.p_value:.2e} ({r.version}, {r.duration_s:.1f}s)")
    if show:
        frames, _ = simulate_frames(lab, g, steps=steps, every=4)
        plot_frames(frames, f"{cid}", None, True)
        for k, rid in lab.tests.get(cid, {}).items():
            r = lab.results.get(rid)
            if r is None:
                continue
            if r.probe == "memory":
                plot_memory_curve(r, None, True)
            elif r.probe == "causality":
                plot_causal_map(r, None, True)
            elif r.probe == "recovery":
                plot_damage_curve(r, None, True)
            elif r.probe == "robustness":
                plot_robustness_curve(r, None, True)
    return lab.scorecard(cid)


def compare_candidates(a: str, b: str, lab=None) -> Dict[str, Any]:
    lab = _lab(lab)
    sa, sb = lab.scorecard(a), lab.scorecard(b)
    rows = [(k, sa.get(k), sb.get(k)) for k in sa.keys()]
    va, vb = lab.vec.get(a), lab.vec.get(b)
    bd = float(np.linalg.norm(va - vb)) if va is not None and vb is not None else float("nan")
    gd = float(np.sqrt(np.mean((lab.space.vector(lab.store[a].params) - lab.space.vector(lab.store[b].params)) ** 2))) \
        if a in lab.store and b in lab.store else float("nan")
    print(f"{'campo':>18} | {a:^24} | {b:^24}")
    for k, x, y in rows:
        print(f"{k:>18} | {str(x):^24} | {str(y):^24}")
    print(f"\ndistância comportamental={bd:.4f}  distância genética={gd:.4f}")
    stat = {}
    for key in ["replication_channel", "replication_memory"]:
        ra, rb = lab.result_of(a, key), lab.result_of(b, key)
        if ra and rb:
            x, y = np.asarray(ra.details["scores"]), np.asarray(rb.details["scores"])
            stat[key] = {"p_a_gt_b": mann_whitney_greater(x, y), "cohens_d": cohens_d(x, y)}
    if stat:
        print("comparação estatística:", stat)
    return {"rows": rows, "behavior_distance": bd, "genetic_distance": gd, "stats": stat}


def replay_candidate(candidate_id: Optional[str] = None, lab=None, env_id: Optional[str] = None, steps: int = 300,
                     gif: bool = True, seed: int = 0):
    lab = _lab(lab); cid = _resolve(lab, candidate_id)
    env = next((e for e in lab.envs if e.env_id == env_id), lab.envs[0])
    frames, _ = simulate_frames(lab, lab.store[cid], env, steps=steps, every=2, seed=seed)
    plot_frames(frames, f"replay {cid} @ {env.env_id}", None, True)
    if gif:
        p = save_animation(frames, os.path.join(lab.paths.plots, f"replay_{cid}_{env.env_id}.gif"))
        if p and IN_NOTEBOOK and IPImage is not None:
            display(IPImage(filename=p))
        return p
    return None


def reproduce_discovery(discovery_id: str, lab=None, deterministic: bool = True) -> Dict[str, Any]:
    """Restaura genoma, semente, protocolo, ambientes, duração e reexecuta o teste com o Validation Engine."""
    lab = _lab(lab)
    rec = lab.discoveries[discovery_id]
    g = genome_from_record(lab.space, rec["genome"])
    prot = rec["protocol"]
    probe = prot["probe"]
    kw = dict(prot.get("kwargs", {}))
    prev = torch.are_deterministic_algorithms_enabled()
    if deterministic:
        torch.use_deterministic_algorithms(True, warn_only=True)
    try:
        res = lab.ve.run_probe(probe, g, prot["seed"], **kw)
    finally:
        torch.use_deterministic_algorithms(prev, warn_only=True)
    same_status = res.status == rec["status"]
    a, b = res.score, rec["score"]
    same_score = (not np.isfinite(a) and not np.isfinite(b)) or (np.isfinite(a) and np.isfinite(b) and abs(a - b) < 1e-3)
    dev_now = lab.ve.sim.device.type
    dev_orig = prot.get("device", dev_now)
    out = {"discovery_id": discovery_id, "probe": probe, "original_status": rec["status"], "reproduced_status": res.status,
           "original_score": b, "reproduced_score": a, "status_match": same_status, "score_match": bool(same_score),
           "genome_hash_match": g.hash() == prot.get("genome_hash"), "engine": res.version,
           "device_original": dev_orig, "device_now": dev_now}
    if dev_orig != dev_now:
        out["note"] = ("dispositivos diferentes: a física determinística é a mesma (dentro da tolerância de float32), mas os "
                       "fluxos aleatórios (ruído intrínseco, assincronia, choques) de CPU e CUDA diferem — espera-se "
                       "concordância estatística (mesmo status), não números idênticos")
    print(json.dumps(to_jsonable(out), indent=1))
    return out


def export_candidate(candidate_id: Optional[str] = None, lab=None, path: Optional[str] = None) -> str:
    lab = _lab(lab); cid = _resolve(lab, candidate_id)
    path = path or os.path.join(lab.paths.exports, f"{cid}.genesis.pt")
    torch.save({"genesis_version": GENESIS_VERSION, "engine": VALIDATION_ENGINE_VERSION,
                "channels_max": lab.cfg.channels_max, "kernel_slots": lab.cfg.kernel_slots, "links_max": lab.cfg.links_max,
                "delay_max": lab.cfg.delay_max, "radius_max": lab.cfg.radius_max,
                "genome": genome_to_record(lab.space, lab.store[cid]), "descriptor": lab.desc.get(cid),
                "results": {k: lab.results[r].to_dict() for k, r in lab.tests.get(cid, {}).items() if r in lab.results},
                "lineage": {a: lab.lineage.records.get(a) for a in [cid] + lab.lineage.ancestors(cid)},
                "hash": lab.store[cid].hash()}, path)
    print(f"exportado: {path} ({human_bytes(os.path.getsize(path))})")
    return path


def import_candidate(path: str, lab=None, evaluate: bool = True) -> str:
    lab = _lab(lab)
    d = _torch_load(path)
    dims = ("channels_max", "kernel_slots", "links_max", "delay_max", "radius_max")
    if any(d.get(k) != getattr(lab.cfg, k) for k in dims):
        raise ValueError(f"dimensões do genoma incompatíveis com esta pesquisa: {[(k, d.get(k), getattr(lab.cfg, k)) for k in dims]}")
    g = genome_from_record(lab.space, d["genome"])
    if g.hash() != d["hash"]:
        raise ValueError("hash do genoma importado não confere (arquivo corrompido?)")
    g.genome_id = new_id("G"); g.origin = f"imported:{os.path.basename(path)}"; g.parent_ids = [d["genome"]["genome_id"]]
    g.generation = lab.counters["generation"]
    if evaluate:
        lab.evaluate_and_integrate([g], {})
        lab.zoo.add(g.genome_id, ["UNKNOWN"], lab.novelty_score.get(g.genome_id, 0.0), lab.counters["generation"])
        lab.schedule_experiments([g.genome_id])
    else:
        lab.store[g.genome_id] = g
    print(f"importado como {g.genome_id}")
    return g.genome_id


def profile_genesis(lab=None, steps: int = 20, batch: Optional[int] = None):
    lab = _lab(lab)
    B = batch or min(lab.cfg.batch_size, 64)
    gs = [Genome(lab.space.random_params(np.random.default_rng(i))) for i in range(B)]
    gb = lab.sim.prepare(gs)
    st = lab.sim.init_state(gb, [lab.envs[0]] * B, list(range(B)))
    try:
        from torch.profiler import profile, ProfilerActivity
        acts = [ProfilerActivity.CPU] + ([ProfilerActivity.CUDA] if torch.cuda.is_available() else [])
        with profile(activities=acts) as prof:
            lab.sim.run(gb, st, steps)
        print(prof.key_averages().table(sort_by="self_cuda_time_total" if torch.cuda.is_available() else "self_cpu_time_total", row_limit=15))
    except Exception as e:
        t0 = time.time(); lab.sim.run(gb, st, steps); _sync()
        print(f"profiler indisponível ({e}); {B * steps / (time.time() - t0):.0f} universo·passos/s")


def results_dataframe(lab=None):
    lab = _lab(lab)
    rows = [{"result_id": k, "candidate": r.candidate_id, "probe": r.probe, "status": r.status, "score": r.score,
             "p": r.p_value, "effect": r.effect_size, "split": r.split, "seconds": r.duration_s, "version": r.version}
            for k, r in lab.results.items()]
    return pd.DataFrame(rows) if HAVE_PANDAS else rows


def list_discoveries(lab=None):
    lab = _lab(lab)
    rows = [{"discovery_id": k, "candidate": v["candidate_id"], "level": v["new_level"], "probe": v["probe"],
             "score": v["score"], "generation": v["generation"], "global_record": v.get("global_record")}
            for k, v in lab.discoveries.items()]
    return pd.DataFrame(rows) if HAVE_PANDAS else rows


def _sync():
    if torch.cuda.is_available():
        torch.cuda.synchronize()

## 14. Suíte de testes de boot (teste do teste)
Roda **antes** da pesquisa. Cada instrumento científico tem controle positivo e negativo.
Se qualquer teste falhar: `GENESIS BOOT FAILED` + stack trace, e a pesquisa não inicia.

In [ ]:
# @title 14. Suíte de testes de boot (teste do teste) { display-mode: "form" }
def _boot_config(device: torch.device) -> GenesisConfig:
    cfg = profile_config("DEBUG", {"device": device.type}, 1234, 1.0, 0)
    cfg.world_size, cfg.population_size, cfg.batch_size = 32, 8, 16
    cfg.screen_steps, cfg.screen_burn_in, cfg.record_every = 48, 16, 4
    cfg.probe.trials, cfg.probe.permutations, cfg.probe.memory_delays = 8, 30, [1, 10, 20]
    cfg.probe.replication_seeds, cfg.probe.robust_trials = 2, 2
    cfg.max_experiments_per_generation = 2
    return cfg


class BootSuite:
    def __init__(self, device: torch.device, root: str):
        self.device, self.root = device, root
        self.results: List[Tuple[str, bool, float, str]] = []

    def run(self, name: str, fn: Callable[[], Optional[str]]):
        t0 = time.time()
        try:
            note = fn() or ""
            self.results.append((name, True, time.time() - t0, note))
        except Exception:
            self.results.append((name, False, time.time() - t0, traceback.format_exc()))

    def mini_lab(self, sub: str) -> "GenesisLab":
        cfg = _boot_config(self.device)
        paths = GenesisPaths(os.path.join(self.root, sub), on_drive=False)
        return GenesisLab(cfg, {"device": self.device.type, "gpu_name": "boot-test"}, paths,
                          EventLogger(paths, verbose=False), self.device)

    # ------------------------------------------------------------------
    def all(self) -> bool:
        cfg = _boot_config(self.device)
        space = ParamSpace(cfg)
        sim = Simulator(cfg, self.device)
        envs = make_environment_suite(cfg.master_seed)
        scr = Screener(sim, cfg)
        ve = ValidationEngine(sim, cfg, envs, scr, space)
        rng = np.random.default_rng(0)
        D = ve.channel_distance(cfg.world_size)
        ctrl = {k: control_genome(space, k, D, cfg.world_size) for k in ["identity", "noise_replace", "diffusion", "link_copy"]}

        def test_state_initialization():
            gs = [Genome(space.random_params(rng)) for _ in range(6)]
            gb = sim.prepare(gs)
            st = sim.init_state(gb, [envs[i % len(envs)] for i in range(6)], list(range(6)))
            assert st.U.shape == (6, cfg.channels_max, cfg.world_size, cfg.world_size)
            assert torch.isfinite(st.U).all()
            inactive = (gb.chm.view(6, -1) == 0)
            assert float(st.U.abs().sum((2, 3))[inactive].sum()) == 0.0, "canal inativo com estado não nulo"
            for e in envs:
                f = make_initial_field(e, 3, 16, 7, np.ones(3, bool)); f2 = make_initial_field(e, 3, 16, 7, np.ones(3, bool))
                assert np.array_equal(f, f2), "inicialização não reprodutível"

        def test_simulation_kernel():
            for shp in range(len(KERNEL_SHAPES)):
                k = build_kernel_np(shp, 3.0, 0.5, 0.15, 0.6, 0.7, cfg.radius_max)
                assert abs(k.sum() - 1) < 1e-5
            g = Genome(space.random_params(rng)); g.params["boundary"] = np.int64(0)
            g.params["k_active"][:] = False; g.params["k_active"][0] = True
            gb = sim.prepare([g]); U = torch.randn(1, cfg.channels_max, cfg.world_size, cfg.world_size, device=self.device)
            P = gb.P
            N = torch.fft.irfft2(torch.fft.rfft2(sim.pad(U, gb, P)).unsqueeze(1) * gb.Khat.unsqueeze(2), s=(gb.Hp, gb.Hp))[..., P:P + gb.S, P:P + gb.S]
            p = g.params
            kk = torch.as_tensor(build_kernel_np(int(p["k_shape"][0]), float(p["k_radius"][0]), float(p["k_mu"][0]), float(p["k_sigma"][0]),
                                                 float(p["k_aniso"][0]), float(p["k_angle"][0]), cfg.radius_max), device=self.device)
            ref = F.conv2d(F.pad(U.view(-1, 1, gb.S, gb.S), (P, P, P, P), mode="circular"), kk.flip(0, 1).view(1, 1, *kk.shape)).view_as(U)
            err = float((N[:, 0] - ref).abs().max())
            assert err < 1e-4, f"FFT conv != conv direta ({err})"
            lap = N[:, gb.K]
            ref_lap = (U.roll(1, 2) + U.roll(-1, 2) + U.roll(1, 3) + U.roll(-1, 3) - 4 * U)
            assert float((lap - ref_lap).abs().max()) < 1e-4, "laplaciano incorreto"
            return f"FFT-conv err={err:.1e}"

        def test_dynamics_shape():
            for b in range(len(BOUNDARIES)):
                gs = [Genome(space.random_params(rng)) for _ in range(4)]
                for g in gs:
                    g.params["boundary"] = np.int64(b); g.params["link_active"][:] = True
                gb = sim.prepare(gs)
                st = sim.init_state(gb, [envs[0]] * 4, [1, 2, 3, 4])
                st = sim.run(gb, st, 5)
                assert st.U.shape == (4, cfg.channels_max, cfg.world_size, cfg.world_size)
                assert st.G is None or st.G.shape[1] == cfg.links_max
            mixed = [Genome(space.random_params(rng)) for _ in range(3)]
            for i, g in enumerate(mixed):
                g.params["boundary"] = np.int64(i)
            gb = sim.prepare(mixed); st = sim.run(gb, sim.init_state(gb, [envs[0]] * 3, [5, 6, 7]), 5)
            assert torch.isfinite(st.U).all()

        def test_no_nan():
            gs = [Genome(space.random_params(rng)) for _ in range(16)]
            boom = Genome(space.random_params(rng))
            boom.params["W"][:] = 2.0; boom.params["g_gain"][:] = 2.0; boom.params["Q"][:] = 1.0; boom.params["sat"][:] = 4.0
            boom.params["max_rate"] = np.float32(1.0); boom.params["diss"][:] = 0
            gs.append(boom)
            gb = sim.prepare(gs)
            st = sim.run(gb, sim.init_state(gb, [envs[1]] * 17, list(range(17))), 60)
            assert torch.isfinite(st.U).all(), "NaN/Inf escapou"
            assert float(st.U.abs().max()) <= cfg.state_bound + 1e-5
            d = scr.screen([boom], envs[:1], 1, steps=40)[0]
            assert d["viability"] in ("EXPLOSIVE", "VIABLE", "NUMERICALLY_UNSTABLE", "PURE_NOISE", "FROZEN")
            return f"lei explosiva classificada como {d['viability']}"

        def test_mutation():
            g = Genome(space.random_params(rng))
            for i in range(200):
                op = MUTATION_OPERATORS[i % len(MUTATION_OPERATORS)]
                p, rec = space.mutate(g.params, rng, op, 1.0, space.random_params(rng))
                for n, s in space.specs.items():
                    v = np.asarray(p[n], np.float64)
                    if s.kind != "bool":
                        assert v.min() >= s.low - 1e-6 and v.max() <= s.high + 1e-6, f"{n} fora dos limites"
                assert p["ch_active"].any() and p["k_active"].any()
                assert rec
                g = Genome(p, parent_ids=[g.genome_id])
            p = space.random_params(rng); p["ch_active"][:] = False; p["ch_active"][0] = True
            p2 = {k: np.array(v) for k, v in p.items()}
            msg = space.duplicate_channel(p2, rng)
            assert p2["ch_active"].sum() == 2 and "dup_channel" in msg
            v = pack_params(space, p); p3 = unpack_params(space, v)
            assert hash_obj(space.clamp(p3)) == hash_obj(space.clamp({k: np.array(x) for k, x in p.items()})), "pack/unpack"

        def test_reproducibility():
            g = Genome(space.random_params(np.random.default_rng(5)))
            outs = []
            for _ in range(2):
                gb = sim.prepare([g, g])
                st = sim.run(gb, sim.init_state(gb, [envs[1]] * 2, [11, 11], groups=[0, 0], run_seed=3), 30)
                outs.append(st.U.clone())
            assert torch.allclose(outs[0], outs[1], atol=1e-6), "mesma semente != mesmo resultado"
            assert torch.allclose(outs[0][0], outs[0][1], atol=1e-6), "cópias idênticas divergiram"
            d1 = scr.screen([g], envs[:2], 9)[0]; d2 = scr.screen([g], envs[:2], 9)[0]
            assert abs(d1["persistence"] - d2["persistence"]) < 1e-5 and abs(d1["causal_spread"] - d2["causal_spread"]) < 1e-5

        def test_causal_probe():
            pos = ve.run_probe("causality", ctrl["diffusion"], 1)
            neg = ve.run_probe("causality", ctrl["identity"], 1)
            assert pos.status == STATUS_PASS, f"controle positivo (difusão) -> {pos.status} {pos.details.get('error', '')}"
            assert neg.status == STATUS_FAIL, f"controle negativo (identidade) -> {neg.status}"
            return f"+:{pos.details['spread_mean']:.2f} −:{neg.details['spread_mean']:.2f}"

        def test_memory_probe():
            pos = ve.run_probe("memory", ctrl["identity"], 2)
            neg = ve.run_probe("memory", ctrl["noise_replace"], 2)
            assert pos.status == STATUS_PASS, f"controle positivo (identidade) -> {pos.status} {pos.details.get('error', '')}"
            assert neg.status != STATUS_PASS, f"controle negativo (ruído) -> {neg.status}"
            assert not pos.details["leakage_detected"]
            return f"+:{pos.score:.2f} −:{neg.status}"

        def test_channel_probe():
            pos = ve.run_probe("channel", ctrl["link_copy"], 3)
            neg = ve.run_probe("channel", ctrl["identity"], 3)
            assert pos.status == STATUS_PASS, f"controle positivo (link) -> {pos.status} {pos.details.get('error', '')}"
            assert neg.status == STATUS_FAIL, f"controle negativo (isolado) -> {neg.status}"
            return f"+: latência={pos.details['latency']} MI={pos.details['mi_bits']:.2f} bits"

        def test_transformation_instrument():
            r_ = np.random.default_rng(1)
            a1, a2 = r_.integers(0, 2, 64), r_.integers(0, 2, 64)
            Xp = np.stack([a1, a2, a1 * a2], 1) + r_.normal(0, 0.1, (64, 3))
            Xn = np.stack([a1, a2], 1) + r_.normal(0, 0.1, (64, 2))
            pos = decode_report(Xp, a1 ^ a2, 2, 60, 4, 1); neg = decode_report(Xn, a1 ^ a2, 2, 60, 4, 1)
            assert pos["linear_p"] < 0.01 and pos["linear_acc"] > 0.85, "XOR com termo não linear deveria ser decodificável"
            assert neg["linear_p"] > 0.01, "XOR de features lineares não deveria ser linearmente decodificável"
            return f"+:{pos['linear_acc']:.2f} −:{neg['linear_acc']:.2f}"

        def test_recovery_probe():
            uni = [Environment("CTRL-U", "control", "uniform", 1, 0.5, 1, 0, 0, 0, 7)]
            pos = ve.run_probe("recovery", ctrl["diffusion"], 4, allow_fallback=True, envs=uni)
            neg = ve.run_probe("recovery", ctrl["identity"], 4, allow_fallback=True, envs=uni)
            assert pos.status == STATUS_PASS, f"controle positivo (difusão sobre campo uniforme) -> {pos.status}"
            assert neg.status == STATUS_FAIL, f"controle negativo (estático) -> {neg.status}"
            return f"+:{pos.score:.2f} −:{neg.score:.2f}"

        def test_archive():
            me = MapElites(["persistence", "causal_spread", "temporal_complexity"], [4, 4, 2])
            d = {"persistence": 0.5, "causal_spread": 0.2, "temporal_complexity": 0.9}
            assert me.try_insert("a", d, 0.3, 0) == (True, True)
            assert me.try_insert("b", d, 0.2, 0) == (False, False)
            assert me.try_insert("c", d, 0.5, 1) == (True, False)
            assert me.cells[me.index(d)]["genome_id"] == "c" and me.history[0]["genome_id"] == "a", "elite histórica perdida"
            na = NoveltyArchive(3, 100)
            V = np.random.default_rng(0).random((10, len(DESCRIPTOR_NAMES))).astype(np.float32)
            nv = na.novelty(V, V); assert nv.shape == (10,) and (nv > 0).all()
            na.maybe_add([f"g{i}" for i in range(10)], V, nv + 1, np.random.default_rng(0)); assert len(na.ids) == 10
            z = Zoo(); dd = {"viability": "VIABLE", "rel_activity": 0.1, "persistence": 0.1, "structures": 0, "periodicity": 0.1,
                             "causal_spread": 0.0, "lyapunov": 0.0, "mobility": 0.0}
            assert Zoo.classify(dd, {}, Thresholds(), False) == ["UNKNOWN"]
            assert "ANOMALOUS" in Zoo.classify(dd, {}, Thresholds(), True)
            lin = Lineage(); g0 = Genome(space.random_params(rng)); g1 = Genome(space.random_params(rng), parent_ids=[g0.genome_id])
            lin.add(g0); lin.add(g1); assert lin.ancestors(g1.genome_id) == [g0.genome_id] and g1.genome_id in lin.tree(g0.genome_id)
            sp = Speciation(0.3); s1 = sp.assign(np.zeros(5), np.zeros(5), 0); s2 = sp.assign(np.ones(5), np.ones(5), 0)
            assert s1 != s2 and sp.assign(np.zeros(5) + 0.01, np.zeros(5), 0) == s1

        def test_checkpoint_save():
            lab = self.mini_lab("ckpt")
            lab.initialize()
            p = lab.save_checkpoint("test")
            assert os.path.exists(p) and not os.path.exists(lab.ckpt.tmp)
            lab.counters["generation"] += 1
            lab.save_checkpoint("test2")
            assert os.path.exists(lab.ckpt.previous)
            return human_bytes(os.path.getsize(p))

        def test_checkpoint_restore():
            lab = self.mini_lab("ckpt")
            st, path = lab.ckpt.load_latest()
            assert st is not None and st["counters"]["generation"] == 1
            lab2 = self.mini_lab("ckpt"); lab2.load_state_dict(st)
            assert len(lab2.population) == len(st["population"]) and set(lab2.store) == set(st["store"])
            with open(lab.ckpt.current, "wb") as f:
                f.write(b"corrupted")
            st2, path2 = lab.ckpt.load_latest()
            assert st2 is not None and path2 == lab.ckpt.previous, "fallback para checkpoint_previous falhou"
            return "corrupção detectada → previous"

        def test_resume():
            lab = self.mini_lab("resume")
            lab.initialize()
            lab.generation_step()
            ev1, gen1 = lab.counters["evaluations"], lab.counters["generation"]
            pop1 = list(lab.population)
            lab.save_checkpoint("t")
            lab2 = self.mini_lab("resume")
            st, _ = lab2.ckpt.load_latest()
            lab2.load_state_dict(st)
            assert lab2.counters["generation"] == gen1 and lab2.counters["evaluations"] == ev1 and lab2.population == pop1
            lab2.generation_step()
            assert lab2.counters["generation"] == gen1 + 1 and lab2.counters["evaluations"] > ev1, "contadores reiniciaram"
            return f"gen {gen1}→{lab2.counters['generation']}, evals {ev1}→{lab2.counters['evaluations']}"

        def test_discovery_reproduction():
            lab = self.mini_lab("repro")
            g = ctrl["identity"]; lab.store[g.genome_id] = g
            res = lab.ve.run_probe("memory", g, 77)
            lab.best_level_global = 99          # teste silencioso: sem relatório/bundle
            lab.register_discovery(g.genome_id, 2, 3, res, ["test"])
            did = list(lab.discoveries)[-1]
            with contextlib.redirect_stdout(io.StringIO()):
                out = reproduce_discovery(did, lab)
            assert out["status_match"] and out["score_match"] and out["genome_hash_match"], f"reprodução divergiu: {out}"

        def test_device_consistency():
            """A física determinística deve coincidir entre CPU e CUDA (tolerância de float32).
            Os geradores aleatórios de CPU e CUDA produzem sequências DIFERENTES para a mesma semente, então ruído
            intrínseco, assincronia e choques ficam desligados aqui (eles são reprodutíveis apenas no mesmo tipo de
            dispositivo). Leis aleatórias são comparadas em 1 passo (caos amplificaria arredondamentos legítimos);
            leis conhecidas e estáveis, em 30 passos."""
            if not torch.cuda.is_available():
                return "sem GPU — pulado (CPU apenas)"
            r_ = np.random.default_rng(3)
            gs = []
            for i in range(6):
                g = Genome(space.random_params(r_))
                g.params["noise"] = np.array(0.0, np.float32)
                g.params["async_p"] = np.array(1.0, np.float32)
                g.params["boundary"] = np.array(i % len(BOUNDARIES), np.int64)
                gs.append(g)
            quiet = dc_replace(envs[0], ext_noise=0.0, shock_period=0, shock_amp=0.0)

            def run(dev, genomes, steps):
                s_ = Simulator(cfg, dev); gb = s_.prepare(genomes)
                st = s_.init_state(gb, [quiet] * len(genomes), list(range(len(genomes))), run_seed=1)
                return s_.run(gb, st, steps).U.cpu()

            def rel_err(a, b):
                return float(((a - b).abs().flatten(1).amax(1) / (a.abs().flatten(1).amax(1) + 1e-6)).max())
            e1 = rel_err(run(torch.device("cpu"), gs, 1), run(torch.device("cuda"), gs, 1))
            ctl = [control_genome(space, k, D, cfg.world_size) for k in ["diffusion", "reaction_diffusion", "link_copy"]]
            e30 = rel_err(run(torch.device("cpu"), ctl, 30), run(torch.device("cuda"), ctl, 30))
            assert e1 < 1e-3, f"1 passo, leis aleatórias sem ruído: CPU vs CUDA erro relativo {e1:.2e}"
            assert e30 < 1e-3, f"30 passos, leis de controle: CPU vs CUDA erro relativo {e30:.2e}"
            return f"rel.err 1 passo={e1:.1e}, 30 passos (controles)={e30:.1e}; ruído: fluxo por dispositivo"

        def test_live_panel():
            lab = self.mini_lab("live")
            lab.initialize()
            parts = {}

            class _H:
                def __init__(self, k): self.k = k
                def update(self, h): parts[self.k] = getattr(h, "data", str(h))

            class _M(LiveMonitor):
                def _open_handles(self, now):
                    best = self.lab.best_candidate()
                    parts.update(head=self._head_html(now), view=self._view_html(now, best), charts=self._charts_html())
                    self.handles = {k: _H(k) for k in ("head", "view", "charts")}
                    self.last_refresh = self.view_ts = self.charts_ts = now
            m = _M(lab, interval=0.0)
            m.mode = "html"
            LIVE.monitor = m
            try:
                m.start()
                lab.generation_step()
                m.render(force=True)
            finally:
                LIVE.monitor = None
            assert m.errors == 0, "erro ao desenhar o painel (veja events.jsonl: live_render_error)"
            assert "AO VIVO" in parts["head"] and "MELHOR CANDIDATO" in parts["view"] and "<svg" in parts["charts"]
            assert "data:image/jpeg;base64," in parts["view"], "mosaico ao vivo não foi gerado"
            jpg = mosaic_jpeg([np.random.default_rng(0).normal(size=(cfg.channels_max, 16, 16))])
            assert jpg[:2] == b"\xff\xd8", "JPEG inválido"
            return f"painel: {sum(len(v) for v in parts.values()) // 1024} KB de HTML por quadro"

        for name, fn in [("test_state_initialization", test_state_initialization), ("test_simulation_kernel", test_simulation_kernel),
                         ("test_dynamics_shape", test_dynamics_shape), ("test_no_nan", test_no_nan),
                         ("test_mutation", test_mutation), ("test_checkpoint_save", test_checkpoint_save),
                         ("test_checkpoint_restore", test_checkpoint_restore), ("test_reproducibility", test_reproducibility),
                         ("test_causal_probe", test_causal_probe), ("test_memory_probe", test_memory_probe),
                         ("test_channel_probe", test_channel_probe), ("test_transformation_instrument", test_transformation_instrument),
                         ("test_recovery_probe", test_recovery_probe), ("test_archive", test_archive),
                         ("test_resume", test_resume), ("test_discovery_reproduction", test_discovery_reproduction),
                         ("test_device_consistency", test_device_consistency), ("test_live_panel", test_live_panel)]:
            self.run(name, fn)
        return all(ok for _, ok, _, _ in self.results)

    def report(self) -> str:
        lines = []
        for name, ok, dt, note in self.results:
            first = note.strip().splitlines()[-1] if note.strip() else ""
            lines.append(f"  {'PASS' if ok else 'FAIL'}  {name:<32} {dt:6.2f}s  {first[:90] if ok else ''}")
        return "\n".join(lines)


def run_boot_tests(device: torch.device, root: str) -> BootSuite:
    test_root = tempfile.mkdtemp(prefix="genesis_boot_")       # disco local (rápido; não suja o Drive)
    suite = BootSuite(device, test_root)
    t0 = time.time()
    ok = suite.all()
    print(suite.report())
    shutil.rmtree(test_root, ignore_errors=True)
    if not ok:
        print("\n╔══════════════════════════════════════╗\n║          GENESIS BOOT FAILED         ║\n╚══════════════════════════════════════╝")
        for name, passed, _, note in suite.results:
            if not passed:
                print(f"\n--- {name} ---\n{note}")
        raise RuntimeError("GENESIS BOOT FAILED — pesquisa não iniciada (veja os stack traces acima)")
    print(f"\nTodos os {len(suite.results)} testes de boot passaram em {time.time() - t0:.1f}s.")
    return suite

## 15. BOOT → HARDWARE DETECTION → AUTO-TUNING
Detecta CPU/GPU e VRAM, escolhe perfil, mede a memória real por batch (busca dobrando até o alvo de VRAM do perfil
— EXTREME ≈ 87%), testa precisão mista e `torch.compile` (só adota se for estável **e** mais rápido) e roda o
microbenchmark. Throughput exibido é **estimativa, não promessa**.

In [ ]:
# @title 15. BOOT → HARDWARE DETECTION → AUTO-TUNING { display-mode: "form" }
def autotune(cfg: GenesisConfig, hw: Dict[str, Any], device: torch.device) -> Dict[str, Any]:
    rep: Dict[str, Any] = {"profile": cfg.profile}
    space = ParamSpace(cfg)
    env = make_environment_suite(cfg.master_seed)[0]
    rng = np.random.default_rng(0)
    proto = [Genome(space.random_params(rng)) for _ in range(32)]
    for g in proto:
        g.params["link_active"][:] = True     # pior caso de memória
        g.params["delay_w"][:] = 0.5
    sim = Simulator(cfg, device)

    def trial(B, steps=3):
        gl = (proto * (B // len(proto) + 1))[:B]
        gb = sim.prepare(gl)
        st = sim.init_state(gb, [env] * B, list(range(B)))
        st = sim.run(gb, st, steps)
        _sync()
        del gb, st

    if device.type == "cuda":
        target = VRAM_TARGET.get(cfg.profile, 0.6)
        free = torch.cuda.mem_get_info()[0]
        best, B = 16, 16
        while B <= 16384:
            try:
                torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
                base = torch.cuda.memory_allocated()
                trial(B)
                peak = torch.cuda.max_memory_allocated() - base
                rep[f"peak@{B}"] = peak
                # margem para gravação de quadros da triagem e para os probes
                if peak * 1.35 > target * free:
                    break
                best = B
                B *= 2
            except Exception as e:
                if is_oom_error(e):
                    torch.cuda.empty_cache()
                    break
                raise
        torch.cuda.empty_cache()
        cfg.batch_size = int(best)
        cap = {"DEBUG": 24, "FAST": 256, "RESEARCH": 1024, "EXTREME": 2048}[cfg.profile]
        # um lote de triagem por geração: resultados novos com mais frequência no painel
        cfg.population_size = int(max(32, min(cap, best // max(1, 2 * cfg.screen_envs))))
        rep["vram_free"] = free; rep["vram_target_fraction"] = target
    else:
        torch.set_num_threads(max(1, os.cpu_count() or 1))
        cfg.batch_size = min(cfg.batch_size, 64)
    # precisão mista: só se estável e mais rápida
    rep["precision"] = "fp32"
    if device.type == "cuda" and cfg.profile in ("RESEARCH", "EXTREME"):
        try:
            Bt = min(cfg.batch_size, 128)
            gl = (proto * (Bt // len(proto) + 1))[:Bt]
            outs, times = {}, {}
            for prec in ["fp32", "bf16" if hw.get("bf16_supported") else "fp16"]:
                c2 = copy.deepcopy(cfg); c2.precision = prec
                s2 = Simulator(c2, device); gb = s2.prepare(gl)
                st = s2.init_state(gb, [env] * Bt, list(range(Bt)), run_seed=1)
                s2.run(gb, st, 3); _sync(); t0 = time.time()
                st = s2.run(gb, st, 40); _sync()
                times[prec] = time.time() - t0; outs[prec] = st.U.float()
            lp = [k for k in outs if k != "fp32"][0]
            rel = float((outs[lp] - outs["fp32"]).abs().mean() / (outs["fp32"].abs().mean() + 1e-9))
            speed = times["fp32"] / max(1e-9, times[lp])
            rep["precision_test"] = {"candidate": lp, "rel_error": rel, "speedup": speed}
            if rel < 0.02 and speed > 1.15:
                cfg.precision = lp; rep["precision"] = lp
        except Exception as e:
            rep["precision_test"] = {"error": repr(e)}
    # torch.compile: testado automaticamente; usado só se estável e mais rápido
    rep["compile"] = False
    if device.type == "cuda" and cfg.profile in ("RESEARCH", "EXTREME") and hasattr(torch, "compile"):
        try:
            Bt = min(cfg.batch_size, 256)
            gl = (proto * (Bt // len(proto) + 1))[:Bt]
            res = {}
            for comp in [False, True]:
                c2 = copy.deepcopy(cfg); c2.use_compile = comp; c2.batch_size = Bt   # compile só vale no batch de triagem
                s2 = Simulator(c2, device); gb = s2.prepare(gl)
                st = s2.init_state(gb, [env] * Bt, list(range(Bt)), run_seed=1)
                s2.run(gb, st, 3); _sync(); t0 = time.time()
                st = s2.run(gb, st, 30); _sync()
                res[comp] = (time.time() - t0, st.U.clone())
            err = float((res[True][1] - res[False][1]).abs().max())
            speed = res[False][0] / max(1e-9, res[True][0])
            rep["compile_test"] = {"speedup": speed, "max_abs_diff": err}
            if speed > 1.15 and err < 1e-3:
                cfg.use_compile = True; rep["compile"] = True
        except Exception as e:
            rep["compile_test"] = {"error": repr(e)[:300]}
    # microbenchmark
    sim = Simulator(cfg, device)
    B = cfg.batch_size
    gl = (proto * (B // len(proto) + 1))[:B]
    gb = sim.prepare(gl)
    st = sim.init_state(gb, [env] * B, list(range(B)))
    sim.run(gb, st, 2); _sync()
    t0 = time.time(); n = 10
    sim.run(gb, st, n); _sync()
    dt = time.time() - t0
    ups = B * n / dt
    per_eval = 2 * cfg.screen_steps * cfg.screen_envs
    rep.update({"batch_size": cfg.batch_size, "population_size": cfg.population_size, "world_size": cfg.world_size,
                "steps_per_sec": n / dt, "universe_steps_per_sec": ups, "universes_per_sec": ups / cfg.screen_steps,
                "estimated_evaluations_per_hour": ups / per_eval * 3600,
                "snapshot_frequency_minutes": cfg.checkpoint_every_minutes})
    del gb, st
    if device.type == "cuda":
        torch.cuda.empty_cache()
    return rep


# ----------------------------------------------------------------------------
print("╔══════════════════════════════════════╗\n║          PROJECT GENESIS Ω           ║\n║               BOOT                   ║\n╚══════════════════════════════════════╝")
random.seed(MASTER_SEED); np.random.seed(MASTER_SEED % (2 ** 32)); torch.manual_seed(MASTER_SEED)
torch.set_flush_denormal(True)   # evita lentidão de subnormais em CPU (campos que decaem a ~0)
PATHS = setup_storage(USE_GOOGLE_DRIVE, GENESIS_ROOT_OVERRIDE)
LOGGER = EventLogger(PATHS)
HW = detect_hardware()
DEVICE = torch.device(HW["device"])
if DEVICE.type == "cuda":
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = False      # dinâmica numérica em FP32 real (sem TF32 em Ampere+)
    torch.backends.cudnn.allow_tf32 = False
    torch.set_float32_matmul_precision("highest")
CFG = profile_config(PROFILE, HW, MASTER_SEED, MAX_SESSION_HOURS, MAX_GENERATIONS_THIS_SESSION)
CFG.live_interval = float(LIVE_UPDATE_SECONDS)
print(f"armazenamento: {PATHS.root} ({'Google Drive' if PATHS.on_drive else 'local — NÃO sobrevive à perda da sessão do Colab'})")
print(f"hardware: {HW['gpu_name']} | VRAM {human_bytes(HW['vram_total'])} | CPU x{HW['cpu_count']} | torch {HW['torch']} CUDA {HW['cuda_version']}")
print(f"perfil: {CFG.profile}")
TUNE = autotune(CFG, HW, DEVICE)
print(f"auto-tuning: batch={CFG.batch_size} população={CFG.population_size} mundo={CFG.world_size}²×{CFG.channels_max} "
      f"precisão={CFG.precision} compile={CFG.use_compile}")
print(f"benchmark (ESTIMATIVA): {TUNE['steps_per_sec']:.1f} passos/s · {TUNE['universe_steps_per_sec']:,.0f} universo·passos/s · "
      f"{TUNE['universes_per_sec']:.1f} universos/s · ~{TUNE['estimated_evaluations_per_hour']:,.0f} avaliações/h")
LOGGER.event("boot", hardware=HW, tuning=TUNE, profile=CFG.profile)

## 16. TEST SUITE (obrigatória antes da pesquisa)

In [ ]:
# @title 16. TEST SUITE (obrigatória antes da pesquisa) { display-mode: "form" }
if RUN_BOOT_TESTS:
    BOOT_SUITE = run_boot_tests(DEVICE, PATHS.root)
    LOGGER.event("boot_tests_passed", n=len(BOOT_SUITE.results))
else:
    print("⚠ testes de boot desativados pelo usuário (RUN_BOOT_TESTS=False)")

## 17. CHECKPOINT SEARCH → NEW GENESIS / RESUME
A configuração científica (tamanho do mundo, canais, thresholds) fica **travada** pelo checkpoint (research config lock);
apenas parâmetros de execução (batch, precisão, compile, limite de sessão) seguem o hardware desta sessão.

In [ ]:
# @title 17. CHECKPOINT SEARCH → NEW GENESIS / RESUME { display-mode: "form" }
RUNTIME_FIELDS = ["batch_size", "precision", "use_compile", "max_session_hours", "max_generations_session",
                  "checkpoint_every_minutes", "dashboard_every_seconds", "population_size", "live_interval"]


def open_lab() -> "GenesisLab":
    probe_ckpt = CheckpointManager(PATHS, LOGGER)
    state, path = (None, "") if FORCE_NEW_GENESIS else probe_ckpt.load_latest()
    if state is not None:
        cfg = GenesisConfig.from_dict(state["config"])
        for f_ in RUNTIME_FIELDS:
            if f_ == "population_size":
                continue
            setattr(cfg, f_, getattr(CFG, f_))
        if cfg.profile != CFG.profile:
            print(f"[resume] perfil do checkpoint ({cfg.profile}) mantido para consistência científica")
        lab = GenesisLab(cfg, HW, PATHS, LOGGER, DEVICE)
        lab.load_state_dict(state)
        lab.counters["sessions"] += 1
        lab.ckpt.last_save_ts = os.path.getmtime(path)
        print(f"RESUME: {path}\n  geração {lab.counters['generation']} · {lab.counters['evaluations']:,} universos avaliados · "
              f"runtime total {fmt_hms(lab.counters['total_runtime'])} · sessão #{lab.counters['sessions']}")
        LOGGER.event("resume", path=path, generation=lab.counters["generation"], evaluations=lab.counters["evaluations"])
    else:
        lab = GenesisLab(CFG, HW, PATHS, LOGGER, DEVICE)
        lab.counters["sessions"] = 1
        print("NEW GENESIS: criando população inicial de universos aleatórios...")
        lab.initialize()
        lab.save_checkpoint("genesis")
    return lab


LAB = open_lab()
print(dashboard_text(LAB, "READY"))
print("\n▼ O painel AO VIVO aparece logo abaixo, na seção 18 (atualiza sozinho a cada segundo).")

### Motor da pesquisa contínua
```
while not champion_gate_passed:
    evolve(); evaluate(); experiment(); archive(); checkpoint()
```
Colab não é infinito: a continuidade é **por checkpoint**. Cada sessão é um trecho da mesma experiência.
OOM → libera cache, reduz batch, registra, restaura o último checkpoint e continua.

In [ ]:
# @title Motor da pesquisa contínua { display-mode: "form" }
def resource_check(lab: "GenesisLab"):
    try:
        usage = shutil.disk_usage(lab.paths.root)
        if usage.free < 2e9:
            for f_ in sorted(glob.glob(os.path.join(lab.paths.plots, "*")))[:-50]:
                os.remove(f_)
            lab.logger.event("storage_pressure", echo=True, free=usage.free)
        if HAVE_PSUTIL and psutil.virtual_memory().percent > 90:
            lab.garbage_collect()
            lab.logger.event("ram_pressure", echo=True, percent=psutil.virtual_memory().percent)
    except Exception as e:
        lab.logger.event("resource_check_failed", error=repr(e))


def run_genesis(lab: "GenesisLab", live_interval: Optional[float] = None) -> str:
    cfg = lab.cfg
    t_end = lab.session_start + cfg.max_session_hours * 3600 - 300
    gens_this_session = 0
    reason = "unknown"
    monitor = LiveMonitor(lab, interval=live_interval or cfg.live_interval, text_every=cfg.dashboard_every_seconds)
    LIVE.monitor = monitor
    verbose_before = lab.logger.verbose
    if monitor.mode == "html":
        lab.logger.verbose = False          # mensagens vão para o painel, não poluem a saída
    lab.logger.event("research_start", echo=True, generation=lab.counters["generation"], session=lab.counters["sessions"])
    monitor.start()
    ref_speed = None
    slow = 0
    try:
        while lab.champion is None:
            if os.path.exists(lab.paths.stop_file):
                reason = "STOP file"; os.remove(lab.paths.stop_file); break
            if time.time() > t_end:
                reason = "MAX_SESSION_HOURS"; break
            if cfg.max_generations_session and gens_this_session >= cfg.max_generations_session:
                reason = "MAX_GENERATIONS_THIS_SESSION"; break
            try:
                row = lab.generation_step()
                gens_this_session += 1
            except Exception as e:
                if not is_oom_error(e):
                    raise
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
                old = cfg.batch_size
                cfg.batch_size = max(2, cfg.batch_size // 2)
                lab.counters["oom_events"] += 1
                lab.logger.event("OOM", echo=True, old_batch=old, new_batch=cfg.batch_size)
                st, p = lab.ckpt.load_latest()
                if st is not None:
                    oom = lab.counters["oom_events"]
                    lab.load_state_dict(st)
                    lab.counters["oom_events"] = oom
                continue
            ups = row["universe_steps_per_s"]
            ref_speed = ups if ref_speed is None else max(ref_speed * 0.99, ups)
            slow = slow + 1 if ups < 0.5 * ref_speed else 0
            if slow >= 3:
                lab.logger.event("performance_degradation", echo=True, speed=ups, reference=ref_speed)
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
                slow = 0
            if gens_this_session % 10 == 0:
                resource_check(lab)
            if time.time() - lab.last_checkpoint_ts > cfg.checkpoint_every_minutes * 60:
                lab.save_checkpoint("periodic")
        if lab.champion is not None:
            reason = "GENESIS CHAMPION"
    except KeyboardInterrupt:
        reason = "pausado pelo usuário"
    except Exception as e:
        reason = f"crash: {e!r}"
        crash = os.path.join(lab.paths.crash, f"crash_{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}.txt")
        with open(crash, "w") as f:
            f.write(traceback.format_exc())
        lab.logger.event("crash", echo=True, error=repr(e), report=crash)
        print(traceback.format_exc())
    finally:
        try:
            lab.save_checkpoint(f"session_end:{reason}")
        except Exception as e:
            print(f"[checkpoint] falha ao salvar no encerramento: {e}")
        rp = session_report(lab, reason)
        final = "GENESIS CHAMPION" if lab.champion else f"PAUSADO — {reason}"
        monitor.finish(final)
        LIVE.monitor = None
        lab.logger.verbose = verbose_before
        if monitor.mode == "text":
            print(dashboard_text(lab, final))
        print(f"⏸ {final}. Estado salvo na geração {lab.counters['generation']}. "
              f"Para continuar, rode esta célula de novo (ou Executar tudo).\nrelatório da sessão: {rp}")
        lab.logger.event("research_stop", echo=False, reason=reason, generation=lab.counters["generation"])
    return reason

## 18. ▶ PESQUISA AO VIVO
A saída desta célula é o **painel ao vivo**: atualiza sozinho a cada segundo enquanto a pesquisa roda
(universos sendo simulados agora, fase e progresso, melhor candidato, descobertas, gráficos por geração).
Não precisa parar nada para ver resultados. ⏸ **Pausar**: botão ■ desta célula (salva tudo).
▶ **Continuar**: rode esta célula de novo (na mesma sessão) ou `Executar tudo` (sessão nova) — retoma do checkpoint.

In [ ]:
# @title ▶ PESQUISA AO VIVO — painel atualizado a cada segundo (rode de novo para continuar) { display-mode: "form" }
if START_RESEARCH:
    STOP_REASON = run_genesis(LAB)
else:
    print("START_RESEARCH=False — pesquisa não iniciada (rode run_genesis(LAB) quando quiser).")

## 19. Observação dos resultados (somente leitura)
MAP-Elites, convergência, melhor candidato, zoológico. Ferramentas: `inspect_candidate()`, `compare_candidates(a, b)`,
`replay_candidate()`, `reproduce_discovery(id)`, `export_candidate()`, `import_candidate(path)`, `profile_genesis()`,
`results_dataframe()`, `list_discoveries()`.

In [ ]:
# @title 19. Observação dos resultados (somente leitura) { display-mode: "form" }
plot_map_elites(LAB.map, os.path.join(PATHS.plots, "map_elites.png"), show=True)
plot_convergence(LAB.history, os.path.join(PATHS.plots, "convergence.png"), show=True)
print("Zoo:", LAB.zoo.counts())
print("Bandit de operadores (recompensa média):", LAB.bandit.probs())
_best = LAB.best_candidate()
if _best is not None:
    inspect_candidate(_best, show=True)
if LAB.champion is None:
    print("\nNenhum GENESIS CHAMPION ainda. Isso não é erro: rode novamente (Run all) para continuar a mesma pesquisa.")